# 그래프 서식 수정본 · 2026-10-13
그래프는 영어로 표기하고 필터 이름은 90도 세로 방향으로 배치합니다. 나눔고딕·흰 배경·색상 체계를 유지하고 겹침 방지를 위해 여백과 크기를 조절합니다.
학습·데이터 분할·평가 지표·저장된 측정값은 변경하지 않았습니다. 전체 실행은 원래 설정대로 학습을 수행할 수 있습니다.
**그래프만 다시 만들 때:** `python render_saved_figures.py --results yolo_filter_figures --output yolo_filter_figures_reference`


# YOLO Ball Detection-Based OK/NG Judgement (inside the rectangle only) + Filter Experiment

**How this differs from rule-based counting**: In the previous run (`judgement_results.csv`), judging directly from the labels made some "normal (1000-series)" images come out as NG. The cause was not the ball detection but the **step that locates the central rectangle (main part), `locate_part`, which picked a wrong small fragment in 8~9% of the images** (`area_frac` came out as 0.14~0.18 instead of the normal 0.47~0.50). This notebook fixes that step first (retry with several parameter sets and keep the best candidate), and then puts a **YOLO (Ultralytics) ball detector** on top of it and judges OK/NG from the real detections.

## Important - about "YOLO accuracy 99%"
Training YOLO normally reports **mAP (box detection accuracy)**. That tells you "how well were the balls found", which is **not the same metric as the final goal, "was the OK/NG judgement correct"**. This notebook feeds the YOLO detections into the rule (top=5 / left=4 / bottom=2 / right=2) and computes the **real OK/NG judgement accuracy**. There is no separate OK/NG table, so the ground truth is defined as **the same rectangle filter + rule applied to the label boxes** (if you have a ground-truth CSV, set `OKNG_CSV`).

## Filter experiment added in this notebook (Section 10 onwards)
This experiment pushes the images **as far as possible**: extreme filters at their maximum strength (colour inversion, binarisation, a heavy blur (sigma = 20 px; the balls stay visible as soft blobs), edge maps, ...), each with a fixed setting and **no tuning**. They are applied before YOLO detection, one by one and in combination, to see how the OK/NG judgement reacts.

| Round | Filters |
|---|---|
| Baseline | **Original** (no filter) |
| Round 1 - 10 single filters | Invert, Otsu Threshold, Solarize, Histogram Equalization, CLAHE (max), Gamma 0.3, Gaussian Blur (max), Sobel Edges, Emboss, Unsharp (max) |
| Round 2 - 5 combined filters | Invert+CLAHE, Blur+Otsu, Sobel+Invert, HistEq+Emboss, Unsharp+Sobel (`A+B` = A first, then B) |

Every filter goes through exactly the same pipeline (filter -> YOLO -> rectangle filter -> rule -> OK/NG) and is scored against the same label-based ground truth. Outputs: 10 graphs per round (20 in total) + 1 overview graph + 1 combination-vs-components graph, summary tables (CSV / JSON) with the parameters used and an exact McNemar test against Original, a per-image prediction CSV, a resumable result cache, a runtime estimate before the long run, one Original-vs-filtered example image per filter (saved **before** the experiment starts, Section 13), and paper-style grid figures for Original + Round 1 and Original + Round 2 (full view, native-resolution zoom and difference maps).

## Data (`/root/추계학술대회/DataSet`)
```
images/{train,val,test}/*.png   <->   labels/{train,val,test}/*.txt   (class cx cy w h, normalised 0~1)
data.yaml
```


> **Revision note (v4)**: besides the 16-method filter comparison (5-fold CV), the notebook now runs **difficulty sweeps** (blur sigma, noise, input resolution, amount of training data; Section 22) and a **filter x noise grid** (Section 23), because one fixed difficulty saturates at 100%. Section 24 draws the **reference 10-graph set** (accuracy bars, class-wise bars, dual axis, confusion matrices, learning curves, fold heat map, over/under-fit verdicts, radar, GPU timeline, dashboard) for every experiment group. Sections 5-8 stay as a development sanity check and are not held-out.

> **Revision note (v5)**: audit of the 'everything is ~100%' result. New Section 4b prints the class balance (majority-class baseline) and looks for near-duplicate images; if there are any, the CV folds keep them together (no leakage across folds). Every result now also carries a **Strict Acc** (every labelled ball found AND no extra detection anywhere, independent of the rectangle), the **majority-class baseline**, and - when `OKNG_CSV` is set - the agreement with a real OK/NG list. The label-based GT and the YOLO judgement share the same rectangle and rule, so the plain accuracy is a ball-detection score, not an independent OK/NG score.

> **Revision note (v6)**: Section 25 checks whether the detector is *guessing* (permutation test, erase-all / remove-one / insert-one interventions, image-level shortcut baseline; all inference-only on cached fold detectors). Section 26 is the **hard mode**: difficulty strata + strict-accuracy McNemar (26a), distribution shift without retraining (26b) and combined low-data + noise stress with retraining (26c). Section 27 prints what to report.

> **Revision note (v7)**: final all-in-one run. Fixed the misleading wording that a sigma = 20 px blur 'wipes out' the balls (it does not: the blur result stays at 100 % accuracy). New Section 28 (inference only): ghost-box diagnosis of the remove-one test with three erase methods and a free-spot control, shortcut diagnosis (image-level classifier on ball-erased images, AUC of global features such as file number / brightness / part position), operating-point sensitivity (confidence x match tolerance, without the fallback images). Section 29: agreement with a real OK/NG list (`REAL_LABEL_CSV` / `REAL_LABEL_DIRS` / `REAL_LABEL_FROM_NAME`). Section 30: **multi-seed replication** (`MS_SEEDS`, default 42/43/44; base seed from the caches) of the normal, hard (noise 40 + 25 % data) and noise-80 conditions with seed mean +- std, cluster-bootstrap CI over near-duplicate groups and per-seed McNemar. Section 31 prints the paper numbers and zips the new tables.

In [ ]:
import os, sys, glob, json, math, time, random, csv, shutil, hashlib
from collections import Counter, defaultdict
import numpy as np
import cv2

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_missing = []
try:
    from joblib import Parallel, delayed
except ImportError:
    _missing.append("joblib")
try:
    from ultralytics import YOLO
except ImportError:
    _missing.append("ultralytics")
try:
    import pandas as pd
except ImportError:
    _missing.append("pandas")
try:
    import yaml
except ImportError:
    _missing.append("pyyaml")
try:
    import torch
    import torch.nn.functional as F
except ImportError:
    _missing.append("torch")
try:
    import kornia
    import kornia.filters
    import kornia.enhance
except ImportError:
    _missing.append("kornia")
if _missing:
    raise RuntimeError(f"The following packages are required. Install them and restart the kernel:\n    pip install {' '.join(_missing)}")

try:
    from IPython.display import display
except ImportError:
    display = print

# ── GPU setup (every filter runs as a GPU tensor op) ──
REQUIRE_GPU = True
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif REQUIRE_GPU:
    raise RuntimeError("No CUDA GPU found (torch.cuda.is_available() == False). Check the CUDA driver / PyTorch-CUDA "
                       "install, or set REQUIRE_GPU = False to run the filters on the CPU (very slow).")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: no GPU found, the filters will run on the CPU.")
print(f"PyTorch: {torch.__version__} / CUDA: {torch.version.cuda}")

# ── Font setup ─────────────────────────────────────────────────────
font_path = r"C:\Users\PUVKID\AppData\Local\Microsoft\Windows\Fonts\NanumGothic.ttf"
font_bold_path = r"C:\Users\PUVKID\AppData\Local\Microsoft\Windows\Fonts\NanumGothicBold.ttf"
_linux_fallback = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
_linux_fallback_bold = "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf"

if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    if os.path.exists(font_bold_path):
        fm.fontManager.addfont(font_bold_path)
    font_name = fm.FontProperties(fname=font_path).get_name()
elif os.path.exists(_linux_fallback):
    fm.fontManager.addfont(_linux_fallback)
    if os.path.exists(_linux_fallback_bold):
        fm.fontManager.addfont(_linux_fallback_bold)
    font_name = fm.FontProperties(fname=_linux_fallback).get_name()
else:
    font_name = "DejaVu Sans"
    print("NanumGothic font not found. Falling back to DejaVu Sans.")

matplotlib.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [font_name, "DejaVu Sans"], "axes.unicode_minus": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#222222", "axes.labelcolor": "#222222",
    "xtick.color": "#222222", "ytick.color": "#222222", "text.color": "#222222",
    "grid.color": "#d9d9d9", "grid.linestyle": "-", "grid.alpha": 0.8,
    "legend.facecolor": "white", "legend.edgecolor": "#bbbbbb", "legend.labelcolor": "#222222",
})

FIG_DIR = "yolo_ok_ng_figures"              # baseline graphs (Sections 1~8)
RUNS_DIR = "yolo_runs"                      # YOLO training runs
FILTER_FIG_DIR = "yolo_filter_figures"      # filter experiment: graphs + tables
FILTER_SAMPLE_DIR = "yolo_filter_samples"   # filter experiment: Original vs filtered examples
FILTERED_ROOT = "yolo_filtered_datasets"    # only used when FILTER_MODE = "train_per_filter"
for _d in (FIG_DIR, FILTER_FIG_DIR, FILTER_SAMPLE_DIR):
    os.makedirs(_d, exist_ok=True)
print(f"Baseline figure folder: {os.path.abspath(FIG_DIR)}")
print(f"Filter experiment figure folder: {os.path.abspath(FILTER_FIG_DIR)}")
print(f"Logical cores: {os.cpu_count()}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### GPU sampler (for the timeline graph at the end)

In [ ]:
import subprocess, threading

GPU_LOG = []      # one dict per sample: time, used_mib, total_mib, util_pct, temp_c  (graph 9 of the final graph set)


class GpuSampler(threading.Thread):
    """Background thread: asks nvidia-smi every `every` seconds for as long as the kernel lives."""

    def __init__(self, every=15):
        super().__init__(daemon=True)
        self.every, self._halt = every, threading.Event()

    def run(self):
        while not self._halt.is_set():
            try:
                out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu",
                                      "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5)
                if out.returncode == 0:
                    used, total, util, temp = [float(v) for v in out.stdout.strip().split("\n")[0].split(",")[:4]]
                    GPU_LOG.append(dict(time=time.time(), used_mib=used, total_mib=total, util_pct=util, temp_c=temp))
            except Exception:
                pass
            self._halt.wait(self.every)

    def stop(self):
        self._halt.set()


if "GPU_SAMPLER" in globals() and GPU_SAMPLER.is_alive():
    GPU_SAMPLER.stop()
GPU_SAMPLER = GpuSampler(every=15)
GPU_SAMPLER.start()
print("GPU sampler started (every 15 s; the timeline is graph 9 of Section 24)")

In [ ]:
# Shared plotting style adapted from the supplied summer conference notebook.
# Graph labels and legends are fitted to avoid overlap; research computations are unchanged.
from reference_plot_style import (configure_reference_style, plot_clean_bar as _reference_bar,
    save_reference_figure, draw_bar, finish_bar, label, draw_reference_dashboard)
configure_reference_style()

def plot_clean_bar(*args, **kwargs):
    kwargs.setdefault("out_dir", FIG_DIR)
    return _reference_bar(*args, **kwargs)


## 1. Settings & data loading

In [ ]:
DATA_ROOT = r"/root/추계학술대회/DataSet"
SPLITS = ["train", "val", "test"]
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

REGION_RULE = {"top": 5, "left": 4, "bottom": 2, "right": 2}
BALL_CLASSES = None      # None: treat every class in the labels as a ball
OKNG_CSV = None          # path to a separate ground-truth OK/NG list (name,label), if you have one

_yaml_path = os.path.join(DATA_ROOT, "data.yaml")
if not os.path.isfile(_yaml_path):
    with open(_yaml_path, "w", encoding="utf-8") as f:
        f.write(f"path: {DATA_ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: ball\n")
    print(f"data.yaml was missing, created a new one: {_yaml_path}")
else:
    try:
        import yaml
        with open(_yaml_path, encoding="utf-8") as f:
            print("data.yaml:", yaml.safe_load(f))
    except Exception as e:
        print("Skipped reading data.yaml:", e)


def read_yolo_boxes(label_path, W, H, classes=BALL_CLASSES):
    boxes = []
    with open(label_path, encoding="utf-8") as f:
        for line in f:
            p = line.split()
            if len(p) < 5:
                continue
            c = int(float(p[0]))
            if classes is not None and c not in classes:
                continue
            xc, yc, bw, bh = (float(v) for v in p[1:5])
            boxes.append((xc * W, yc * H, bw * W, bh * H))
    return boxes


def list_dataset(root, splits):
    items, no_label = [], []
    for sp in splits:
        img_dir = os.path.join(root, "images", sp)
        lab_dir = os.path.join(root, "labels", sp)
        if not os.path.isdir(img_dir):
            print(f"[{sp}] image folder not found -> skipped: {img_dir}")
            continue
        paths = sorted(p for p in glob.glob(os.path.join(img_dir, "*")) if p.lower().endswith(IMG_EXTS))
        n_ok = 0
        for p in paths:
            stem = os.path.splitext(os.path.basename(p))[0]
            lp = os.path.join(lab_dir, stem + ".txt")
            if os.path.isfile(lp):
                items.append(dict(split=sp, stem=stem, key=f"{sp}/{stem}", img=p, lab=lp))
                n_ok += 1
            else:
                no_label.append(p)
        print(f"[{sp}] {len(paths)} images -> {n_ok} matched with labels")
    if no_label:
        print(f"WARNING: {len(no_label)} images excluded because they have no label file (e.g. {no_label[:3]})")
    return items


dataset = list_dataset(DATA_ROOT, SPLITS)
if not dataset:
    raise RuntimeError(f"No images to evaluate. Check DATA_ROOT and the folder layout: {DATA_ROOT}")
by_split = Counter(it["split"] for it in dataset)
print(f"Total {len(dataset)} images  {dict(by_split)}")

## 2. Locating the rectangle (main part) - [fixed] retry with several parameter sets

Problem found in the previous run: when the part was located with a single pass of local standard deviation -> Otsu -> opening, about 8~9% of the images ended up with a **wrong small fragment** (less than 1/3 of the normal area). All balls were then pushed "outside the rectangle", and an image that is actually normal was judged NG as a whole (0 balls in top / left / bottom / right).

**Fix**: try several combinations of the opening kernel size (`open_ksize`) and the texture window (`std_ksize`), and among the resulting candidates pick the one with the **largest area ratio** (only candidates that became oversized by merging with a neighbouring part are excluded; the part itself is always the big blob near the image centre). This greatly reduces the cases where a small fragment is picked.

In [ ]:
def _ellipse(k):
    k = max(1, int(k)) | 1
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))


LOC_PARAMS = dict(
    work_long=736,
    std_ksize_grid=(15, 21, 9),        # [fixed] retry with several texture windows
    close_ksize=9,
    open_ksize_grid=(61, 41, 81, 101, 31),  # [fixed] retry with several opening kernels
    min_area_frac=0.10,
    max_area_frac=0.85,
    inside_margin_frac=0.01,
)


def _locate_part_once(sm, h, w, std_ksize, open_ksize, close_ksize):
    f = sm.astype(np.float32)
    k = std_ksize
    mean = cv2.boxFilter(f, -1, (k, k))
    sq = cv2.boxFilter(f * f, -1, (k, k))
    std = np.sqrt(np.clip(sq - mean * mean, 0, None))
    u8 = cv2.normalize(std, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, th = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, _ellipse(close_ksize))

    pad = cv2.copyMakeBorder(th, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0)
    inv = 255 - pad
    cv2.floodFill(inv, np.zeros((h + 4, w + 4), np.uint8), (0, 0), 0)
    filled = th | inv[1:-1, 1:-1]

    op = cv2.morphologyEx(filled, cv2.MORPH_OPEN, _ellipse(open_ksize))
    n, lab, stats, cent = cv2.connectedComponentsWithStats(op)
    if n <= 1:
        return None
    cid = int(lab[h // 2, w // 2])
    if cid == 0:
        cand = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= 0.05 * h * w]
        if not cand:
            return None
        cid = min(cand, key=lambda i: math.hypot(cent[i][0] - w / 2, cent[i][1] - h / 2))
    comp = (lab == cid).astype(np.uint8) * 255
    cs, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cs:
        return None
    hull = cv2.convexHull(max(cs, key=cv2.contourArea))
    area_frac = cv2.contourArea(hull) / float(h * w)
    M = cv2.moments(hull)
    if M["m00"] <= 0:
        return None
    return dict(poly=hull.reshape(-1, 2).astype(np.float32), cx=M["m10"] / M["m00"], cy=M["m01"] / M["m00"],
                area_frac=area_frac)


def locate_part(gray, P=LOC_PARAMS):
    # [fixed] try every std_ksize x open_ksize combination and pick the candidate with the largest area ratio inside the min~max range.
    H, W = gray.shape
    s = min(1.0, P["work_long"] / max(H, W))
    sm = cv2.resize(gray, (max(1, int(round(W * s))), max(1, int(round(H * s)))), interpolation=cv2.INTER_AREA) if s < 1.0 else gray
    h, w = sm.shape
    sx, sy = W / w, H / h

    best = None
    for std_ksize in P["std_ksize_grid"]:
        for open_ksize in P["open_ksize_grid"]:
            r = _locate_part_once(sm, h, w, std_ksize, open_ksize, P["close_ksize"])
            if r is None:
                continue
            if P["min_area_frac"] <= r["area_frac"] <= P["max_area_frac"]:
                if best is None or r["area_frac"] > best["area_frac"]:
                    best = r
    if best is None:
        # if no candidate is inside the range (part shot unusually large or small), use the largest candidate overall and flag it as suspect
        for std_ksize in P["std_ksize_grid"]:
            for open_ksize in P["open_ksize_grid"]:
                r = _locate_part_once(sm, h, w, std_ksize, open_ksize, P["close_ksize"])
                if r is not None and (best is None or r["area_frac"] > best["area_frac"]):
                    best = r
        if best is not None:
            best["out_of_range"] = True
    if best is None:
        return None

    best.setdefault("out_of_range", False)
    poly = best["poly"] * np.array([sx, sy], np.float32)
    return dict(poly=poly, cx=float(best["cx"] * sx), cy=float(best["cy"] * sy),
                area_frac=float(best["area_frac"]), ok=not best["out_of_range"])

## 3. OK/NG judgement module

Ground-truth labels and YOLO detections are judged by **exactly the same function**, so only the "detector performance difference" is compared.

In [ ]:
def assign_region(x, y, cx, cy):
    ang = math.degrees(math.atan2(-(y - cy), x - cx))
    if -45 <= ang < 45:
        return "right"
    elif 45 <= ang < 135:
        return "top"
    elif ang >= 135 or ang < -135:
        return "left"
    return "bottom"


def judge_boxes(pts, part, img_wh, params=LOC_PARAMS):
    W, H = img_wh
    if part is None:
        cx, cy = W / 2.0, H / 2.0
        inside = [True] * len(pts)
    else:
        poly = part["poly"].reshape(-1, 1, 2).astype(np.float32)
        cx, cy = part["cx"], part["cy"]
        margin = params["inside_margin_frac"] * max(W, H)
        inside = [cv2.pointPolygonTest(poly, (float(x), float(y)), True) >= -margin for x, y in pts]

    counts = {k: 0 for k in REGION_RULE}
    for (x, y), ins in zip(pts, inside):
        if ins:
            counts[assign_region(x, y, cx, cy)] += 1

    ok = all(counts[k] == REGION_RULE[k] for k in REGION_RULE)
    diff = {k: counts[k] - REGION_RULE[k] for k in REGION_RULE if counts[k] != REGION_RULE[k]}
    reason = ", ".join(f"{k} {v:+d}" for k, v in diff.items())
    return dict(counts=counts, pred="OK" if ok else "NG", reason=reason, inside=inside)

## 4. Label-based ground truth (GT) - re-judged with the improved `locate_part`

First (without YOLO) the ground truth is recomputed from the label boxes + the improved rectangle detection. Check that `loc_fail` / `out_of_range` decreased compared to before. This value is used as the reference (ground truth) for all later YOLO judgements.

In [ ]:
def _gt_worker(item):
    cv2.setNumThreads(1)
    g = cv2.imread(item["img"], cv2.IMREAD_GRAYSCALE)
    if g is None:
        return dict(item, error=True)
    H, W = g.shape
    boxes = read_yolo_boxes(item["lab"], W, H)
    part = locate_part(g)
    j = judge_boxes([(b[0], b[1]) for b in boxes], part, (W, H))
    return dict(item, error=False, W=W, H=H, gt_boxes=boxes,
                part=part, loc_fail=part is None,
                loc_suspect=(part is not None and not part["ok"]),
                area_frac=float("nan") if part is None else part["area_frac"],
                gt_pred=j["pred"], gt_reason=j["reason"], gt_counts=j["counts"])


t0 = time.time()
gt_results = Parallel(n_jobs=-1, backend="loky", verbose=5)(delayed(_gt_worker)(it) for it in dataset)
print(f"Ground truth (GT) computed: {len(gt_results)} images, {(time.time() - t0) / 60:.1f} min")
gt_results = [r for r in gt_results if not r.get("error")]
gt_by_key = {r["key"]: r for r in gt_results}

n = len(gt_results)
print(f"\nRectangle detection failures (loc_fail): {sum(r['loc_fail'] for r in gt_results)}/{n}  "
      f"(out of range (loc_suspect): {sum(r['loc_suspect'] for r in gt_results)}/{n})")
print(f"GT OK {sum(r['gt_pred']=='OK' for r in gt_results)} / NG {sum(r['gt_pred']=='NG' for r in gt_results)}")

## 4b. Data audit (new in v5): class balance, label check, near-duplicate groups

Run this before training. It answers three questions that decide whether a ~100% accuracy means anything: (1) how high is the score of a detector that always answers the majority class, (2) how are the OK / NG labels built, (3) do near-identical images exist (they would sit in the train and the test fold at the same time and inflate every number). If near-duplicates exist and `GROUP_FOLDS = True`, Section 11 deals whole duplicate groups into the CV folds. If the largest group is bigger than one fold, the threshold is too loose and grouping is switched off (lower `DUP_MAD`).


In [ ]:
# ── 4b. Data audit ───────────────────────────────────────────────────
GROUP_FOLDS = True      # keep near-duplicate images in the same CV fold (Section 11 reads this)
DUP_SIZE = 128          # thumbnail size of the duplicate check
DUP_MAD = 1.0           # near-duplicate = mean absolute gray difference of the thumbnails below this (gray levels)

_n_gt = len(gt_results)
_gt_ok = sum(r["gt_pred"] == "OK" for r in gt_results)
_gt_ng = _n_gt - _gt_ok
_major = "OK" if _gt_ok >= _gt_ng else "NG"
print(f"GT class balance: OK {_gt_ok} ({100 * _gt_ok / _n_gt:.1f}%) / NG {_gt_ng} ({100 * _gt_ng / _n_gt:.1f}%)")
print(f"=> a detector that always answers '{_major}' already scores {100 * max(_gt_ok, _gt_ng) / _n_gt:.1f}%  (report this majority-class baseline next to every accuracy)")

print("\nLabelled balls per image:")
for cls in ("OK", "NG"):
    cnt = Counter(len(r["gt_boxes"]) for r in gt_results if r["gt_pred"] == cls)
    print(f"  GT {cls}: " + (", ".join(f"{k} balls x{v}" for k, v in sorted(cnt.items())) or "-"))
print("\nNOTE: GT and the YOLO judgement use the same rectangle (`part`) and the same rule, and detections outside the rectangle are ignored.\n"
      "      The OK/NG accuracy therefore measures 'are the balls inside the rectangle found exactly', not an independent OK/NG label.\n"
      "      Set OKNG_CSV (name,label) for agreement with a real OK/NG list, and read 'Strict Acc' (all labelled balls found, no extra detection anywhere).")


def find_dup_groups(items, size=DUP_SIZE, mad_thresh=DUP_MAD):
    """Union-find over image pairs whose size x size gray thumbnails differ by less than mad_thresh on average."""
    X = np.zeros((len(items), size * size), np.float32)
    for i, it in enumerate(items):
        g = cv2.imread(it["img"], cv2.IMREAD_GRAYSCALE)
        X[i] = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    parent = list(range(len(items)))

    def find_root(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    n_pairs = 0
    for i in range(len(items) - 1):
        d = np.abs(X[i + 1:] - X[i]).mean(axis=1)
        for j in np.nonzero(d < mad_thresh)[0]:
            n_pairs += 1
            ra, rb = find_root(i), find_root(i + 1 + int(j))
            if ra != rb:
                parent[rb] = ra
    return {it["key"]: find_root(i) for i, it in enumerate(items)}, n_pairs


_audit_items = [it for it in dataset if it["key"] in gt_by_key]
_t0 = time.time()
_gid, _n_pairs = find_dup_groups(_audit_items)
_groups = defaultdict(list)
for _k, _g in _gid.items():
    _groups[_g].append(_k)
_multi = [v for v in _groups.values() if len(v) > 1]
_sizes = sorted((len(v) for v in _groups.values()), reverse=True)
print(f"\nNear-duplicate check ({DUP_SIZE}px thumbnails, mean abs diff < {DUP_MAD:g}): {_n_pairs} similar pairs, "
      f"{len(_multi)} groups with >1 image ({sum(len(v) for v in _multi)} images), largest group {_sizes[0]}  [{time.time() - _t0:.0f} s]")
print(f"  groups spanning several splits: {sum(len({x.split('/')[0] for x in v}) > 1 for v in _multi)}"
      f" | groups with mixed OK/NG labels: {sum(len({gt_by_key[x]['gt_pred'] for x in v}) > 1 for v in _multi)}")

DUP_GROUP_OF = {}
if not _multi:
    print("  -> no near-duplicates found: the plain stratified folds are fine (GROUP_FOLDS has no effect).")
elif _sizes[0] > 0.2 * len(_audit_items):
    print(f"  WARNING: the largest group ({_sizes[0]}) is bigger than one CV fold -> DUP_MAD is too loose for this data, grouping switched off. "
          "Lower DUP_MAD (e.g. 0.5) and run this cell again.")
elif GROUP_FOLDS:
    DUP_GROUP_OF = _gid
    print("  -> duplicates found: Section 11 keeps every group inside one fold (the weights / result caches get a '_grp' tag, so old random-fold caches are not reused).")
else:
    print("  -> duplicates found but GROUP_FOLDS = False: the CV numbers may be optimistic.")


## 5. Training the YOLO ball detector (on the original images)

`data.yaml` (checked / created above) is used as is. You can retrain after adjusting `EPOCHS`, `IMGSZ` and `MODEL_SIZE`. If you already have trained weights (`best.pt`), put the path in `TRAINED_WEIGHTS` and set `DO_TRAIN=False` to skip training.

This model is the detector for the **Original** baseline and, in `FILTER_MODE = "inference_only"`, for every filter of the filter experiment.

In [ ]:
DO_TRAIN = True
MODEL_SIZE = "yolov8n.pt"   # use yolov8s.pt / yolov8m.pt for higher accuracy (slower)
EPOCHS = 100
IMGSZ = 1280                # the originals are large and the balls are small, so larger than the default 640
BATCH = 8
TRAINED_WEIGHTS = None      # path to already-trained weights (use together with DO_TRAIN=False)

# ── [added] Oversampling + augmentation (train split only; val/test are never touched) ──
# Images are grouped by their label-based OK/NG judgement; the minority group is repeated (symlinks + label copies in a
# separate folder, the original dataset is not modified) until the two groups are about equally large.
OVERSAMPLE = True
OVERSAMPLE_MAX_FACTOR = 6   # a minority image appears at most this many times
AUG = dict(flipud=0.5, fliplr=0.5, scale=0.3, translate=0.1, hsv_v=0.4)   # makes the repeated copies differ from each other
OVERSAMPLED_ROOT = "yolo_oversampled"


def _link_or_copy(src, dst):
    if os.path.lexists(dst):
        os.remove(dst)
    try:
        os.symlink(os.path.abspath(src), dst)
    except (OSError, NotImplementedError):
        shutil.copyfile(src, dst)


def build_oversampled_dataset(pairs, dst_root, val_dir, max_factor=OVERSAMPLE_MAX_FACTOR):
    """pairs: [(key, image_path, label_path)] of the TRAIN split. Returns (data_yaml_path, info dict)."""
    ok = [p for p in pairs if gt_by_key[p[0]]["gt_pred"] == "OK"]
    ng = [p for p in pairs if gt_by_key[p[0]]["gt_pred"] == "NG"]
    minor, major = (ok, ng) if len(ok) < len(ng) else (ng, ok)
    factor = 1 if not minor else int(min(max_factor, max(1, round(len(major) / len(minor)))))
    minor_keys = {p[0] for p in minor}
    img_dir = os.path.join(dst_root, "images", "train")
    lab_dir = os.path.join(dst_root, "labels", "train")
    for d in (img_dir, lab_dir):
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d, exist_ok=True)
    n_out = 0
    for key, img, lab in pairs:
        stem, ext = os.path.splitext(os.path.basename(img))
        for r in range(factor if key in minor_keys else 1):
            name = stem if r == 0 else f"{stem}_dup{r}"
            _link_or_copy(img, os.path.join(img_dir, name + ext))
            shutil.copyfile(lab, os.path.join(lab_dir, name + ".txt"))
            n_out += 1
    cfg = dict(path=os.path.abspath(dst_root), train="images/train", val=os.path.abspath(val_dir), names={0: "ball"})
    yaml_path = os.path.join(dst_root, "data.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    info = dict(n_ok=len(ok), n_ng=len(ng), factor=factor, minority="OK" if minor is ok else "NG", n_after=n_out)
    return yaml_path, info


train_yaml = _yaml_path
if DO_TRAIN:
    _tr_pairs = [(it["key"], it["img"], it["lab"]) for it in dataset if it["split"] == "train" and it["key"] in gt_by_key]
    _n_ok = sum(gt_by_key[k]["gt_pred"] == "OK" for k, _, _ in _tr_pairs)
    print(f"Train split GT balance: OK {_n_ok} / NG {len(_tr_pairs) - _n_ok}  (total {len(_tr_pairs)})")
    if OVERSAMPLE and _tr_pairs:
        _val_dir = os.path.join(DATA_ROOT, "images", "val")
        if not os.path.isdir(_val_dir):          # [fixed] no val folder: let YOLO validate on the train images
            _val_dir = os.path.join(DATA_ROOT, "images", "train")
        train_yaml, _info = build_oversampled_dataset(_tr_pairs, OVERSAMPLED_ROOT, _val_dir)
        print(f"Oversampling: minority = {_info['minority']} x{_info['factor']}  ->  {len(_tr_pairs)} -> {_info['n_after']} train images")
    model = YOLO(MODEL_SIZE)
    train_result = model.train(
        data=train_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        project=RUNS_DIR, name="ball_detect", seed=SEED, patience=30, **AUG,
    )
    best_weights = str(train_result.save_dir / "weights" / "best.pt")
else:
    if not TRAINED_WEIGHTS:
        raise RuntimeError("With DO_TRAIN=False, set TRAINED_WEIGHTS to the path of the trained weights.")
    best_weights = TRAINED_WEIGHTS

print(f"\nWeights in use: {best_weights}")
yolo_model = YOLO(best_weights)

## 6. YOLO detection -> rectangle filter -> OK/NG judgement (Original images)

The centres of the detection boxes go straight into `judge_boxes` (exactly the same function used to judge the labels). Adjust `CONF_THRES` if needed (lower: higher recall, more false detections; higher: the opposite).

This runs in a single process on the GPU with a progress bar (`yolo_model` from Section 5 is reused).


In [ ]:
CONF_THRES = 0.25
IOU_THRES = 0.5

try:
    from tqdm.auto import tqdm
except ImportError:            # no progress bar available: iterate silently
    tqdm = lambda it, **kw: it

# [added] The OK/NG rule needs the EXACT ball count, so the confidence threshold matters a lot. Pick it on the VAL split
# (balanced accuracy, so the majority class cannot dominate), never on test.
TUNE_CONF = bool(by_split.get("test"))   # [fixed] without a test split, val is what gets scored, so tuning on it would leak
if TUNE_CONF:
    _val = [it for it in dataset if it["split"] == "val" and it["key"] in gt_by_key]
    if len(_val) >= 20:
        _raw = []
        for it in tqdm(_val, desc="conf sweep (val)", unit="img"):
            _p = yolo_model.predict(it["img"], imgsz=IMGSZ, conf=0.05, iou=IOU_THRES, verbose=False)[0]
            _xy = _p.boxes.xywh.cpu().numpy() if len(_p.boxes) else np.zeros((0, 4))
            _cf = _p.boxes.conf.cpu().numpy() if len(_p.boxes) else np.zeros(0)
            _raw.append((it, _xy, _cf))
        _best = (-1.0, CONF_THRES)
        for _c in np.arange(0.10, 0.80, 0.05):
            _hit = {"OK": [0, 0], "NG": [0, 0]}
            for it, _xy, _cf in _raw:
                _g = gt_by_key[it["key"]]
                _pts = [(float(x), float(y)) for (x, y, _, _), s in zip(_xy, _cf) if s >= _c]
                _pred = judge_boxes(_pts, _g["part"], (_g["W"], _g["H"]))["pred"]
                _hit[_g["gt_pred"]][0] += int(_pred == _g["gt_pred"])
                _hit[_g["gt_pred"]][1] += 1
            _rec = [h / n for h, n in _hit.values() if n]
            _bal = float(np.mean(_rec)) if _rec else 0.0
            if _bal > _best[0] + 1e-9:
                _best = (_bal, float(_c))
        CONF_THRES = round(_best[1], 2)
        print(f"CONF_THRES tuned on val: {CONF_THRES} (balanced accuracy {100 * _best[0]:.2f}%)")
    else:
        print(f"Too few val images ({len(_val)}) to tune CONF_THRES, keeping {CONF_THRES}")

# One process, one model (`yolo_model` from Section 5). The GPU is the bottleneck here, so extra worker processes
# would only add start-up time and GPU memory (every process loads its own model and CUDA context), and joblib
# prints nothing until a whole chunk is finished, which looks like a frozen notebook.
results, t0 = [], time.time()
for item in tqdm(dataset, desc="YOLO judgement", unit="img"):
    gt = gt_by_key.get(item["key"])
    if gt is None:                        # unreadable image: no ground truth was computed for it
        results.append(dict(item, error=True))
        continue
    pred = yolo_model.predict(item["img"], imgsz=IMGSZ, conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
    xywh = pred.boxes.xywh.cpu().numpy() if len(pred.boxes) else np.zeros((0, 4))
    pts = [(float(x), float(y)) for x, y, _, _ in xywh]
    j = judge_boxes(pts, gt["part"], (gt["W"], gt["H"]))
    results.append(dict(item, error=False, W=gt["W"], H=gt["H"], pred_boxes=xywh.tolist(),
                        part=gt["part"], loc_fail=gt["part"] is None,
                        yolo_pred=j["pred"], yolo_reason=j["reason"], yolo_counts=j["counts"], n_det=len(pts)))
print(f"YOLO judgement done: {len(results)} images, {(time.time() - t0) / 60:.1f} min")
results = [r for r in results if not r.get("error")]
for r in results:
    r["gt_pred"] = gt_by_key[r["key"]]["gt_pred"]


## 7. Accuracy evaluation (YOLO judgement vs label-based ground truth) - Original images

In [ ]:
def _pct(a, b):
    return 100.0 * a / b if b else 0.0

n = len(results)
acc = _pct(sum(r["yolo_pred"] == r["gt_pred"] for r in results), n)
print(f"All {n} images  OK/NG judgement accuracy (vs label-based GT): {acc:.2f}%")

tp = sum(r["gt_pred"] == "NG" and r["yolo_pred"] == "NG" for r in results)
fn = sum(r["gt_pred"] == "NG" and r["yolo_pred"] == "OK" for r in results)
fp = sum(r["gt_pred"] == "OK" and r["yolo_pred"] == "NG" for r in results)
tn = sum(r["gt_pred"] == "OK" and r["yolo_pred"] == "OK" for r in results)
ng_recall = _pct(tp, tp + fn); ok_recall = _pct(tn, tn + fp)
precision = _pct(tp, tp + fp) if (tp + fp) else 0.0
f1 = 2 * precision * ng_recall / (precision + ng_recall) if (precision + ng_recall) else 0.0
print(f"NG recall {ng_recall:.2f}% | OK recall {ok_recall:.2f}% | NG Precision {precision:.2f}% | F1 {f1:.2f}%")
print(f"Balanced accuracy {(ok_recall + ng_recall) / 2:.2f}%   (plain accuracy hides a collapse onto the majority class)")
print("GT balance per split: " + ", ".join(f"{sp} OK {sum(r['split'] == sp and r['gt_pred'] == 'OK' for r in results)}/NG {sum(r['split'] == sp and r['gt_pred'] == 'NG' for r in results)}" for sp in SPLITS))
print(f"Confusion matrix:  NG->NG {tp}  NG->OK {fn}  OK->NG {fp}  OK->OK {tn}")

print("\n[Accuracy per split]")
for sp in SPLITS:
    rs = [r for r in results if r["split"] == sp]
    if not rs:
        continue
    a = _pct(sum(r["yolo_pred"] == r["gt_pred"] for r in rs), len(rs))
    print(f"  {sp:5s} {len(rs):5d} images  accuracy {a:6.2f}%")

print("\n[Top 10 reasons for mismatches (judged differently from the label-based GT)]")
mism = [r for r in results if r["yolo_pred"] != r["gt_pred"]]
for reason, cnt in Counter((r["gt_pred"], r["yolo_pred"], r["yolo_reason"]) for r in mism).most_common(10):
    print(f"  {cnt:4d} images  GT {reason[0]} -> YOLO {reason[1]}   (count difference: {reason[2]})")

# (optional) compare with a separate ground-truth OK/NG list
if OKNG_CSV and os.path.isfile(OKNG_CSV):
    ref = {}
    with open(OKNG_CSV, newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            ref[os.path.splitext(row["name"].strip())[0]] = row["label"].strip().upper()
    rr = [(r, ref[r["stem"]]) for r in results if r["stem"] in ref]
    if rr:
        a2 = _pct(sum(r["yolo_pred"] == y for r, y in rr), len(rr))
        print(f"\n[Ground-truth CSV comparison] {len(rr)} images  YOLO agreement {a2:.2f}%")

csv_path = os.path.join(FIG_DIR, "yolo_judgement_results.csv")
with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    wr = csv.writer(f)
    wr.writerow(["key", "split", "gt_pred", "yolo_pred", "match", "yolo_reason", "n_det", "loc_fail"])
    for r in results:
        wr.writerow([r["key"], r["split"], r["gt_pred"], r["yolo_pred"],
                     int(r["gt_pred"] == r["yolo_pred"]), r["yolo_reason"], r["n_det"], int(r["loc_fail"])])
print(f"\nPer-image results saved: {csv_path}")

print("\nNOTE: these numbers are NOT held-out (the detector trained on the train images and best.pt was picked on val). "
      "Use them as a sanity check; the paper numbers are the cross-validated ones from Section 14 on.")

## 8. Baseline graphs (4) - Original images

In [ ]:
plot_clean_bar(["Overall"], [acc], "", "OK/NG Judgement Accuracy (%)", "01_overall_accuracy.png",
               ymax=110, threshold=99, threshold_label="99% Threshold", show_min=False)

split_labels = [sp for sp in SPLITS if any(r["split"] == sp for r in results)]
split_acc = [_pct(sum(r["yolo_pred"] == r["gt_pred"] for r in results if r["split"] == sp),
                   sum(r["split"] == sp for r in results)) for sp in split_labels]
plot_clean_bar(split_labels, split_acc, "Split", "Accuracy (%)", "02_split_accuracy.png", ymax=110)

plot_clean_bar(["OK Recall", "NG Recall", "Precision", "F1"], [ok_recall, ng_recall, precision, f1],
               "Metric", "Score (%)", "03_ok_ng_metrics.png", ymax=110, show_min=False)

cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                fontsize=18, fontweight="bold", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred OK", "Pred NG"], fontsize=13, fontweight="bold")
ax.set_yticks([0, 1]); ax.set_yticklabels(["True OK", "True NG"], fontsize=13, fontweight="bold")
ax.set_title("Confusion Matrix", fontsize=15, fontweight="bold")
fig.tight_layout()
cm_path = os.path.join(FIG_DIR, "04_confusion_matrix.png")
save_reference_figure(fig, cm_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {cm_path}")

## 9. Visual check

Yellow = rectangle, green = YOLO detection (inside the rectangle), red = YOLO detection (outside the rectangle, excluded). Look first at the images that were judged differently from the label-based ground truth - you can tell whether YOLO missed balls / found extra ones, or whether the rectangle is misplaced.

In [ ]:
def draw_case(r, scale=0.3):
    g = cv2.imread(r["img"], cv2.IMREAD_GRAYSCALE)
    vis = cv2.cvtColor(cv2.normalize(g, None, 0, 255, cv2.NORM_MINMAX), cv2.COLOR_GRAY2BGR)
    part = r.get("part")
    if part is not None:
        cv2.polylines(vis, [np.array(part["poly"], np.int32).reshape(-1, 1, 2)], True, (0, 255, 255), 8)
        cx, cy = part["cx"], part["cy"]
        margin = LOC_PARAMS["inside_margin_frac"] * max(r["W"], r["H"])
        poly = np.array(part["poly"], np.float32).reshape(-1, 1, 2)
    for x, y, bw, bh in r["pred_boxes"]:
        ins = True
        if part is not None:
            ins = cv2.pointPolygonTest(poly, (float(x), float(y)), True) >= -margin
        col = (0, 220, 0) if ins else (0, 0, 255)
        cv2.rectangle(vis, (int(x - bw / 2), int(y - bh / 2)), (int(x + bw / 2), int(y + bh / 2)), col, 6)
    return cv2.cvtColor(cv2.resize(vis, (0, 0), fx=scale, fy=scale), cv2.COLOR_BGR2RGB)


def show_cases(rs, title, n_show=6, cols=3, scale=0.3):
    rs = rs[:n_show]
    if not rs:
        print(f"[{title}] no matching images.")
        return
    rows = (len(rs) + cols - 1) // cols
    plt.figure(figsize=(6 * cols, 5 * rows))
    for i, r in enumerate(rs):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(draw_case(r, scale))
        plt.axis("off")
        c = r["yolo_counts"]
        plt.title(f"{r['key']}  GT {r['gt_pred']} -> YOLO {r['yolo_pred']}\n"
                  f"detected {c['top']}/{c['left']}/{c['bottom']}/{c['right']} (top/left/bottom/right)  {r['yolo_reason']}", fontsize=10)
    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


rng = random.Random(SEED)
show_cases(rng.sample(results, min(6, len(results))), "(1) 6 random images")
show_cases([r for r in results if r["yolo_pred"] != r["gt_pred"]], "(2) Images judged differently from the label-based GT")
show_cases([r for r in results if r["loc_fail"]], "(3) Images where the rectangle detection failed")

## 10. Filter experiment - filter library

This experiment uses **extreme filters at their maximum strength** - changes you can see at a glance (a colour-inverted image, a black-and-white image, a heavily blurred image, an edge map, ...) - to find out how far the OK/NG judgement can be pushed before it breaks. There is **no parameter tuning**: every filter runs with the fixed, deliberately strong setting below.

Each filter takes a `(1,1,H,W)` GPU tensor in `[0,1]` and returns a tensor of the same shape. Filters run at the **full image resolution**, so the label boxes and the rectangle from `locate_part` stay valid. All of them are GPU tensor operations.

**Round 1 - 10 single filters**

| Filter | What it does | Setting |
|---|---|---|
| Invert | negative image (black <-> white) | - |
| Otsu Threshold | black-and-white image, threshold chosen automatically | - |
| Solarize | everything brighter than the image mean is inverted | threshold = image mean |
| Histogram Equalization | global histogram equalization | - |
| CLAHE (max) | local contrast stretching with a huge clip limit | clip_limit = 40, 16 x 16 tiles |
| Gamma 0.3 | strong brightening | gamma = 0.3 |
| Gaussian Blur (max) | heavy smoothing (the balls stay visible as soft blobs; fine texture and noise are removed) | sigma = 20 px |
| Sobel Edges | edge map, stretched to the full range | - |
| Emboss | relief image | gain = 4 |
| Unsharp (max) | extreme sharpening with halos | amount = 8, sigma = 2 |


In [ ]:
def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def to_gpu_tensor(gray_np):
    t = torch.from_numpy(gray_np).float().to(DEVICE) / 255.0
    return t.unsqueeze(0).unsqueeze(0)  # (1,1,H,W)


def to_gray_np(t):
    arr = t.detach().squeeze().clamp(0, 1).cpu().numpy()
    return (arr * 255.0).round().astype(np.uint8)


def identity_filter(t):
    """'Original' baseline: no filtering."""
    return t


# ── A filter together with the parameters it is run with ──────────
class ParamFilter:
    """A filter function bound to its parameters, so tables and caches show exactly what was run."""

    def __init__(self, name, fn, params):
        self.name, self.fn, self.params = name, fn, dict(params)

    def __call__(self, t):
        return self.fn(t, **self.params)

    def describe(self):
        return ", ".join(f"{k}={v}" for k, v in self.params.items()) or "-"

    def signature(self):
        return json.dumps({self.name: self.params}, sort_keys=True)


class Chain:
    """Two filters applied one after the other (first stage first)."""

    def __init__(self, *stages):
        self.stages = stages

    def __call__(self, t):
        for stage in self.stages:
            t = stage(t)
        return t

    def describe(self):
        return " > ".join(f"{s.name}[{s.describe()}]" for s in self.stages)

    def signature(self):
        return "|".join(s.signature() for s in self.stages)


# ── 1) Invert (negative image) ───────────────────────────────────
def invert_gpu(t):
    return 1.0 - t


# ── 2) Otsu threshold (black-and-white image, automatic threshold) ──
def otsu_threshold_gpu(t):
    x = (t.clamp(0, 1) * 255).round()
    hist = torch.histc(x, bins=256, min=0, max=256)            # bin i = gray level i
    p = hist / hist.sum()
    levels = torch.arange(256, device=t.device, dtype=t.dtype)
    omega = torch.cumsum(p, dim=0)                              # share of pixels <= level
    mu = torch.cumsum(p * levels, dim=0)
    sigma_between = (mu[-1] * omega - mu) ** 2 / (omega * (1 - omega)).clamp(min=1e-12)
    thr = torch.argmax(sigma_between)
    return (x > thr).to(t.dtype)


# ── 3) Solarize (everything brighter than the image mean is inverted) ──
def solarize_gpu(t):
    return torch.where(t > t.mean(), 1.0 - t, t)


# ── 4) Histogram equalization (global) ───────────────────────────
def hist_equalize_gpu(t):
    return kornia.enhance.equalize(t)


# ── 5) CLAHE with a huge clip limit (maximum local contrast) ─────
def clahe_gpu(t, clip_limit=40.0, tiles=16):
    return kornia.enhance.equalize_clahe(t, clip_limit=clip_limit, grid_size=(tiles, tiles))


# ── 6) Gamma (strong brightening) ────────────────────────────────
def gamma_gpu(t, gamma=0.3):
    return t.clamp(0, 1) ** gamma


# ── 7) Gaussian blur with a huge sigma (fine texture disappears; the balls stay visible as soft blobs) ──
def gaussian_blur_gpu(t, sigma=20.0):
    k = int(sigma * 6) | 1                                      # odd kernel covering +-3 sigma
    return kornia.filters.gaussian_blur2d(t, (k, k), (sigma, sigma), border_type="reflect")


# ── 8) Sobel edge map (stretched to the full range) ──────────────
def sobel_edges_gpu(t):
    g = kornia.filters.sobel(t, normalized=True)
    return (g / g.amax().clamp(min=1e-6)).clamp(0, 1) ** 0.5


# ── 9) Emboss (relief) ───────────────────────────────────────────
def emboss_gpu(t, gain=4.0):
    k = torch.tensor([[-2.0, -1.0, 0.0], [-1.0, 0.0, 1.0], [0.0, 1.0, 2.0]], device=t.device, dtype=t.dtype).view(1, 1, 3, 3)   # weights sum to 0: flat areas -> mid-gray
    out = F.conv2d(F.pad(t, (1, 1, 1, 1), mode="reflect"), k)
    return (gain * out + 0.5).clamp(0, 1)


# ── 10) Unsharp mask with a huge amount ──────────────────────────
def unsharp_gpu(t, amount=8.0, sigma=2.0):
    k = int(sigma * 6) | 1
    blurred = kornia.filters.gaussian_blur2d(t, (k, k), (sigma, sigma), border_type="reflect")
    return (t + amount * (t - blurred)).clamp(0, 1)


**Round 2 - 5 combined filters** (my own picks of Round 1 filters; `A+B` means *A first, then B*)

| Combination | Idea |
|---|---|
| `Invert+CLAHE` | negative image, then maximum local contrast |
| `Blur+Otsu` | blur everything small away, then cut the result into black and white (a rough segmentation) |
| `Sobel+Invert` | edge map as dark lines on white, like a pencil drawing |
| `HistEq+Emboss` | stretch the histogram, then turn the result into a relief |
| `Unsharp+Sobel` | extreme sharpening, then edges (noise and halos become edges) |

Section 19 puts every combination next to its two parts.


In [ ]:
# ── The filters of the experiment: function + fixed (maximum-strength) parameters ──
FILTER_SPECS = {
    "Invert": dict(fn=invert_gpu, params={}),
    "Otsu Threshold": dict(fn=otsu_threshold_gpu, params={}),
    "Solarize": dict(fn=solarize_gpu, params={}),
    "Histogram Equalization": dict(fn=hist_equalize_gpu, params={}),
    "CLAHE (max)": dict(fn=clahe_gpu, params=dict(clip_limit=40.0, tiles=16)),
    "Gamma 0.3": dict(fn=gamma_gpu, params=dict(gamma=0.3)),
    "Gaussian Blur (max)": dict(fn=gaussian_blur_gpu, params=dict(sigma=20.0)),
    "Sobel Edges": dict(fn=sobel_edges_gpu, params={}),
    "Emboss": dict(fn=emboss_gpu, params=dict(gain=4.0)),
    "Unsharp (max)": dict(fn=unsharp_gpu, params=dict(amount=8.0, sigma=2.0)),
}
ROUND1_ORDER = list(FILTER_SPECS)

ROUND2_COMBOS = {   # combined filter -> (first stage, second stage); "A+B" = A first, then B
    "Invert+CLAHE": ("Invert", "CLAHE (max)"),
    "Blur+Otsu": ("Gaussian Blur (max)", "Otsu Threshold"),
    "Sobel+Invert": ("Sobel Edges", "Invert"),
    "HistEq+Emboss": ("Histogram Equalization", "Emboss"),
    "Unsharp+Sobel": ("Unsharp (max)", "Sobel Edges"),
}


def make_filter(name):
    spec = FILTER_SPECS[name]
    return ParamFilter(name, spec["fn"], spec["params"])


def build_filter_lists():
    """(Round 1, Round 2) lists of (name, filter)."""
    r1 = [(n, make_filter(n)) for n in ROUND1_ORDER]
    r2 = [(cn, Chain(make_filter(a), make_filter(b))) for cn, (a, b) in ROUND2_COMBOS.items()]
    return r1, r2


FILTERS_ROUND1, FILTERS_ROUND2 = build_filter_lists()
print(f"Round 1: {len(FILTERS_ROUND1)} single filters registered: {[n for n, _ in FILTERS_ROUND1]}")
print(f"Round 2: {len(FILTERS_ROUND2)} combined filters registered: {[n for n, _ in FILTERS_ROUND2]}")


## 11. Filter experiment - settings and evaluation functions

**`FILTER_MODE`** decides how a filter meets the YOLO detector:

| Mode | What happens | Cost |
|---|---|---|
| `"inference_only"` (default) | The YOLO trained on the **original** images (Section 5) is reused. Only the input image is filtered at judgement time. Measures whether a filter helps / hurts *this* detector. | Fast - no extra training |
| `"train_per_filter"` | The whole dataset is filtered and written to `yolo_filtered_datasets/<filter>/`, and a **separate YOLO is trained for each filter** with the same hyper-parameters as Section 5. Each model is evaluated on its own filtered images. This is the fairer comparison, because a detector trained on unfiltered images can be penalised by a strong filter (distribution shift). | Slow - 15 trainings (finished filters / trainings are reused when you re-run; the cache key contains the filter's parameters, so changed parameters never reuse old filtered images or weights) |

**With extreme filters the two modes answer different questions.** `inference_only` shows how much the detector depends on the normal look of the images (a detector that never saw an inverted or binarised image will probably fail on it). `train_per_filter` shows whether the information is still in the filtered image (a plain inversion can be learned again; a blur that really removed the balls could not - sigma = 20 px does not).

The "Original" baseline always uses the Section 5 model on the unfiltered images. Ground truth and the rectangle (`part`) always come from the **original** images (Section 4), so only the detector's input changes.

Other settings: **`RUN_PRESET`** bundles the three values `EVAL_SPLITS`, `QUICK_TEST_N` and `SKIP_FILTERS` - `"quick"` (default: 30 random images per filter, every split = a smoke test that finishes fast), `"final"` (the whole **test** split and every filter = the paper run; the model never saw the test images in training, so this is the cleanest comparison) or `"custom"` (your own `CUSTOM_*` values). Sections 6~8 (baseline graphs) still use every split.

**Resume:** every finished method is saved to `<filter figure folder>/result_cache/` and skipped when the notebook is run again, so a crash or a stopped kernel does not cost the hours already spent. A saved result is only reused when the images, ground truth, detector, thresholds and the filter's parameters are all unchanged. Set `REUSE_RESULTS = False` (or delete the folder) to evaluate everything again.


**Cross-validation (`USE_CV = True`, default).** The data has no test split (400 train / 100 val), and 100 scored images cannot separate 99% from 100% (95% CI 96.3 - 100). So the 500 images are split into `CV_FOLDS = 5` stratified folds (by label-based OK/NG). For each fold the detector is trained on the other four folds (minus `CV_VAL_FRAC` = 10% as an inner validation set for early stopping) and judges only the held-out fold, so all 500 images are scored by a detector that never saw them. **Original is trained with the same recipe as every filter** (`FILTER_EPOCHS`, augmentation, oversampling) - before, Original had 100 epochs and the filters 30.

Besides OK/NG accuracy, each method now reports metrics that keep differing when the accuracy is saturated: *ball recall* (labelled balls that were found), *mAP50-95* on the held-out fold, *mean minimum confidence* (how close each image was to failing), *centre error* in pixels, extra detections per image and the accuracy spread across folds. `RUN_PRESET = "quick"` runs fold 0 only; `CV_RUN_FOLDS = [0, 1]` runs chosen folds. The optional `STRESS_*` settings (fewer training images, lower resolution, noise) are off by default; turn one on when everything still ties at 100%.

In [ ]:
# ── Experiment settings ─────────────────────────────────────────
FILTER_MODE = "train_per_filter"    # "inference_only" | "train_per_filter"
# [changed] was "inference_only": a YOLO trained on normal images sees inverted / binarised / blurred inputs it has never
# seen, so it predicts (almost) nothing and every image collapses to NG (or OK) -> the accuracy just equals the class share.
FILTER_EPOCHS = 30                  # epochs per filter in train_per_filter mode (the Original model uses EPOCHS)

# ── [added] Stratified K-fold cross-validation (train_per_filter mode) ──────────────────────────
# Why: with the fixed 400/100 split there is no test set, the 100 scored images also picked the checkpoint (best.pt),
# and even 100/100 correct only gives a 95% CI of 96.3 - 100. In CV every one of the 500 images is judged exactly
# once by a detector that never trained on it (80% of the images, minus an inner 10% val used for early stopping),
# so N = 500. Original and every filter use the SAME training recipe (FILTER_EPOCHS, AUG, oversampling) -> fair.
USE_CV = True
CV_FOLDS = 5
CV_RUN_FOLDS = None        # None: "quick" -> [0] only, otherwise every fold; or e.g. [0, 1]
CV_VAL_FRAC = 0.10         # inner validation share of the non-test images (early stopping / best.pt only)

# ── [added] Optional stress conditions (default: all off). A harder detector task is where filters can differ. ──
STRESS_TRAIN_FRAC = 1.0    # e.g. 0.25 -> train on 25% of the training images only (low-data regime)
STRESS_IMGSZ = None        # e.g. 640  -> train / predict at a lower resolution than IMGSZ
STRESS_NOISE_STD = 0.0     # e.g. 15   -> Gaussian noise (gray levels, fixed per image) added BEFORE the filter
CV_IMGSZ = STRESS_IMGSZ or IMGSZ
_STRESS_TAG = (f"_n{STRESS_NOISE_STD:g}" if STRESS_NOISE_STD else "")

if USE_CV and FILTER_MODE != "train_per_filter":
    print("NOTE: USE_CV needs FILTER_MODE = 'train_per_filter' (otherwise one detector is scored on its own training images) -> CV switched off.")
    USE_CV = False


def _noisy(gray, key):
    if not STRESS_NOISE_STD:
        return gray
    rng = np.random.default_rng(int(hashlib.md5(key.encode("utf-8")).hexdigest()[:8], 16))
    return np.clip(gray.astype(np.float32) + rng.normal(0.0, STRESS_NOISE_STD, gray.shape), 0, 255).astype(np.uint8)

# RUN_PRESET: "quick"  -> smoke test : 30 random images per filter, every split
#             "final"  -> paper run  : the whole test split, every filter (nothing skipped)
#             "custom" -> use the three CUSTOM_* values below
RUN_PRESET = "final"    # [changed] was "quick": 30 images -> accuracy can only be k/30 (53.33 = 16/30, 100 = 30/30)
CUSTOM_EVAL_SPLITS = list(SPLITS)   # e.g. ["test"] to score only the test split
CUSTOM_QUICK_TEST_N = None          # e.g. 30 -> N random images per filter
CUSTOM_SKIP_FILTERS = set()         # e.g. {"Invert"} to leave a filter out

if RUN_PRESET == "quick":
    EVAL_SPLITS, QUICK_TEST_N, SKIP_FILTERS = list(SPLITS), 30, set()
elif RUN_PRESET == "final":
    # [fixed] this dataset may have no test split (by_split comes from Section 1): fall back test -> val -> everything available
    _fallback = next(([sp] for sp in ("test", "val") if by_split.get(sp)), [sp for sp in SPLITS if by_split.get(sp)])
    if _fallback != ["test"] and not USE_CV:
        print(f"WARNING: no test split found (images per split: {dict(by_split)}) -> scoring {_fallback} instead. "
              "Numbers on a split the detector trained on are optimistic; keep that in mind for the paper.")
    EVAL_SPLITS, QUICK_TEST_N, SKIP_FILTERS = _fallback, None, set()
elif RUN_PRESET == "custom":
    EVAL_SPLITS, QUICK_TEST_N, SKIP_FILTERS = list(CUSTOM_EVAL_SPLITS), CUSTOM_QUICK_TEST_N, set(CUSTOM_SKIP_FILTERS)
else:
    raise ValueError(f"Unknown RUN_PRESET: {RUN_PRESET!r} (use 'quick', 'final' or 'custom')")

eval_items = [it for it in dataset if it["split"] in EVAL_SPLITS and it["key"] in gt_by_key]

# ── [added] CV folds: every image gets one fold id (stratified by the label-based OK/NG judgement) ──
CV_ITEMS = [it for it in dataset if it["key"] in gt_by_key]


GROUP_FOLDS = globals().get("GROUP_FOLDS", False)       # [added v5] set / decided in Section 4b
DUP_GROUP_OF = globals().get("DUP_GROUP_OF", {})


def assign_folds(items, k, seed):
    rng = random.Random(seed)
    fold_of = {}
    if not (GROUP_FOLDS and DUP_GROUP_OF):          # plain stratified split (unchanged behaviour, old caches stay valid)
        for cls in ("OK", "NG"):
            keys = sorted(it["key"] for it in items if gt_by_key[it["key"]]["gt_pred"] == cls)
            rng.shuffle(keys)
            for i, key in enumerate(keys):
                fold_of[key] = i % k
        return fold_of
    # [added v5] near-duplicate images stay together: whole groups are dealt to the emptiest fold (per class, biggest first)
    groups = defaultdict(list)
    for it in items:
        groups[DUP_GROUP_OF[it["key"]]].append(it["key"])
    by_cls = {"OK": [], "NG": []}
    for g in groups.values():
        n_ok = sum(gt_by_key[x]["gt_pred"] == "OK" for x in g)
        by_cls["OK" if n_ok * 2 > len(g) else "NG"].append(sorted(g))
    for cls in ("OK", "NG"):
        units = sorted(by_cls[cls])
        rng.shuffle(units)
        units.sort(key=len, reverse=True)            # stable: big groups first, ties keep the shuffled order
        sizes = [0] * k
        for g in units:
            f = min(range(k), key=lambda x: (sizes[x], x))
            for key in g:
                fold_of[key] = f
            sizes[f] += len(g)
    return fold_of


FOLD_OF = assign_folds(CV_ITEMS, CV_FOLDS, SEED) if USE_CV else {}
FOLD_TAG = "_grp" if (USE_CV and GROUP_FOLDS and DUP_GROUP_OF) else ""     # part of every weights / result cache key
if USE_CV:
    print("Fold sizes: " + ", ".join(f"fold {f}: {sum(v == f for v in FOLD_OF.values())}" for f in range(CV_FOLDS))
          + ("  (near-duplicate groups kept together)" if FOLD_TAG else ""))


def fold_split(k):
    """(train, val, test) item lists of fold k. test = fold k; val = CV_VAL_FRAC of the rest (stratified)."""
    test = [it for it in CV_ITEMS if FOLD_OF[it["key"]] == k]
    rest = [it for it in CV_ITEMS if FOLD_OF[it["key"]] != k]
    rng = random.Random(SEED + 100 + k)
    train, val = [], []
    for cls in ("OK", "NG"):
        grp = [it for it in rest if gt_by_key[it["key"]]["gt_pred"] == cls]
        rng.shuffle(grp)
        nv = max(1, round(len(grp) * CV_VAL_FRAC)) if grp else 0
        val += grp[:nv]
        tr = grp[nv:]
        train += tr[:max(1, round(len(tr) * STRESS_TRAIN_FRAC))] if tr else []
    return train, val, test


_run_folds = []
if USE_CV:
    _run_folds = list(CV_RUN_FOLDS) if CV_RUN_FOLDS is not None else ([0] if RUN_PRESET == "quick" else list(range(CV_FOLDS)))
    eval_items = [it for it in CV_ITEMS if FOLD_OF[it["key"]] in _run_folds]
    EVAL_SPLITS = [f"{CV_FOLDS}-fold CV, folds {_run_folds}"]
    QUICK_TEST_N = None
    print(f"Cross-validation: {CV_FOLDS} folds, running {_run_folds}  (train frac {STRESS_TRAIN_FRAC:g}, imgsz {CV_IMGSZ}, noise {STRESS_NOISE_STD:g})")
    for _f in _run_folds:
        _tr, _va, _te = fold_split(_f)
        _g = lambda xs, c: sum(gt_by_key[x["key"]]["gt_pred"] == c for x in xs)
        print(f"  fold {_f}: train {len(_tr)} (OK {_g(_tr, 'OK')}/NG {_g(_tr, 'NG')}) | val {len(_va)} | test {len(_te)} (OK {_g(_te, 'OK')}/NG {_g(_te, 'NG')})")
if QUICK_TEST_N is not None and len(eval_items) > QUICK_TEST_N:
    # [changed] stratified: half OK, half NG (a plain random sample of 30 can contain almost one class only)
    _rng = random.Random(SEED)
    _ok_items = [it for it in eval_items if gt_by_key[it["key"]]["gt_pred"] == "OK"]
    _ng_items = [it for it in eval_items if gt_by_key[it["key"]]["gt_pred"] == "NG"]
    _h = QUICK_TEST_N // 2
    eval_items = (_rng.sample(_ok_items, min(_h, len(_ok_items))) + _rng.sample(_ng_items, min(QUICK_TEST_N - _h, len(_ng_items))))
if not eval_items:
    raise RuntimeError(f"No images to score for EVAL_SPLITS={EVAL_SPLITS}.")

_n_gt_ok = sum(gt_by_key[it["key"]]["gt_pred"] == "OK" for it in eval_items)
_n_gt_ng = len(eval_items) - _n_gt_ok
print(f"RUN_PRESET = {RUN_PRESET}  (EVAL_SPLITS = {EVAL_SPLITS}, QUICK_TEST_N = {QUICK_TEST_N}, SKIP_FILTERS = {sorted(SKIP_FILTERS)})")
if RUN_PRESET == "quick":
    print("NOTE: the 'quick' preset is a smoke test, not a final result. Set RUN_PRESET = \"final\" for the paper run.")
print(f"FILTER_MODE = {FILTER_MODE}")
print(f"Images scored per filter: {len(eval_items)}  (GT OK {_n_gt_ok} / GT NG {_n_gt_ng}),  splits = {EVAL_SPLITS}")
if len(eval_items) < 100:
    print(f"WARNING: only {len(eval_items)} images are scored -> accuracy moves in steps of {100 / len(eval_items):.2f} pp and the 95% CI is wide. "
          "Use the whole test split (RUN_PRESET = 'final') or add val to CUSTOM_EVAL_SPLITS.")
if _n_gt_ok == 0 or _n_gt_ng == 0:
    print("WARNING: the ground truth contains only one class -> recall / precision / F1 of the missing class will be 0.0.")


def safe_filename(name):
    return "".join(c if c.isalnum() else "_" for c in name).strip("_").lower()


def apply_filter_gray(gray, filter_fn):
    """Apply a filter to a uint8 gray image. Returns (filtered uint8 gray image, filter time in seconds)."""
    t = to_gpu_tensor(gray)
    _sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = filter_fn(t)
    _sync()
    dt = time.perf_counter() - t0
    return to_gray_np(out), dt


def save_filter_example(name, gray, filtered, scale=0.25, overwrite=True):
    """Save one 'Original | filtered' side-by-side example image per filter (overwrite=False keeps an existing file)."""
    out_path = os.path.join(FILTER_SAMPLE_DIR, f"{safe_filename(name)}_example.png")
    if not overwrite and os.path.isfile(out_path):
        return
    def _prep(img, label):
        vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        vis = cv2.resize(vis, (0, 0), fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        cv2.putText(vis, label, (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 5, cv2.LINE_AA)
        cv2.putText(vis, label, (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)
        return vis
    side = cv2.hconcat([_prep(gray, "Original"), _prep(filtered, name)])
    cv2.imwrite(out_path, side)


def yolo_centers(model, source):
    """Run YOLO on an image path or a BGR ndarray and return the detection-box centres."""
    pred = model.predict(source, imgsz=IMGSZ, conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
    xywh = pred.boxes.xywh.cpu().numpy() if len(pred.boxes) else np.zeros((0, 4))
    return [(float(x), float(y)) for x, y, _, _ in xywh]


PER_IMAGE = {}   # method name -> one dict per image (saved to per_image_predictions.csv, used for the McNemar test)


def mcnemar_exact(b, c):
    """Exact two-sided McNemar test (binomial, p = 0.5) on the discordant pairs.
    b = images the reference judged right and the tested method wrong, c = the opposite."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1))
    return min(1.0, 2 * tail / 2 ** n)      # int / int stays exact for very large n (no float overflow)


def mcnemar_between(ref_name, test_name):
    """Paired comparison of two methods on the same images. 'right' = judgement equals the label-based GT.
    Returns (newly_wrong, newly_correct, p) of test_name relative to ref_name, or (None, None, None)."""
    ref = {r["key"]: r["match"] for r in PER_IMAGE.get(ref_name, [])}
    tst = {r["key"]: r["match"] for r in PER_IMAGE.get(test_name, [])}
    keys = ref.keys() & tst.keys()
    if not keys:
        return None, None, None
    b = sum(ref[k] == 1 and tst[k] == 0 for k in keys)
    c = sum(ref[k] == 0 and tst[k] == 1 for k in keys)
    return b, c, mcnemar_exact(b, c)


def summarize(name, rows, filter_ms, yolo_ms):
    """rows: one dict per image with gt_pred / pred / gt_counts / counts. NG is the positive class."""
    n = len(rows)
    tp = sum(r["gt_pred"] == "NG" and r["pred"] == "NG" for r in rows)
    fn = sum(r["gt_pred"] == "NG" and r["pred"] == "OK" for r in rows)
    fp = sum(r["gt_pred"] == "OK" and r["pred"] == "NG" for r in rows)
    tn = sum(r["gt_pred"] == "OK" and r["pred"] == "OK" for r in rows)
    ng_recall, ok_recall = _pct(tp, tp + fn), _pct(tn, tn + fp)
    precision = _pct(tp, tp + fp)
    f1 = 2 * precision * ng_recall / (precision + ng_recall) if (precision + ng_recall) else 0.0
    reg = {k: _pct(sum(r["counts"][k] == r["gt_counts"][k] for r in rows), n) for k in REGION_RULE}
    PER_IMAGE[name] = [dict(method=name, key=r["key"], split=r["split"], gt_pred=r["gt_pred"], pred=r["pred"],
                            match=int(r["pred"] == r["gt_pred"]),
                            **{f"count_{k}": r["counts"][k] for k in REGION_RULE},
                            **{k: r[k] for k in ("fold", "n_gt", "miss", "extra", "loc_err_px", "min_conf", "ref_label") if k in r}) for r in rows]
    # [added] continuous detection-quality metrics (they do not saturate at 100% like the OK/NG accuracy)
    _vals = lambda k: [r[k] for r in rows if r.get(k) is not None]
    n_gt = sum(r.get("n_gt", 0) for r in rows)
    nan = float("nan")
    ball_recall = 100.0 * (1 - sum(r.get("miss", 0) for r in rows) / n_gt) if n_gt else nan
    extra_per_img = float(np.mean(_vals("extra"))) if _vals("extra") else nan
    loc_err_px = float(np.mean(_vals("loc_err_px"))) if _vals("loc_err_px") else nan
    min_conf = 100.0 * float(np.mean(_vals("min_conf"))) if _vals("min_conf") else nan
    # [added v5] numbers that put the plain accuracy in context
    gt_ok_pct = _pct(tn + fp, n)
    _sr = [r for r in rows if r.get("miss") is not None and r.get("extra") is not None]
    strict_acc = _pct(sum(r["miss"] == 0 and r["extra"] == 0 for r in _sr), len(_sr)) if _sr else nan     # every labelled ball found, no extra detection anywhere
    _rr = [r for r in rows if r.get("ref_label") in ("OK", "NG")]
    ref_acc = _pct(sum(r["pred"] == r["ref_label"] for r in _rr), len(_rr)) if _rr else nan                # agreement with the independent OK/NG list
    gt_vs_ref_acc = _pct(sum(r["gt_pred"] == r["ref_label"] for r in _rr), len(_rr)) if _rr else nan       # how well the label rule itself matches that list
    nw = nc = mp = None
    if name != "Original":
        nw, nc, mp = mcnemar_between("Original", name)
    return dict(name=name, n_images=n, accuracy=_pct(tp + tn, n), bal_acc=(ng_recall + ok_recall) / 2,
                gt_ok_pct=gt_ok_pct, majority_acc=max(gt_ok_pct, 100 - gt_ok_pct), strict_acc=strict_acc,
                ref_acc=ref_acc, ref_n=len(_rr), gt_vs_ref_acc=gt_vs_ref_acc,
                pred_ng_pct=_pct(tp + fp, n), ball_recall=ball_recall, extra_per_img=extra_per_img,
                loc_err_px=loc_err_px, min_conf=min_conf,
                top_acc=reg["top"], left_acc=reg["left"], bottom_acc=reg["bottom"], right_acc=reg["right"],
                ok_recall=ok_recall, ng_recall=ng_recall, precision=precision, f1=f1,
                avg_time_ms=filter_ms, yolo_time_ms=yolo_ms, tp=tp, fn=fn, fp=fp, tn=tn,
                newly_wrong=nw, newly_correct=nc, mcnemar_p=mp)


REF_LABEL = {}      # [added v5] independent OK/NG list (OKNG_CSV: name,label), keyed by image stem
if OKNG_CSV and os.path.isfile(OKNG_CSV):
    with open(OKNG_CSV, newline="", encoding="utf-8-sig") as _f:
        for _r in csv.DictReader(_f):
            REF_LABEL[os.path.splitext(_r["name"].strip())[0]] = _r["label"].strip().upper()
    print(f"OKNG_CSV: {len(REF_LABEL)} reference labels loaded")


def _row(it, pts):
    gt = gt_by_key[it["key"]]
    j = judge_boxes(pts, gt["part"], (gt["W"], gt["H"]))
    return dict(key=it["key"], split=it["split"], gt_pred=gt["gt_pred"], pred=j["pred"],
                gt_counts=gt["gt_counts"], counts=j["counts"], ref_label=REF_LABEL.get(it["stem"]))


# ── Mode 1: filter at judgement time, reuse the YOLO trained on the original images ──
class _Progress:
    """Prints a status line at most every `every` seconds (and at the end) with elapsed time and a rough ETA,
    so that a long run does not look like a frozen notebook."""

    def __init__(self, tag, total, every=30):
        self.tag, self.total, self.every = tag, total, every
        self.t0 = self.last = time.time()

    def update(self, done):
        now = time.time()
        if done == self.total or now - self.last >= self.every:
            self.last = now
            el = now - self.t0
            eta = el / done * (self.total - done) if done else 0.0
            print(f"    [{self.tag}] {done}/{self.total} images | {el / 60:.1f} min elapsed | ~{eta / 60:.1f} min left")


def evaluate_inference_only(name, filter_fn, items, model, save_example=True):
    rows, t_filter, t_yolo, example_saved = [], 0.0, 0.0, False
    prog = _Progress(name, len(items))
    for i, it in enumerate(items):
        gray = cv2.imread(it["img"], cv2.IMREAD_GRAYSCALE)
        filtered, dt = apply_filter_gray(gray, filter_fn)
        t_filter += dt
        if save_example and not example_saved and name != "Original":
            save_filter_example(name, gray, filtered, overwrite=False)   # Section 13 already saved the sample
            example_saved = True
        bgr = cv2.cvtColor(filtered, cv2.COLOR_GRAY2BGR)
        t0 = time.perf_counter()
        pts = yolo_centers(model, bgr)
        t_yolo += time.perf_counter() - t0
        rows.append(_row(it, pts))
        prog.update(i + 1)
    return summarize(name, rows, t_filter / len(items) * 1000, t_yolo / len(items) * 1000)


# ── Mode 2: filter the whole dataset, train one YOLO per filter ──
FILTER_WEIGHTS_JSON = os.path.join(RUNS_DIR, "filter_weights.json")


def _filtered_yaml(root):
    try:
        with open(_yaml_path, encoding="utf-8") as f:
            cfg = yaml.safe_load(f) or {}
    except Exception:
        cfg = {}
    cfg.setdefault("names", {0: "ball"})
    cfg["path"] = os.path.abspath(root)
    for k in ("train", "val", "test"):
        cfg.pop(k, None)
    for sp in sorted({it["split"] for it in dataset}):
        cfg[sp] = f"images/{sp}"
    path = os.path.join(root, "data.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    return path


def filtered_root_for(name, filter_fn):
    """Folder of the filtered dataset. A filter object with `root_key` (see SAME in Section 22) shares one folder."""
    key = getattr(filter_fn, "root_key", None) or filter_cache_key(name, filter_fn)
    return os.path.join(FILTERED_ROOT, key + _STRESS_TAG)


def build_filtered_dataset(name, filter_fn):
    """Write the filtered copy of every image (+ its label file) and return (root, avg filter time in ms)."""
    root = filtered_root_for(name, filter_fn)
    meta_path = os.path.join(root, "meta.json")
    all_items = [it for it in dataset if it["key"] in gt_by_key]
    if os.path.isfile(meta_path):
        with open(meta_path, encoding="utf-8") as f:
            meta = json.load(f)
        if meta.get("n_images") == len(all_items):
            print(f"    [{name}] reusing the existing filtered dataset: {root}")
            return root, meta["avg_filter_ms"]

    total, example_saved = 0.0, False
    prog = _Progress(f"{name} filtering", len(all_items))
    for i, it in enumerate(all_items):
        img_out = os.path.join(root, "images", it["split"], it["stem"] + ".png")
        lab_out = os.path.join(root, "labels", it["split"], it["stem"] + ".txt")
        os.makedirs(os.path.dirname(img_out), exist_ok=True)
        os.makedirs(os.path.dirname(lab_out), exist_ok=True)
        shutil.copyfile(it["lab"], lab_out)
        gray = _noisy(cv2.imread(it["img"], cv2.IMREAD_GRAYSCALE), it["key"])
        filtered, dt = apply_filter_gray(gray, filter_fn)
        total += dt
        cv2.imwrite(img_out, filtered, [cv2.IMWRITE_PNG_COMPRESSION, 1])
        if not example_saved:
            save_filter_example(name, gray, filtered, overwrite=False)   # Section 13 already saved the sample
            example_saved = True
        prog.update(i + 1)
    avg_ms = total / len(all_items) * 1000
    _filtered_yaml(root)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(dict(n_images=len(all_items), avg_filter_ms=avg_ms), f)
    return root, avg_ms


def get_filter_weights(name, root):
    """Train a YOLO on the filtered dataset (or reuse weights from an earlier run)."""
    cache = {}
    if os.path.isfile(FILTER_WEIGHTS_JSON):
        with open(FILTER_WEIGHTS_JSON, encoding="utf-8") as f:
            cache = json.load(f)
    key = os.path.basename(os.path.normpath(root))     # filter name + a hash of its parameters
    key += (f"_os{OVERSAMPLE_MAX_FACTOR}" if OVERSAMPLE else "") + f"_e{FILTER_EPOCHS}"   # [added] a changed recipe never reuses old weights
    w = cache.get(key)
    if w and os.path.isfile(w):
        print(f"    [{name}] reusing trained weights: {w}")
        return w
    m = YOLO(MODEL_SIZE)
    data_yaml = _filtered_yaml(root)
    if OVERSAMPLE:     # [added] same oversampling as the Original model, on the filtered train images
        pairs = [(it["key"], os.path.join(root, "images", "train", it["stem"] + ".png"),
                  os.path.join(root, "labels", "train", it["stem"] + ".txt"))
                 for it in dataset if it["split"] == "train" and it["key"] in gt_by_key]
        if pairs:
            _v = os.path.join(os.path.abspath(root), "images", "val")
            if not os.path.isdir(_v):
                _v = os.path.join(os.path.abspath(root), "images", "train")
            data_yaml, _ = build_oversampled_dataset(pairs, os.path.abspath(root) + "_os", _v)
    res = m.train(data=data_yaml, epochs=FILTER_EPOCHS, imgsz=IMGSZ, batch=BATCH,
                  project=RUNS_DIR, name=f"ball_detect_{key}", seed=SEED, patience=15, **AUG)
    w = str(res.save_dir / "weights" / "best.pt")
    cache[key] = w
    os.makedirs(RUNS_DIR, exist_ok=True)
    with open(FILTER_WEIGHTS_JSON, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2)
    return w


def evaluate_train_per_filter(name, filter_fn, items):
    root, filter_ms = build_filtered_dataset(name, filter_fn)
    model = YOLO(get_filter_weights(name, root))
    rows, t_yolo = [], 0.0
    for i, it in enumerate(items):
        img_path = os.path.join(root, "images", it["split"], it["stem"] + ".png")
        t0 = time.perf_counter()
        pts = yolo_centers(model, img_path)
        t_yolo += time.perf_counter() - t0
        rows.append(_row(it, pts))
        if (i + 1) % 100 == 0 or (i + 1) == len(items):
            print(f"    [{name}] {i + 1}/{len(items)} images")
    return summarize(name, rows, filter_ms, t_yolo / len(items) * 1000)


# ── [added] Mode 3: stratified K-fold CV (USE_CV) ──────────────────────────────────────────────
def yolo_predict_full(model, source, imgsz):
    pred = model.predict(source, imgsz=imgsz, conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
    xywh = pred.boxes.xywh.cpu().numpy() if len(pred.boxes) else np.zeros((0, 4))
    conf = pred.boxes.conf.cpu().numpy() if len(pred.boxes) else np.zeros(0)
    return xywh, conf


def ball_match_stats(gt_boxes, xywh, conf, tol_frac=0.5):
    """Detection quality of one image, independent of the OK/NG rule. A prediction matches a labelled ball when
    its centre is within tol_frac x box size of the label centre (greedy nearest-first, one-to-one).
    miss = labelled balls nobody found, extra = predictions without a ball, loc_err_px = mean centre error of the
    matches, min_conf = lowest confidence among the matches (0 if a ball was missed) = how close the image was to failing."""
    G = np.array([[b[0], b[1]] for b in gt_boxes], dtype=float).reshape(-1, 2)
    gsz = np.array([max(b[2], b[3]) for b in gt_boxes], dtype=float)
    P = np.asarray(xywh, dtype=float).reshape(-1, 4)[:, :2]
    conf = np.asarray(conf, dtype=float)
    if len(G) == 0:
        return dict(n_gt=0, miss=0, extra=len(P), loc_err_px=None, min_conf=None)
    cand = sorted((float(np.hypot(*(G[i] - P[j]))), i, j) for i in range(len(G)) for j in range(len(P)))
    used_g, used_p, errs, confs = set(), set(), [], []
    for d, i, j in cand:
        if i in used_g or j in used_p or d > tol_frac * gsz[i]:
            continue
        used_g.add(i); used_p.add(j); errs.append(d); confs.append(float(conf[j]))
    miss = len(G) - len(used_g)
    return dict(n_gt=len(G), miss=miss, extra=len(P) - len(used_p),
                loc_err_px=float(np.mean(errs)) if errs else None,
                min_conf=0.0 if (miss or not confs) else float(min(confs)))


def build_fold_dataset(root, fold):
    """Symlinks the filtered images of `root` into <root>_cv<K>f<fold>/{train,val,test} (train is oversampled).
    Returns (data.yaml path, info)."""
    train, val, test = fold_split(fold)
    dst = os.path.abspath(root) + f"_cv{CV_FOLDS}f{fold}"

    def _pairs(items):
        return [(it["key"], os.path.join(root, "images", it["split"], it["stem"] + ".png"),
                 os.path.join(root, "labels", it["split"], it["stem"] + ".txt")) for it in items]

    for sub, items in (("val", val), ("test", test)):
        idir, ldir = os.path.join(dst, "images", sub), os.path.join(dst, "labels", sub)
        for d in (idir, ldir):
            shutil.rmtree(d, ignore_errors=True)
            os.makedirs(d, exist_ok=True)
        for _, img, lab in _pairs(items):
            _link_or_copy(img, os.path.join(idir, os.path.basename(img)))
            shutil.copyfile(lab, os.path.join(ldir, os.path.basename(lab)))
    yaml_path, info = build_oversampled_dataset(_pairs(train), dst, os.path.join(dst, "images", "val"),
                                                max_factor=OVERSAMPLE_MAX_FACTOR if OVERSAMPLE else 1)
    with open(yaml_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    cfg["test"] = os.path.join(dst, "images", "test")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    return yaml_path, info


def get_fold_weights(name, root, fold):
    """Train (or reuse) the detector of one fold. Returns (weights path, data.yaml path)."""
    data_yaml, info = build_fold_dataset(root, fold)
    cache = {}
    if os.path.isfile(FILTER_WEIGHTS_JSON):
        with open(FILTER_WEIGHTS_JSON, encoding="utf-8") as f:
            cache = json.load(f)
    key = (f"{os.path.basename(os.path.normpath(root))}_cv{CV_FOLDS}f{fold}{FOLD_TAG}_os{OVERSAMPLE_MAX_FACTOR if OVERSAMPLE else 0}"
           f"_e{FILTER_EPOCHS}_tf{STRESS_TRAIN_FRAC:g}_sz{CV_IMGSZ}_v{CV_VAL_FRAC:g}")
    w = cache.get(key)
    if w and os.path.isfile(w):
        print(f"    [{name}] fold {fold}: reusing trained weights")
        return w, data_yaml
    print(f"    [{name}] fold {fold}: training on {info['n_after']} images ...")
    m = YOLO(MODEL_SIZE)
    res = m.train(data=data_yaml, epochs=FILTER_EPOCHS, imgsz=CV_IMGSZ, batch=BATCH, project=RUNS_DIR,
                  name=f"ball_detect_{key}", seed=SEED, patience=15, **AUG)
    w = str(res.save_dir / "weights" / "best.pt")
    cache[key] = w
    os.makedirs(RUNS_DIR, exist_ok=True)
    with open(FILTER_WEIGHTS_JSON, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2)
    return w, data_yaml


def fold_map(model, data_yaml):
    """mAP50 / mAP50-95 (%) of a fold's detector on the fold's held-out test images."""
    try:
        v = model.val(data=data_yaml, split="test", imgsz=CV_IMGSZ, batch=BATCH, conf=0.001, iou=0.7,
                      verbose=False, plots=False, project=RUNS_DIR, name="cv_val", exist_ok=True)
        return 100.0 * float(v.box.map50), 100.0 * float(v.box.map)
    except Exception as e:
        print(f"    mAP skipped ({type(e).__name__}: {e})")
        return float("nan"), float("nan")


def read_yolo_history(weights_path):
    """Per-epoch curves of a finished YOLO run (results.csv next to weights/). Returns None when it is missing."""
    csv_path = os.path.join(os.path.dirname(os.path.dirname(str(weights_path))), "results.csv")
    if not os.path.isfile(csv_path):
        return None
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        return None
    df.columns = [c.strip() for c in df.columns]

    def col(name):
        return [float(x) for x in df[name].tolist()] if name in df.columns else []
    return dict(loss=col("train/box_loss"), val_loss=col("val/box_loss"),
                map50=col("metrics/mAP50(B)"), map50_95=col("metrics/mAP50-95(B)"))


def _nanmean(xs):
    xs = [x for x in xs if x is not None and not math.isnan(x)]
    return float(np.mean(xs)) if xs else float("nan")


def evaluate_cv(name, filter_fn, items):
    """Each image is judged by the detector of the fold in which it is the held-out test image."""
    root, filter_ms = build_filtered_dataset(name, filter_fn)
    by_fold = defaultdict(list)
    for it in items:
        by_fold[FOLD_OF[it["key"]]].append(it)
    rows, t_yolo, fold_acc, m50, m5095 = [], 0.0, [], [], []
    histories = {}
    for fold in sorted(by_fold):
        w, data_yaml = get_fold_weights(name, root, fold)
        hist = read_yolo_history(w)
        if hist:
            histories[fold] = hist
        model = YOLO(w)
        a, b = fold_map(model, data_yaml)
        m50.append(a); m5095.append(b)
        fold_rows = []
        for it in by_fold[fold]:
            img_path = os.path.join(root, "images", it["split"], it["stem"] + ".png")
            t0 = time.perf_counter()
            xywh, conf = yolo_predict_full(model, img_path, CV_IMGSZ)
            t_yolo += time.perf_counter() - t0
            row = _row(it, [(float(x), float(y)) for x, y, _, _ in xywh])
            row.update(fold=fold, **ball_match_stats(gt_by_key[it["key"]]["gt_boxes"], xywh, conf))
            fold_rows.append(row)
        fold_acc.append(_pct(sum(r["pred"] == r["gt_pred"] for r in fold_rows), len(fold_rows)))
        print(f"    [{name}] fold {fold}: accuracy {fold_acc[-1]:.2f}% | mAP50-95 {b:.1f}% | {len(fold_rows)} held-out images")
        rows += fold_rows
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    r = summarize(name, rows, filter_ms, t_yolo / len(rows) * 1000)
    r.update(map50=_nanmean(m50), map50_95=_nanmean(m5095), fold_acc=fold_acc, fold_std=float(np.std(fold_acc)),
             histories=histories)
    return r


def _result_line(r):
    s = (f"accuracy {r['accuracy']:.2f}% (majority baseline {r.get('majority_acc', float('nan')):.1f}%) | balanced {r['bal_acc']:.2f}% | NG recall {r['ng_recall']:.2f}% | "
         f"predicted NG {r['pred_ng_pct']:.1f}%")
    if r.get("strict_acc") is not None and not math.isnan(r["strict_acc"]):
        s += f" | strict {r['strict_acc']:.2f}%"
    if r.get("map50_95") is not None and not math.isnan(r["map50_95"]):
        s += f" | mAP50-95 {r['map50_95']:.1f}% | ball recall {r['ball_recall']:.2f}% | center err {r['loc_err_px']:.2f}px"
    return s + f" | filter {r['avg_time_ms']:.2f} ms/img"


# ── Resume support: every finished method is saved and skipped when the notebook is run again ──
RESULT_CACHE_DIR = os.path.join(FILTER_FIG_DIR, "result_cache")
REUSE_RESULTS = True      # False -> evaluate everything again (or delete the result_cache folder)
_EVAL_SIG = hashlib.md5(json.dumps(
    [[it["key"], gt_by_key[it["key"]]["gt_pred"], gt_by_key[it["key"]]["gt_counts"]] for it in eval_items],
    sort_keys=True).encode("utf-8")).hexdigest()


def _weights_id():
    p = getattr(yolo_model, "ckpt_path", None)
    if p and os.path.isfile(str(p)):
        st = os.stat(str(p))
        return f"{p}:{st.st_size}:{int(st.st_mtime)}"
    return str(p or "")


def _result_path_and_sig(name, filter_fn):
    """Cache file + signature of everything that decides a result: images and ground truth, mode, detector,
    thresholds and the filter's parameters. A changed signature means the saved result is ignored."""
    sig = json.dumps(dict(
        v=4, name=name, mode=FILTER_MODE, eval=_EVAL_SIG, weights=("cv" if USE_CV else _weights_id()), conf=CONF_THRES, iou=IOU_THRES,
        imgsz=CV_IMGSZ if USE_CV else IMGSZ,
        cv=dict(folds=CV_FOLDS, val=CV_VAL_FRAC, train_frac=STRESS_TRAIN_FRAC, noise=STRESS_NOISE_STD, **({"fold": FOLD_TAG} if FOLD_TAG else {})) if USE_CV else None,
        filt=filter_fn.signature() if hasattr(filter_fn, "signature") else "",
        train=dict(epochs=FILTER_EPOCHS, batch=BATCH, model=MODEL_SIZE, oversample=OVERSAMPLE, aug=AUG) if FILTER_MODE == "train_per_filter" else None), sort_keys=True)
    tag = hashlib.md5(sig.encode("utf-8")).hexdigest()[:8]
    return os.path.join(RESULT_CACHE_DIR, f"{filter_cache_key(name, filter_fn)}_{tag}.json"), sig


def cached_result(name, filter_fn):
    """The saved result of a method, or None. Also restores its per-image rows for the McNemar test."""
    if not REUSE_RESULTS:
        return None
    path, sig = _result_path_and_sig(name, filter_fn)
    if not os.path.isfile(path):
        return None
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    if d.get("signature") != sig:
        return None
    PER_IMAGE[name] = d["rows"]
    r = d["result"]
    if name != "Original":     # the McNemar numbers refer to the Original rows that are loaded / computed in this run
        r["newly_wrong"], r["newly_correct"], r["mcnemar_p"] = mcnemar_between("Original", name)
    return r


def save_result(name, filter_fn, r):
    path, sig = _result_path_and_sig(name, filter_fn)
    os.makedirs(RESULT_CACHE_DIR, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(dict(signature=sig, result=r, rows=PER_IMAGE[name]), f, ensure_ascii=False)


def filter_params_str(filter_fn):
    """Human-readable parameters of a filter (shown in the tables)."""
    return filter_fn.describe() if hasattr(filter_fn, "describe") else "-"


def filter_cache_key(name, filter_fn):
    """Folder / weights key: the filter name plus a short hash of its parameters."""
    sig = filter_fn.signature() if hasattr(filter_fn, "signature") else ""
    return safe_filename(name) + (("_" + hashlib.md5(sig.encode("utf-8")).hexdigest()[:8]) if sig else "")


def run_round(round_tag, filters):
    results_round, t_round = [], time.time()
    for name, fn in filters:
        if name in SKIP_FILTERS:
            print(f"\n[{round_tag}] {name}: skipped (SKIP_FILTERS)")
            continue
        r = cached_result(name, fn)
        if r is not None:
            print(f"\n[{round_tag}] {name}: reusing the saved result ({os.path.basename(RESULT_CACHE_DIR)}/, REUSE_RESULTS = True)")
            print("  -> " + _result_line(r))
            results_round.append(r)
            continue
        print(f"\n[{round_tag}] {name} ...")
        t0 = time.time()
        if FILTER_MODE == "inference_only":
            r = evaluate_inference_only(name, fn, eval_items, yolo_model)
        elif FILTER_MODE == "train_per_filter":
            r = evaluate_cv(name, fn, eval_items) if USE_CV else evaluate_train_per_filter(name, fn, eval_items)
        else:
            raise ValueError(f"Unknown FILTER_MODE: {FILTER_MODE!r}")
        r["params"] = filter_params_str(fn)
        save_result(name, fn, r)
        results_round.append(r)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("  -> " + _result_line(r) + f" | {(time.time() - t0) / 60:.1f} min")
        if r["pred_ng_pct"] >= 95 or r["pred_ng_pct"] <= 5:
            print("     WARNING: almost every image got the same judgement -> the detector collapsed (accuracy = class share, not skill)")
    print(f"\n{round_tag} finished in {(time.time() - t_round) / 60:.1f} min")
    return results_round

## 12. Graphs and tables

Each round is drawn with the same 10 metrics (10 graphs per round, 20 in total), using `plot_clean_bar` in the same style as before. The dotted grey line in every graph is the **Original (no filter)** result, so you can see at a glance which filters are above / below the baseline. Alongside the graphs, each round produces a table (shown below and saved as CSV).

Region accuracy = share of images where the number of detected balls in that region equals the ground-truth count (from the label boxes). OK / NG class accuracy = recall of that class. Precision / F1 are for the NG class. Processing time = average filter time per image (the YOLO time is in the table). Each table also shows how many images became newly correct / newly wrong compared with Original, an exact McNemar p-value on those paired images, and a 95% Wilson confidence interval for the accuracy (a difference smaller than the interval width is easily chance) (with ~20 comparisons a stricter threshold such as 0.05 / 20 = 0.0025 is safer than 0.05).

In [ ]:
METRIC_SPECS = [
    dict(key="accuracy", ylabel="Overall Accuracy (%)", fname="01_overall_accuracy"),
    dict(key="top_acc", ylabel="Top Region Accuracy (%)", fname="02_top_region_accuracy"),
    dict(key="left_acc", ylabel="Left Region Accuracy (%)", fname="03_left_region_accuracy"),
    dict(key="bottom_acc", ylabel="Bottom Region Accuracy (%)", fname="04_bottom_region_accuracy"),
    dict(key="right_acc", ylabel="Right Region Accuracy (%)", fname="05_right_region_accuracy"),
    dict(key="ok_recall", ylabel="OK Class Accuracy (%)", fname="06_ok_class_accuracy"),
    dict(key="ng_recall", ylabel="NG Class Accuracy (%)", fname="07_ng_class_accuracy"),
    dict(key="precision", ylabel="Precision (%)", fname="08_precision"),
    dict(key="f1", ylabel="F1 Score (%)", fname="09_f1_score"),
    dict(key="avg_time_ms", ylabel="Avg Filter Time (ms/image)", fname="10_avg_filter_time", time_metric=True),
    # [added] continuous metrics that do not saturate at 100% (only drawn when the run produced them, i.e. in CV mode)
    dict(key="ball_recall", ylabel="Ball Recall (%)", fname="11_ball_recall"),
    dict(key="map50_95", ylabel="mAP50-95 (%)", fname="12_map50_95"),
    dict(key="min_conf", ylabel="Mean Min. Confidence (%)", fname="13_min_confidence"),
    dict(key="loc_err_px", ylabel="Mean Center Error (px)", fname="14_center_error", time_metric=True),
    dict(key="strict_acc", ylabel="Strict Accuracy (%)", fname="15_strict_accuracy"),   # [added v5]
]


def make_all_metric_plots(results_round, round_tag, baseline):
    names = [r["name"] for r in results_round]
    n_bars = len(names)
    paths = []
    for spec in METRIC_SPECS:
        values = [r.get(spec["key"]) for r in results_round]
        if any(v is None or (isinstance(v, float) and math.isnan(v)) for v in values + [baseline.get(spec["key"])]):
            continue                                  # metric not available in this run
        is_time = spec.get("time_metric", False)
        paths.append(plot_clean_bar(
            labels=names, values=values,
            xlabel="Filter", ylabel=spec["ylabel"], save_name=f"fig_{round_tag}_{spec['fname']}.png",
            value_fmt="{:.2f}" if is_time else "{:.2f}%",
            ymax=None if is_time else 110,
            threshold=None if is_time else baseline[spec["key"]],
            threshold_label=None if is_time else f"Original {baseline[spec['key']]:.2f}%",
            rotation=35 if n_bars >= 5 else 12,
            figsize=(13.5, 6.6) if n_bars >= 5 else (11, 5.8),
            higher_is_better=not is_time,
            out_dir=FILTER_FIG_DIR,
            legend_above=True,
            value_fontsize=14 if n_bars >= 8 else 17,   # 10 bars: 17 pt value labels touch each other
        ))
    return paths


TABLE_COLUMNS = [
    ("name", "Filter"), ("accuracy", "Accuracy (%)"),
    ("top_acc", "Top (%)"), ("left_acc", "Left (%)"), ("bottom_acc", "Bottom (%)"), ("right_acc", "Right (%)"),
    ("bal_acc", "Balanced Acc (%)"), ("pred_ng_pct", "Predicted NG (%)"), ("ok_recall", "OK Recall (%)"), ("ng_recall", "NG Recall (%)"), ("precision", "Precision (%)"), ("f1", "F1 (%)"),
    ("ball_recall", "Ball Recall (%)"), ("map50", "mAP50 (%)"), ("map50_95", "mAP50-95 (%)"), ("min_conf", "Min Conf (%)"),
    ("loc_err_px", "Center Err (px)"), ("extra_per_img", "Extra Det./img"), ("fold_std", "Fold Acc Std (pp)"),
    ("strict_acc", "Strict Acc (%)"), ("majority_acc", "Majority-class Baseline (%)"),
    ("ref_acc", "Acc vs OKNG_CSV (%)"), ("gt_vs_ref_acc", "Label Rule vs OKNG_CSV (%)"),
    ("avg_time_ms", "Filter Time (ms/img)"), ("yolo_time_ms", "YOLO Time (ms/img)"),
    ("newly_correct", "Newly Correct vs Original"), ("newly_wrong", "Newly Wrong vs Original"),
    ("mcnemar_p", "McNemar p vs Original"), ("params", "Parameters"),
]


def wilson_ci(k, n, z=1.96):
    """95% Wilson score interval (in %) for k successes out of n."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * (centre - half), 100 * (centre + half)


def results_to_df(results_round, baseline):
    """Table with the Original baseline as the first row."""
    rows = [baseline] + list(results_round)
    df = pd.DataFrame([{label: r.get(key) for key, label in TABLE_COLUMNS} for r in rows])
    df["Parameters"] = df["Parameters"].fillna("-")
    for col in ("Strict Acc (%)", "Majority-class Baseline (%)", "Acc vs OKNG_CSV (%)", "Label Rule vs OKNG_CSV (%)"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    if df["Acc vs OKNG_CSV (%)"].isna().all():            # no OKNG_CSV -> hide the two reference columns
        df = df.drop(columns=["Acc vs OKNG_CSV (%)", "Label Rule vs OKNG_CSV (%)"])
    df.insert(2, "Accuracy 95% CI", ["%.1f - %.1f" % wilson_ci(r["tp"] + r["tn"], r["n_images"]) for r in rows])
    df.insert(2, "Acc. change vs Original (pp)", df["Accuracy (%)"] - baseline["accuracy"])
    for col in ("Newly Correct vs Original", "Newly Wrong vs Original"):
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    df["McNemar p vs Original"] = pd.to_numeric(df["McNemar p vs Original"], errors="coerce")
    df["Significant (p<0.05)"] = ["-" if pd.isna(p) else ("Yes" if p < 0.05 else "No") for p in df["McNemar p vs Original"]]
    lead = ["Filter", "Parameters", "Accuracy (%)", "Accuracy 95% CI", "Acc. change vs Original (pp)", "Newly Correct vs Original",
            "Newly Wrong vs Original", "McNemar p vs Original", "Significant (p<0.05)"]
    df = df[lead + [col for col in df.columns if col not in lead]]
    return df.round({col: (4 if col == "McNemar p vs Original" else 2) for col in df.columns
                     if col not in ("Filter", "Parameters", "Accuracy 95% CI", "Significant (p<0.05)", "Newly Correct vs Original", "Newly Wrong vs Original")})


def show_and_save_table(df, round_tag):
    path = os.path.join(FILTER_FIG_DIR, f"table_{round_tag}.csv")
    df.to_csv(path, index=False, encoding="utf-8-sig")
    display(df)
    print(f"Saved: {path}")

## 13. Filter samples, grid figures and runtime estimate (before the experiment)

Every filter (Round 1 and Round 2) is applied to **one sample image** and the result is saved right away, so you can see what each filter does before the long evaluation starts.

- A printed **sanity check** per filter: mean absolute change in gray levels and the share of changed pixels. A filter that leaves the image unchanged is reported with a warning.
- One `Original | filtered` side-by-side image per filter -> `yolo_filter_samples/<filter>_example.png`
- Paper-style grids for Round 1 and Round 2, each in three versions (`grid_original_round1.png`, `..._zoom.png`, `..._diff.png`; the same for round 2):
  - **full view** - the whole image, shrunk to `GRID_PANEL_MAX_DIM` pixels
  - **zoom** - the same panels cropped at the **native resolution** around the balls (`GRID_ZOOM_PX`)
  - **diff** - `|filtered - Original|` in the zoom window, multiplied by `DIFF_GAIN`
- A warm-up call (so the first image of the first filter is not slowed by CUDA / YOLO initialisation) and a **rough runtime estimate** for the experiment, based on the single-image timings measured here

`SAMPLE_KEY` picks the sample image (`None` = first scored image). `GRID_ZOOM` (fractions `(x0, y0, x1, y1)`) overrides the automatic zoom window; `GRID_ZOOM_AUTO = False` skips the zoom and diff figures.


In [ ]:
SAMPLE_KEY = None          # e.g. "test/<stem>"; None -> first scored image
GRID_ZOOM = None           # None = whole image in the base grids; or (x0, y0, x1, y1) as fractions, e.g. (0.3, 0.3, 0.7, 0.7)
GRID_ZOOM_AUTO = False      # also make the native-resolution zoom and the difference grids
GRID_ZOOM_PX = (700, 560)  # size (width, height) in original pixels of the automatic zoom window, centred on the balls
DIFF_GAIN = 1              # |filtered - Original| is multiplied by this in the difference grids (raise it for mild changes)
GRID_PANEL_MAX_DIM = 700   # long side of every grid panel in pixels
GRID_COLS = 4

_eval_by_key = {it["key"]: it for it in eval_items}
if SAMPLE_KEY is not None and SAMPLE_KEY not in _eval_by_key:
    raise KeyError(f"SAMPLE_KEY {SAMPLE_KEY!r} is not among the scored images (EVAL_SPLITS={EVAL_SPLITS}).")
sample_item = _eval_by_key[SAMPLE_KEY] if SAMPLE_KEY is not None else eval_items[0]
_t0 = time.perf_counter()
sample_gray = cv2.imread(sample_item["img"], cv2.IMREAD_GRAYSCALE)
read_sample_ms = (time.perf_counter() - _t0) * 1000
print(f"Sample image: {sample_item['key']}  ({sample_gray.shape[1]}x{sample_gray.shape[0]})")

# warm-up: the first GPU filter call and the first YOLO call are slow (CUDA / cuDNN / model initialisation)
apply_filter_gray(sample_gray, FILTERS_ROUND1[0][1])
_bgr = cv2.cvtColor(sample_gray, cv2.COLOR_GRAY2BGR)
yolo_centers(yolo_model, _bgr)
_t0 = time.perf_counter()
yolo_centers(yolo_model, _bgr)
yolo_sample_ms = (time.perf_counter() - _t0) * 1000
print(f"Single-image timings: read {read_sample_ms:.1f} ms, YOLO {yolo_sample_ms:.1f} ms")

# apply every filter once to the sample image and save the Original-vs-filtered example right away
filter_samples = {}      # filter name -> filtered uint8 image
filter_sample_ms = {}    # filter name -> filter time on the sample (ms)
filter_sample_gb = {}    # filter name -> peak GPU memory on the sample (GB)
filter_sample_stats = {}  # filter name -> (mean |change| in gray levels, % of pixels changed)
for _tag, _filters in (("Round 1", FILTERS_ROUND1), ("Round 2", FILTERS_ROUND2)):
    for _name, _fn in _filters:
        _t0 = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        _filtered, _dt = apply_filter_gray(sample_gray, _fn)
        filter_sample_gb[_name] = torch.cuda.max_memory_allocated() / 1024 ** 3 if torch.cuda.is_available() else 0.0
        _delta = np.abs(_filtered.astype(np.int16) - sample_gray.astype(np.int16))
        filter_sample_stats[_name] = (float(_delta.mean()), 100.0 * float((_delta > 0).mean()))
        filter_samples[_name] = _filtered
        filter_sample_ms[_name] = _dt * 1000
        save_filter_example(_name, sample_gray, _filtered)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  [{_tag}] {_name:<24s} saved  (filter {_dt * 1000:9.1f} ms, peak GPU {filter_sample_gb[_name]:5.2f} GB, total {time.time() - _t0:5.1f} s)")
        print(f"      changed the image: mean |change| {filter_sample_stats[_name][0]:6.2f} gray levels, "
              f"{filter_sample_stats[_name][1]:5.1f}% of the pixels differ" + ("   <-- WARNING: image unchanged" if filter_sample_stats[_name][1] == 0 else ""))
print(f"\n{len(filter_samples)} example images saved -> {os.path.abspath(FILTER_SAMPLE_DIR)}")
_weak = [n for n, (m, _) in filter_sample_stats.items() if m < 1.0]
if _weak:
    print(f"WARNING - filters that changed the sample by less than 1 gray level on average: {_weak}\n"
          "  They should be extreme; check their parameters in FILTER_SPECS.")
if torch.cuda.is_available():
    _total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    _worst = max(filter_sample_gb, key=filter_sample_gb.get)
    print(f"Peak GPU memory on this image: {filter_sample_gb[_worst]:.2f} GB ({_worst}) of {_total_gb:.1f} GB"
          + ("  <- close to the limit: a larger image may run out of memory" if filter_sample_gb[_worst] > 0.8 * _total_gb else ""))


def print_runtime_estimate():
    """Rough estimate: (filter time + YOLO time + image read) measured on the sample x number of scored images."""
    n_img = len(eval_items)
    fixed_ms = yolo_sample_ms + read_sample_ms
    rounds = (("Baseline", [("Original", None)]), ("Round 1", FILTERS_ROUND1),
              ("Round 2", FILTERS_ROUND2))
    print(f"\nRough runtime estimate  (FILTER_MODE = {FILTER_MODE}, {n_img} images per filter, RUN_PRESET = {RUN_PRESET})")
    total, n_methods = 0.0, 0
    for tag, filters in rounds:
        round_min = 0.0
        for name, _ in filters:
            if name in SKIP_FILTERS:
                continue
            n_methods += 1
            m = ((0.0 if name == "Original" else filter_sample_ms[name]) + fixed_ms) * n_img / 1000.0 / 60.0
            round_min += m
            if m >= 1.0:
                print(f"    {name:<24s} ~{m:8.1f} min")
        total += round_min
        print(f"  {tag:<9s} ~{round_min:8.1f} min")
    print(f"  {'Total':<9s} ~{total:8.1f} min  (~{total / 60:.1f} h)   [single-image based; the real time also depends on disk and GPU load]")
    if FILTER_MODE == "train_per_filter":
        if USE_CV:
            print(f"  NOTE: CV adds {n_methods * len(_run_folds)} YOLO trainings ({n_methods} methods x {len(_run_folds)} folds, "
                  f"{FILTER_EPOCHS} epochs each; about 2 min each for 360 images on an RTX 4090). Finished folds are reused on re-runs.")
        else:
            print(f"  NOTE: train_per_filter adds one YOLO training per filter ({n_methods - 1} trainings) on top of this.")


print_runtime_estimate()


def _auto_zoom_fractions():
    """A native-resolution window around the balls: the top-most ball and its 4 nearest neighbours."""
    H, W = sample_gray.shape
    zw, zh = min(GRID_ZOOM_PX[0], W), min(GRID_ZOOM_PX[1], H)
    boxes = gt_by_key[sample_item["key"]].get("gt_boxes") or []
    if boxes:
        pts = np.array([(b[0], b[1]) for b in boxes], dtype=float)
        top = pts[np.argmin(pts[:, 1])]
        near = pts[np.argsort(np.hypot(pts[:, 0] - top[0], pts[:, 1] - top[1]))[:5]]
        cx, cy = near.mean(axis=0)
    else:
        cx, cy = W / 2, H / 2
    x0 = int(np.clip(cx - zw / 2, 0, W - zw))
    y0 = int(np.clip(cy - zh / 2, 0, H - zh))
    return (x0 / W, y0 / H, (x0 + zw) / W, (y0 + zh) / H)


def _grid_panel(img, zoom=None):
    zoom = GRID_ZOOM if zoom is None else zoom
    if zoom is not None:
        h, w = img.shape
        x0, y0, x1, y1 = zoom
        img = img[int(y0 * h):int(y1 * h), int(x0 * w):int(x1 * w)]
    h, w = img.shape
    s = GRID_PANEL_MAX_DIM / max(h, w)
    if s < 1.0:
        img = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
    return img


def make_grid_figure(filter_names, save_name, mode="image", zoom=None, ncols=GRID_COLS):
    """Grid of (a) Original followed by the given filters, all applied to the same sample image.
    mode="image": the filtered images; mode="diff": |filtered - Original| multiplied by DIFF_GAIN."""
    panels = [("Original", sample_gray)]
    for n in filter_names:
        if mode == "diff":
            d = np.abs(filter_samples[n].astype(np.int16) - sample_gray.astype(np.int16))
            panels.append((f"{n}\n(mean change {filter_sample_stats[n][0]:.2f})", np.clip(d * DIFF_GAIN, 0, 255).astype(np.uint8)))
        else:
            panels.append((n, filter_samples[n]))
    nrows = math.ceil(len(panels) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.6 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, (ax, (label, img)) in enumerate(zip(axes, panels)):
        ax.imshow(_grid_panel(img, zoom), cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"({chr(97 + i)}) {label}", fontsize=16, fontweight="bold")
        ax.axis("off")
    for ax in axes[len(panels):]:
        ax.axis("off")
    fig.tight_layout()
    out_path = os.path.join(FILTER_SAMPLE_DIR, save_name)
    save_reference_figure(fig, out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {out_path}")
    return out_path


_r1_names = [n for n, _ in FILTERS_ROUND1]
_r2_names = [n for n, _ in FILTERS_ROUND2]
grid_round1_path = make_grid_figure(_r1_names, "grid_original_round1.png")
grid_round2_path = make_grid_figure(_r2_names, "grid_original_round2.png")
if GRID_ZOOM_AUTO:
    _zoom = GRID_ZOOM if GRID_ZOOM is not None else _auto_zoom_fractions()
    print(f"Zoom window (fractions x0, y0, x1, y1): {tuple(round(v, 3) for v in _zoom)}")
    for _tag, _names in (("round1", _r1_names), ("round2", _r2_names)):
        make_grid_figure(_names, f"grid_original_{_tag}_zoom.png", zoom=_zoom)
        make_grid_figure(_names, f"grid_original_{_tag}_diff.png", mode="diff", zoom=_zoom)


## 14. Baseline - Original (no filter)

Scored with the same function and the same images as every filter (image -> gray -> YOLO), so the comparison is like for like.

In [ ]:
print("=" * 70)
print(f"Baseline: Original (no filter) x {len(eval_items)} images")
print("=" * 70)
baseline_result = cached_result("Original", identity_filter)
if baseline_result is None:
    if USE_CV:     # same recipe as every filter: train per fold on the unfiltered images, score the held-out fold
        baseline_result = evaluate_cv("Original", identity_filter, eval_items)
    else:
        baseline_result = evaluate_inference_only("Original", identity_filter, eval_items, yolo_model, save_example=False)
    save_result("Original", identity_filter, baseline_result)
else:
    print(f"  reusing the saved result ({os.path.basename(RESULT_CACHE_DIR)}/, REUSE_RESULTS = True)")
print("  -> " + _result_line(baseline_result) + f" | OK recall {baseline_result['ok_recall']:.2f}% | F1 {baseline_result['f1']:.2f}%")


## 15. Round 1 - run (10 single filters)


In [ ]:
print("=" * 70)
print(f"Round 1 - {len(FILTERS_ROUND1)} single filters x {len(eval_items)} images  (FILTER_MODE = {FILTER_MODE})")
print("=" * 70)
results_round1 = run_round("Round 1", FILTERS_ROUND1)

## 16. Round 1 - graphs (10) and table


In [ ]:
round1_fig_paths = make_all_metric_plots(results_round1, "round1", baseline_result)
print(f"\nRound 1: {len(round1_fig_paths)} graphs saved.")
df_round1 = results_to_df(results_round1, baseline_result)
show_and_save_table(df_round1, "round1")

## 17. Round 2 - run (5 combined filters)


In [ ]:
print("=" * 70)
print(f"Round 2 - {len(FILTERS_ROUND2)} combined filters x {len(eval_items)} images  (FILTER_MODE = {FILTER_MODE})")
print("=" * 70)
results_round2 = run_round("Round 2", FILTERS_ROUND2)

## 18. Round 2 - graphs (10) and table


In [ ]:
round2_fig_paths = make_all_metric_plots(results_round2, "round2", baseline_result)
print(f"\nRound 2: {len(round2_fig_paths)} graphs saved.")
df_round2 = results_to_df(results_round2, baseline_result)
show_and_save_table(df_round2, "round2")

## 19. Combined filters vs their parts (table + graph)

For every Round 2 combination this puts the combination next to its two components (both parts are Round 1 filters) and answers: **is the combination better than the best of its parts?** `Gain vs Best Component` is in percentage points; `McNemar p` compares the combination with its best component on the same images. A combination whose component was skipped (`SKIP_FILTERS`) is left out.


In [ ]:
COMBO_PARTS = ROUND2_COMBOS   # combined filter -> (first stage, second stage), defined in Section 10


def components_vs_combination():
    lookup = {r["name"]: r for r in results_round1}
    rows = []
    for r in results_round2:
        if r["name"] not in COMBO_PARTS:
            continue
        pa, pb = COMBO_PARTS[r["name"]]
        if pa not in lookup or pb not in lookup:
            continue                                  # a component was skipped
        ra, rb = lookup[pa], lookup[pb]
        best = ra if ra["accuracy"] >= rb["accuracy"] else rb
        _, _, p = mcnemar_between(best["name"], r["name"])
        gain = r["accuracy"] - best["accuracy"]
        rows.append({
            "Combination": r["name"], "Accuracy (%)": r["accuracy"],
            "Component A": pa, "A Acc. (%)": ra["accuracy"],
            "Component B": pb, "B Acc. (%)": rb["accuracy"],
            "Best Component": best["name"], "Gain vs Best Component (pp)": gain,
            "Verdict": "Combination better" if gain > 0 else ("Component better" if gain < 0 else "Tie"),
            "McNemar p vs Best Component": p,
            "Significant (p<0.05)": "-" if p is None else ("Yes" if p < 0.05 else "No"),
            "Combination F1 (%)": r["f1"], "Best Component F1 (%)": best["f1"],
        })
    df = pd.DataFrame(rows)
    return df.round({c: (4 if c.startswith("McNemar") else 2) for c in df.columns if df[c].dtype.kind == "f"})


def plot_components_vs_combination(df, save_name="fig_combo_vs_components.png"):
    x = np.arange(len(df))
    w = 0.27
    fig, ax = plt.subplots(figsize=(15, 6.8))
    series = [("Combination", df["Accuracy (%)"].values, [""] * len(df), "#1a9850"),
              ("Component A", df["A Acc. (%)"].values, df["Component A"].tolist(), "#fdae61"),
              ("Component B", df["B Acc. (%)"].values, df["Component B"].tolist(), "#74add1")]
    handles = []
    for k, (lab, vals, names, col) in enumerate(series):
        bars = ax.bar(x + (k - 1) * w, vals, width=w, color=col, edgecolor="none", zorder=3)
        handles.append(bars[0])
        for b, v, nm in zip(bars, vals, names):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.6, f"{v:.1f}", ha="center", va="bottom",
                    fontsize=12, fontweight="bold", color="#1a1a1a")
            if nm:
                ax.text(b.get_x() + b.get_width() / 2, 2, nm, rotation=90, ha="center", va="bottom",
                        fontsize=11, fontweight="bold", color="#1a1a1a", zorder=4)
    base_acc = baseline_result["accuracy"]
    h = ax.axhline(base_acc, color="#888888", linestyle=":", linewidth=1.4, zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels(df["Combination"].tolist(), rotation=12)
    ax.set_xlabel("Combined Filter", fontsize=25, fontweight="bold")
    ax.set_ylabel("Overall Accuracy (%)", fontsize=25, fontweight="bold")
    ax.set_ylim(0, 115)
    ax.grid(axis="y", zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight("bold")
    for spine in ax.spines.values():
        spine.set_color("#222222")
        spine.set_linewidth(0.9)
    leg = ax.legend(handles + [h], ["Combination", "Component A", "Component B", f"Original {base_acc:.2f}%"],
                    loc="lower right", bbox_to_anchor=(1.0, 1.01), ncol=4, frameon=True, fontsize=17)
    for text in leg.get_texts():
        text.set_fontweight("bold")
    fig.tight_layout()
    out_path = os.path.join(FILTER_FIG_DIR, save_name)
    save_reference_figure(fig, out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {out_path}")
    return out_path


df_combo = components_vs_combination()
if len(df_combo):
    show_and_save_table(df_combo, "combo_vs_components")
    plot_components_vs_combination(df_combo)
else:
    print("No combination has both of its components scored (check SKIP_FILTERS).")


## 20. Overall summary and saving


In [ ]:
import platform, datetime
_ver = lambda mod: getattr(sys.modules.get(mod), "__version__", "n/a")
environment = dict(
    date=datetime.datetime.now().isoformat(timespec="seconds"), python=platform.python_version(),
    torch=_ver("torch"), kornia=_ver("kornia"), ultralytics=_ver("ultralytics"),
    opencv=cv2.__version__, numpy=np.__version__,
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    seed=SEED, conf=CONF_THRES, iou=IOU_THRES, imgsz=IMGSZ, weights=_weights_id(),
)
all_results = {"environment": environment, "filter_mode": FILTER_MODE, "run_preset": RUN_PRESET, "eval_splits": EVAL_SPLITS,
               "cv": dict(use=USE_CV, folds=CV_FOLDS, run_folds=_run_folds, val_frac=CV_VAL_FRAC, train_frac=STRESS_TRAIN_FRAC,
                          imgsz=CV_IMGSZ, noise_std=STRESS_NOISE_STD, filter_epochs=FILTER_EPOCHS),
               "n_images": len(eval_items), "baseline": baseline_result,
               "round1": results_round1, "round2": results_round2}
json_path = os.path.join(FILTER_FIG_DIR, "results_summary.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

# combined table: Original + Round 1 + Round 2
df_all = pd.concat([
    df_round1.iloc[:1].assign(Round="Baseline"),
    df_round1.iloc[1:].assign(Round="Round 1"),
    df_round2.iloc[1:].assign(Round="Round 2"),
], ignore_index=True)
df_all.insert(0, "Round", df_all.pop("Round"))
csv_all = os.path.join(FILTER_FIG_DIR, "table_all.csv")
df_all.to_csv(csv_all, index=False, encoding="utf-8-sig")

# per-image predictions of every method (for error analysis and paired tests)
pi_path = os.path.join(FILTER_FIG_DIR, "per_image_predictions.csv")
pd.DataFrame([r for rows in PER_IMAGE.values() for r in rows]).to_csv(pi_path, index=False, encoding="utf-8-sig")
print(f"Saved: {json_path}\nSaved: {csv_all}\nSaved: {pi_path}")

# overview graph: overall accuracy of every method
ov = [baseline_result] + list(results_round1) + list(results_round2)
plot_clean_bar(
    labels=[r["name"] for r in ov], values=[r["accuracy"] for r in ov],
    xlabel="Filter", ylabel="Overall Accuracy (%)", save_name="fig_all_overview_overall_accuracy.png",
    value_fmt="{:.2f}%", ymax=110, rotation=35, figsize=(21, 7.5), value_fontsize=13,
    threshold=baseline_result["accuracy"], threshold_label=f"Original {baseline_result['accuracy']:.2f}%",
    out_dir=FILTER_FIG_DIR, legend_above=True,
)

print(f"\nRUN_PRESET = {RUN_PRESET}, {len(eval_items)} images per filter, splits = {EVAL_SPLITS}")
print(f"Original (no filter): accuracy {baseline_result['accuracy']:.2f}%  NG recall {baseline_result['ng_recall']:.2f}%")
for tag, rs in (("Round 1 - single filters", results_round1), ("Round 2 - combined filters", results_round2)):
    print(f"\n[{tag}] ranked by overall accuracy")
    _nz = lambda v: 0.0 if v is None or (isinstance(v, float) and math.isnan(v)) else v
    for r in sorted(rs, key=lambda x: (-x["accuracy"], -_nz(x.get("map50_95")))):     # accuracy ties -> mAP50-95 decides
        d = r["accuracy"] - baseline_result["accuracy"]
        extra = (f"  mAP50-95 {r['map50_95']:5.1f}%  ball recall {r['ball_recall']:6.2f}%  center err {r['loc_err_px']:5.2f}px"
                 if _nz(r.get("map50_95")) else "")
        print(f"  {r['name']:<24s} accuracy {r['accuracy']:6.2f}% ({d:+.2f} pp)  NG recall {r['ng_recall']:6.2f}%{extra}  "
              f"filter {r['avg_time_ms']:9.2f} ms/img")

# significance against Original (exact McNemar on the paired per-image judgements)
sig = [(r["name"], r["accuracy"] - baseline_result["accuracy"], r["newly_wrong"], r["newly_correct"], r["mcnemar_p"])
       for r in list(results_round1) + list(results_round2) if r.get("mcnemar_p") is not None]
if sig:
    bonf = 0.05 / len(sig)
    print(f"\n[Significance vs Original - exact McNemar test]   * p < 0.05   ** p < {bonf:.4f} (Bonferroni, {len(sig)} comparisons)")
    for nm, d, nw, nc, p in sorted(sig, key=lambda x: x[4]):
        mark = "**" if p < bonf else ("*" if p < 0.05 else "")
        print(f"  {nm:<24s} {d:+7.2f} pp   newly wrong {nw:4d} / newly correct {nc:4d}   p = {p:.4g} {mark}")

n_fig = len(round1_fig_paths) + len(round2_fig_paths)
n_samples = len(glob.glob(os.path.join(FILTER_SAMPLE_DIR, "*_example.png")))
print(f"\nGraphs: {n_fig} (+1 overview, +1 combination-vs-components) -> {os.path.abspath(FILTER_FIG_DIR)}")
n_grids = len(glob.glob(os.path.join(FILTER_SAMPLE_DIR, "grid_*.png")))
print(f"Original-vs-filtered example images: {n_samples} (+{n_grids} grid figures) -> {os.path.abspath(FILTER_SAMPLE_DIR)}")


## 21. Notes for the next step

- In (2) of Section 9, if YOLO really missed balls (fewer green boxes than the labels), lower `CONF_THRES` or raise `EPOCHS` / `IMGSZ` and retrain. If it found balls where there are none (green / red boxes on places without a ball), raise `CONF_THRES` or add hard negatives to the training data.
- If (3) remains, add more values to `open_ksize_grid` / `std_ksize_grid` in `LOC_PARAMS` to widen the candidates.
- If you have a separate ground-truth OK/NG CSV, fill in `OKNG_CSV` to check the accuracy against the real ground truth instead of the label-based one.
- Extreme filters: with `FILTER_MODE = "inference_only"` the YOLO of Section 5 has only seen normal images, so filters like Invert, Otsu Threshold or Gaussian Blur (max) will probably break the detection - the accuracy then drops towards "everything is NG". That measures how much the detector depends on the normal look of the images, not whether the information is gone. `FILTER_MODE = "train_per_filter"` trains a YOLO per filter and shows whether the information survives (a plain inversion should be learnable again; a blur that really removed the balls would not be - sigma = 20 px does not).
- A run on all images takes a while: run once with `RUN_PRESET = "quick"` (default; a smoke test on 30 images), read the runtime estimate printed in Section 13, then switch to `RUN_PRESET = "final"`.
- For a paper-grade comparison use `RUN_PRESET = "final"` with `USE_CV = True`: all 500 images are scored by detectors that never trained on them. Scoring the train / val splits with a model that was trained on them inflates every row, including Original.
- If every method still ties at 100% accuracy, read the continuous metrics (ball recall, mAP50-95, min. confidence, centre error) or switch on one `STRESS_*` option (e.g. `STRESS_TRAIN_FRAC = 0.25`).
- The McNemar p-values compare each method with Original on the same images (one comparison at a time). With about 15 comparisons some can fall below 0.05 by chance, so the summary also prints a Bonferroni threshold. `per_image_predictions.csv` holds the judgement of every method for every image if you want to look at individual cases.
- `results_summary.json` also stores the software versions, GPU, seed, thresholds and detector weights of the run (`environment`), which a paper's reproducibility section needs.
- If the run is interrupted, just run the notebook again from the top: finished methods are read from `result_cache/` (Section 11).


## 22. Difficulty sweeps (graded experiments)

One fixed difficulty ends at 100% for almost every method. A **sweep** changes one thing step by step - like the blur-sigma list of the ResNet50 notebook - so the accuracy walks from ~100% down to the point where the detector breaks, and the numbers differ from level to level. Each level is a full **5-fold CV** run of the *Original* (unfiltered) detector under that condition:

| Sweep | What changes | Default levels |
|---|---|---|
| `blur` | Gaussian blur sigma applied to every image (px at full resolution) | 0, 8, 16, 32, 64, 128, 256 |
| `noise` | Gaussian noise (gray-level std) added to every image | 0, 10, 20, 40, 80, 120 |
| `resolution` | detector input size (`imgsz`) for training and inference | 1280, 960, 640, 480, 320, 224 |
| `data` | share of the training images that is used | 100%, 50%, 25%, 10%, 5% |

The first level of every sweep is the normal baseline (reused, not trained again). `RUN_PRESET = "quick"` keeps 3 levels per sweep and runs fold 0 only. Finished levels are cached (`result_cache/`, weights), so an interrupted run continues where it stopped. `SWEEP_CLEAN_FILTERED = True` deletes the filtered image copies of a level after it is scored (each copy is several GB); set it to `False` if disk space is no problem.

In [ ]:
RUN_SWEEPS = True
SWEEP_LEVELS = dict(
    blur=[0, 8, 16, 32, 64, 128, 256],
    noise=[0, 10, 20, 40, 80, 120],
    resolution=[IMGSZ, 960, 640, 480, 320, 224],      # multiples of 32
    data=[1.0, 0.5, 0.25, 0.1, 0.05],
)
SWEEP_ORDER = ["blur", "noise", "resolution", "data"]
SWEEP_CLEAN_FILTERED = True


class _OriginalRef:
    """Identity filter that shares the 'original' image folder with the baseline (no extra copy on disk)."""
    root_key = "original"

    def __call__(self, t):
        return t

    def describe(self):
        return "-"

    def signature(self):
        return ""


SAME = _OriginalRef()


def set_condition(noise=0.0, train_frac=1.0, imgsz=None):
    """Switches the global stress settings that the CV functions read (noise, training share, detector size)."""
    global STRESS_NOISE_STD, STRESS_TRAIN_FRAC, STRESS_IMGSZ, CV_IMGSZ, _STRESS_TAG
    STRESS_NOISE_STD, STRESS_TRAIN_FRAC, STRESS_IMGSZ = noise, train_frac, imgsz
    CV_IMGSZ = imgsz or IMGSZ
    _STRESS_TAG = f"_n{noise:g}" if noise else ""


def cleanup_filtered(name, fn):
    root = filtered_root_for(name, fn)
    shutil.rmtree(root, ignore_errors=True)
    for d in glob.glob(os.path.abspath(root) + f"_cv{CV_FOLDS}f*") + glob.glob(os.path.abspath(root) + "_os"):
        shutil.rmtree(d, ignore_errors=True)


_BASE_COND = dict(noise=STRESS_NOISE_STD, frac=STRESS_TRAIN_FRAC, imgsz=STRESS_IMGSZ)
_BASELINE_VALUE = dict(blur=0, noise=_BASE_COND["noise"], resolution=_BASE_COND["imgsz"] or IMGSZ, data=_BASE_COND["frac"])
_LABEL = dict(blur=lambda v: "Original" if v == 0 else f"σ{v:g}", noise=lambda v: "Original" if v == 0 else f"noise {v:g}",
              resolution=lambda v: f"{v}px", data=lambda v: f"{int(round(100 * v))}%")
SWEEP_TITLE = dict(blur=("blur strength", "Gaussian blur σ (px)"), noise=("noise level", "Gaussian noise std (gray levels)"),
                   resolution=("input resolution", "Detector input size (px)"), data=("training data amount", "Share of training images used"))


def _sweep_levels(axis):
    lv = list(SWEEP_LEVELS[axis])
    base = _BASELINE_VALUE[axis]
    lv = [base] + [v for v in lv if v != base]
    if RUN_PRESET == "quick" and len(lv) > 3:
        lv = [lv[0], lv[len(lv) // 2], lv[-1]]
    return lv


def _level_name_and_cond(axis, v):
    cond = dict(_BASE_COND)
    if axis == "blur":
        return f"Blur sigma={v:g}", ParamFilter(f"Blur sigma={v:g}", gaussian_blur_gpu, dict(sigma=float(v))), cond
    if axis == "noise":
        cond["noise"] = v
        return f"Noise std={v:g}", SAME, cond
    if axis == "resolution":
        cond["imgsz"] = v
        return f"Res {v}", SAME, cond
    cond["frac"] = v
    return f"Data {int(round(100 * v))}pct", SAME, cond


def run_condition(name, fn, noise, frac, imgsz):
    """One 5-fold CV run of `fn` under a stress condition (cached; the settings are restored afterwards)."""
    prev = (STRESS_NOISE_STD, STRESS_TRAIN_FRAC, STRESS_IMGSZ)
    set_condition(noise, frac, imgsz)
    try:
        r = cached_result(name, fn)
        if r is not None:
            print(f"  [{name}] reusing the saved result")
            return r
        t0 = time.time()
        r = evaluate_cv(name, fn, eval_items)
        r["params"] = f"{filter_params_str(fn)} | noise {noise:g}, train share {frac:g}, imgsz {CV_IMGSZ}"
        save_result(name, fn, r)
        if SWEEP_CLEAN_FILTERED and not (fn is SAME and not noise):
            cleanup_filtered(name, fn)
        r["minutes"] = (time.time() - t0) / 60
        return r
    finally:
        set_condition(*prev)


SWEEP_RESULTS, NOISE_BY_LEVEL = {}, {}
if RUN_SWEEPS and not USE_CV:
    print("Sweeps need USE_CV = True (FILTER_MODE = 'train_per_filter') -> skipped.")
elif RUN_SWEEPS:
    _n_new = sum(len(_sweep_levels(a)) - 1 for a in SWEEP_ORDER) * len(_run_folds)
    print(f"Sweeps: {_n_new} new trainings ({FILTER_EPOCHS} epochs each, roughly {_n_new * 2 / 60:.1f} h on a fast GPU; "
          "low-resolution and low-data levels are faster).")
    for axis in SWEEP_ORDER:
        res = []
        for v in _sweep_levels(axis):
            lbl = _LABEL[axis](v)
            if v == _BASELINE_VALUE[axis]:
                r = dict(baseline_result, label=lbl)
            else:
                nm, fn, cond = _level_name_and_cond(axis, v)
                print(f"\n[Sweep {axis}] {lbl}  ({nm}) ...")
                r = dict(run_condition(nm, fn, cond["noise"], cond["frac"], cond["imgsz"]), label=lbl)
                print("  -> " + _result_line(r))
            res.append(r)
            if axis == "noise":
                NOISE_BY_LEVEL[v] = r
        SWEEP_RESULTS[axis] = res
        print(f"\n[Sweep {axis}] summary")
        print(f"  {'level':>8s} | {'accuracy':>8s} | {'bal.acc':>7s} | {'ball rec.':>9s} | {'mAP50-95':>8s} | {'fold std':>8s}")
        for r in res:
            print(f"  {r['label']:>8s} | {r['accuracy']:7.2f}% | {r['bal_acc']:6.2f}% | {r['ball_recall']:8.2f}% | "
                  f"{r['map50_95']:7.1f}% | {r['fold_std']:7.2f}")

## 23. Filter x noise grid (does a filter help when the image is degraded?)

On clean images most filters tie. Here every Round-1 filter is applied to **noisy** images (noise first, then the filter - a filter acts as a restoration step) and compared with the *unfiltered noisy* images of the same noise level (McNemar p-values in the tables refer to that noisy baseline). `GRID_NOISE_LEVELS` picks the noise levels, `GRID_FILTER_NAMES` the filters (`None` = all 10 single filters).

In [ ]:
RUN_FILTER_GRID = True
GRID_NOISE_LEVELS = [40, 80]
GRID_FILTER_NAMES = None          # e.g. ["Gaussian Blur (max)", "CLAHE (max)"]; None = all Round-1 filters

GRID_RESULTS = {}
if RUN_FILTER_GRID and not USE_CV:
    print("The grid needs USE_CV = True -> skipped.")
elif RUN_FILTER_GRID:
    _grid_filters = [(n, f) for n, f in FILTERS_ROUND1 if (GRID_FILTER_NAMES is None or n in GRID_FILTER_NAMES) and n not in SKIP_FILTERS]
    _n_new = len(GRID_NOISE_LEVELS) * len(_grid_filters) * len(_run_folds)
    print(f"Grid: {len(GRID_NOISE_LEVELS)} noise levels x {len(_grid_filters)} filters, {_n_new} trainings (roughly {_n_new * 2 / 60:.1f} h).")
    for nz in GRID_NOISE_LEVELS:
        ref = NOISE_BY_LEVEL.get(nz)
        if ref is None:
            ref = dict(run_condition(f"Noise std={nz:g}", SAME, nz, _BASE_COND["frac"], _BASE_COND["imgsz"]), label=f"noise {nz:g}")
        rs = []
        for fname, fn in _grid_filters:
            nm = f"{fname} | noise {nz:g}"
            print(f"\n[Grid noise {nz:g}] {fname} ...")
            r = run_condition(nm, fn, nz, _BASE_COND["frac"], _BASE_COND["imgsz"])
            r["newly_wrong"], r["newly_correct"], r["mcnemar_p"] = mcnemar_between(ref["name"], nm)
            print("  -> " + _result_line(r))
            rs.append(dict(r, label=fname))
        GRID_RESULTS[nz] = [dict(ref, label=f"Original (noise {nz:g})")] + rs
        print(f"\n[Grid noise {nz:g}] ranked by accuracy (baseline = unfiltered noisy images: {ref['accuracy']:.2f}%)")
        for r in sorted(rs, key=lambda x: -x["accuracy"]):
            print(f"  {r['label']:<24s} {r['accuracy']:6.2f}% ({r['accuracy'] - ref['accuracy']:+6.2f} pp)  mAP50-95 {r['map50_95']:5.1f}%")

## 24. Final result graphs - the 10-graph set (reference style) for every experiment group

For every group (the 16-method **filter comparison**, the four **sweeps**, the **grid** per noise level) the same 10 graph types of the ResNet50 reference notebook are drawn (in this notebook\'s own light style; `GRAPH_THEME = "dark"` switches to the dark style of the reference), 10 PNG files per group in `yolo_filter_figures/ref_graphs_<group>/`:

1. overall accuracy (bars + dashed line) - 2. accuracy per class (OK / NG / ball detection) - 3. dual axis accuracy vs mAP50-95 (replaces ROC-AUC: the OK/NG rule has no score) - 4. confusion matrices (baseline vs the hardest level) - 5. learning curves of the baseline (mean +- std over folds, from YOLO `results.csv`) - 6. heat map level x fold (held-out accuracy) - 7. over/under-fit verdicts per level (from the loss curves) - 8. radar chart - 9. GPU VRAM / utilisation / temperature timeline - 10. dashboard with a ranking table.

`SHOW_INLINE = False` saves the files without showing them in the notebook. Tables (CSV) are saved per group as well.

In [ ]:
from matplotlib.gridspec import GridSpec

SHOW_INLINE = True
GRAPH_THEME = "light"     # "light" = the style of the rest of this notebook; "dark" = the dark style of the ResNet50 reference notebook
DARK_RC = {
    "figure.facecolor": "#0d1117", "axes.facecolor": "#161b22", "axes.edgecolor": "#444", "axes.labelcolor": "#aaa",
    "xtick.color": "#aaa", "ytick.color": "#aaa", "grid.color": "#333", "grid.linestyle": "--", "grid.alpha": 0.5,
    "legend.facecolor": "#222", "legend.labelcolor": "white", "text.color": "white",
}


def diagnose_fit_yolo(h):
    """'good' | 'overfit' | 'underfit' | 'unknown' from one fold's YOLO curves (box loss + mAP50-95 on 0-1)."""
    tl, vl, mp = h.get("loss") or [], h.get("val_loss") or [], h.get("map50_95") or []
    if not tl or not vl:
        return "unknown"
    if (mp[-1] if mp else 0.0) < 0.30:
        return "underfit"
    if vl[-1] > 1.25 * min(vl) and (vl[-1] - tl[-1]) > 0.25 * tl[-1]:
        return "overfit"
    return "good"


def _pad_stack(hists, key):
    series = [list(h[key]) for h in hists if h.get(key)]
    if not series:
        return None
    L = max(len(s) for s in series)
    return np.array([s + [s[-1]] * (L - len(s)) for s in series], dtype=float)


def make_reference_graphs(group_key, results, title, xlabel, show_gpu=True):
    """results[0] = baseline of the group. Returns the list of saved PNG paths."""
    out_dir = os.path.join(FILTER_FIG_DIR, f"ref_graphs_{group_key}")
    os.makedirs(out_dir, exist_ok=True)
    n = len(results)
    labels = [r.get("label", r["name"]) for r in results]
    col = lambda k: np.array([np.nan if r.get(k) is None else r[k] for r in results], dtype=float)
    accs, ok_rec, ng_rec, ball, mapv, minc = (col(k) for k in ("accuracy", "ok_recall", "ng_recall", "ball_recall", "map50_95", "min_conf"))
    COLORS = [plt.cm.plasma(i / max(n - 1, 1)) for i in range(n)]
    C3 = ["#4CAF50", "#F44336", "#2196F3"]
    CLS = ["OK", "NG", "Ball detection"]
    fold_n = max(len(r.get("fold_acc") or []) for r in results)
    tag, saved = f"[{group_key}]", []
    wfig = max(13, 0.8 * n + 7)

    def savefig(name, show=True):
        path = os.path.join(out_dir, name)
        save_reference_figure(plt.gcf(), path, dpi=150, bbox_inches="tight", facecolor=plt.rcParams["figure.facecolor"])
        plt.show() if (SHOW_INLINE and show) else plt.close()
        print(f"  saved: {path}")
        saved.append(path)

    def xt(ax, rot=25, fs=9):
        ax.set_xticks(range(n)); ax.set_xticklabels(labels, fontsize=fs, rotation=rot, ha="right")

    dark = GRAPH_THEME == "dark"
    TXT, GOLD = ("white", "#FFD700") if dark else ("#222222", "#c99700")
    with plt.rc_context(DARK_RC if dark else {}):
        # 1 ─ overall accuracy
        print(f"\n{tag} [graph 1/10] overall accuracy by {title}")
        fig, ax = plt.subplots(figsize=(wfig, 5))
        bars = ax.bar(range(n), accs, color=COLORS, edgecolor="#555", linewidth=0.8, zorder=3)
        ax.plot(range(n), accs, "o--", color=GOLD, lw=1.8, ms=7, zorder=4)
        for b, a in zip(bars, accs):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5, f"{a:.1f}%", ha="center", va="bottom", fontsize=9.5, color=TXT, fontweight="bold")
        ax.axhline(np.nanmean(accs), color="#FF6B6B", ls="--", lw=1.5, label=f"mean {np.nanmean(accs):.1f}%")
        ax.set_xlabel(xlabel, fontsize=12); ax.set_ylabel("Held-out accuracy (%)", fontsize=12)
        ax.set_title(f"OK/NG judgement accuracy by {title}\n(YOLOv8n | {CV_FOLDS}-fold CV | {results[0]['n_images']} images)", fontsize=14, fontweight="bold")
        ax.set_ylim(max(0, np.nanmin(accs) - 10), min(105, np.nanmax(accs) + 8)); ax.legend(fontsize=10); ax.grid(axis="y", zorder=0)
        xt(ax); plt.tight_layout(); savefig("01_overall_accuracy.png")

        # 2 ─ per-class accuracy
        print(f"\n{tag} [graph 2/10] accuracy per class")
        fig, ax = plt.subplots(figsize=(wfig + 1, 5.5)); width = 0.27; x = np.arange(n)
        for ci, (vals, c) in enumerate(zip((ok_rec, ng_rec, ball), C3)):
            rects = ax.bar(x + (ci - 1) * width, vals, width, label=CLS[ci], color=c, alpha=0.85, edgecolor="#555", linewidth=0.6)
            for rc, v in zip(rects, vals):
                ax.text(rc.get_x() + rc.get_width() / 2, rc.get_height() + 0.3, f"{v:.0f}", ha="center", va="bottom", fontsize=7.5, color=TXT)
        xt(ax, fs=9); ax.set_xlabel(xlabel, fontsize=12); ax.set_ylabel("Accuracy (%)", fontsize=12)
        ax.set_title("Accuracy per class (OK / NG / ball detection)", fontsize=14, fontweight="bold")
        ax.legend(fontsize=10, ncol=3, loc="lower right", bbox_to_anchor=(1.0, 1.01)); ax.grid(axis="y"); ax.set_ylim(0, 115); plt.tight_layout(); savefig("02_per_class_accuracy.png")

        # 3 ─ accuracy vs mAP50-95 (dual axis)
        print(f"\n{tag} [graph 3/10] mAP50-95 vs accuracy (dual axis)")
        fig, ax1 = plt.subplots(figsize=(wfig, 5)); ax2 = ax1.twinx()
        ax1.plot(range(n), accs, "o-", color="#4FC3F7", lw=2.2, ms=8, label="Accuracy (%)", zorder=4)
        ax2.plot(range(n), mapv, "s--", color="#FFB74D", lw=2.0, ms=7, label="mAP50-95 (%)", zorder=4)
        ax1.fill_between(range(n), accs, alpha=0.15, color="#4FC3F7"); ax2.fill_between(range(n), mapv, alpha=0.12, color="#FFB74D")
        ax1.set_xlabel(xlabel, fontsize=12); ax1.set_ylabel("Accuracy (%)", fontsize=12, color="#4FC3F7"); ax2.set_ylabel("mAP50-95 (%)", fontsize=12, color="#FFB74D")
        xt(ax1); ax1.set_title(f"mAP50-95 vs accuracy by {title}", fontsize=14, fontweight="bold")
        l1, b1 = ax1.get_legend_handles_labels(); l2, b2 = ax2.get_legend_handles_labels()
        ax1.legend(l1 + l2, b1 + b2, fontsize=10, loc="lower left"); ax1.grid(axis="y"); plt.tight_layout(); savefig("03_map_vs_accuracy.png")

        # 4 ─ confusion matrices (baseline vs hardest)
        print(f"\n{tag} [graph 4/10] confusion matrices (baseline vs hardest condition)")
        hard = int(np.nanargmin(accs[1:])) + 1 if n > 1 else 0
        pair = [0] if hard == 0 else [0, hard]
        fig, axes = plt.subplots(1, len(pair), figsize=(6 * len(pair), 5.5)); axes = np.atleast_1d(axes)
        for ax, idx in zip(axes, pair):
            r = results[idx]; cm = np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]]); cmn = cm / (cm.sum(axis=1, keepdims=True) + 1e-9)
            im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, f"{cm[i, j]}\n({cmn[i, j] * 100:.1f}%)", ha="center", va="center", fontsize=10, color="white" if cmn[i, j] > 0.55 else ("#ccc" if dark else "#222222"))
            ax.set_xticks([0, 1]); ax.set_xticklabels(["OK", "NG"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["OK", "NG"])
            ax.set_xlabel("Predicted class", fontsize=11); ax.set_ylabel("True class", fontsize=11)
            ax.set_title(f"Confusion matrix - {labels[idx]}\nAccuracy: {r['accuracy']:.1f}%", fontsize=12, fontweight="bold")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.suptitle("Confusion matrices: baseline vs hardest condition", fontsize=14, fontweight="bold", y=1.01); plt.tight_layout(); savefig("04_confusion_matrices.png")

        # 5 ─ learning curves of the baseline
        print(f"\n{tag} [graph 5/10] learning curves of the baseline (mean over folds)")
        hs = list((results[0].get("histories") or {}).values())
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        if hs:
            for key, color, lab in (("map50", "#4FC3F7", "mAP50"), ("map50_95", "#FFB74D", "mAP50-95")):
                m = _pad_stack(hs, key)
                if m is not None:
                    mean, std, ep = m.mean(0) * 100, m.std(0) * 100, range(1, m.shape[1] + 1)
                    ax1.plot(ep, mean, color=color, lw=2, label=f"val {lab}"); ax1.fill_between(ep, mean - std, mean + std, alpha=0.2, color=color)
            for key, color, lab in (("loss", "#4FC3F7", "train"), ("val_loss", "#FFB74D", "val")):
                m = _pad_stack(hs, key)
                if m is not None:
                    mean, std, ep = m.mean(0), m.std(0), range(1, m.shape[1] + 1)
                    ax2.plot(ep, mean, color=color, lw=2, label=f"{lab} box loss"); ax2.fill_between(ep, mean - std, mean + std, alpha=0.2, color=color)
        else:
            for ax in (ax1, ax2):
                ax.text(0.5, 0.5, "results.csv not found", ha="center", va="center", transform=ax.transAxes)
        ax1.set_title(f"Validation mAP - {labels[0]}\n(mean ± std over {len(hs)} folds)", fontsize=12, fontweight="bold")
        ax1.set_xlabel("Epoch"); ax1.set_ylabel("mAP (%)"); ax2.set_title(f"Box loss - {labels[0]}\n(mean ± std over {len(hs)} folds)", fontsize=12, fontweight="bold")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("box loss")
        for ax in (ax1, ax2):
            ax.grid()
            if hs: ax.legend()
        plt.tight_layout(); savefig("05_learning_curves_baseline.png")

        # 6 ─ heat map level x fold
        print(f"\n{tag} [graph 6/10] heat map condition x fold")
        hm = np.full((n, max(fold_n, 1)), np.nan)
        for i, r in enumerate(results):
            fa = r.get("fold_acc") or []
            hm[i, :len(fa)] = fa
        fig, ax = plt.subplots(figsize=(max(8, 1.6 * hm.shape[1] + 4), max(4, 0.55 * n + 2)))
        vmin = min(50, np.floor(np.nanmin(hm) - 5)) if np.isfinite(hm).any() else 50
        im = ax.imshow(hm, cmap="RdYlGn", aspect="auto", vmin=vmin, vmax=100)
        for i in range(hm.shape[0]):
            for j in range(hm.shape[1]):
                if np.isfinite(hm[i, j]):
                    ax.text(j, i, f"{hm[i, j]:.1f}", ha="center", va="center", fontsize=9, color="black" if 60 < hm[i, j] < 95 else "white")
        ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels([f"Fold {k + 1}" for k in range(hm.shape[1])], fontsize=10)
        ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=9); ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel(xlabel, fontsize=12)
        ax.set_title(f"Held-out accuracy (%) by {title} and fold", fontsize=14, fontweight="bold"); plt.colorbar(im, ax=ax, label="Accuracy (%)")
        plt.tight_layout(); savefig("06_heatmap_accuracy_by_fold.png")

        # 7 ─ over / under-fit verdicts
        print(f"\n{tag} [graph 7/10] over/under-fit verdicts")
        good, over, under = [], [], []
        for r in results:
            st = [diagnose_fit_yolo(h) for h in (r.get("histories") or {}).values()]
            good.append(st.count("good")); over.append(st.count("overfit")); under.append(st.count("underfit"))
        fig, ax = plt.subplots(figsize=(wfig, 5)); x = np.arange(n)
        p1 = ax.bar(x, good, color="#4CAF50", label="good"); ax.bar(x, over, bottom=good, color="#F44336", label="overfit")
        ax.bar(x, under, bottom=[g + o for g, o in zip(good, over)], color="#FFC107", label="underfit")
        for b, v in zip(p1, good):
            if v > 0: ax.text(b.get_x() + b.get_width() / 2, v / 2, str(v), ha="center", va="center", fontsize=10, color="white", fontweight="bold")
        xt(ax); ax.set_xlabel(xlabel, fontsize=12); ax.set_ylabel(f"Folds (of {fold_n})", fontsize=12)
        ax.set_title(f"Over/under-fit verdict by {title}\n({fold_n} folds, judged from the loss and mAP curves)", fontsize=14, fontweight="bold")
        ax.legend(fontsize=10); ax.grid(axis="y"); ax.set_ylim(0, fold_n + 2); plt.tight_layout(); savefig("07_fit_diagnosis.png")

        # 8 ─ radar
        print(f"\n{tag} [graph 8/10] radar chart")
        rad = list(dict.fromkeys([0, n // 2, hard])) if n > 2 else list(range(n))
        names = ["Accuracy", "OK accuracy", "NG accuracy", "Ball detection", "mAP50-95", "Min. confidence"]
        ang = [k / float(len(names)) * 2 * np.pi for k in range(len(names))]; ang += ang[:1]
        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
        for idx, color in zip(rad, ["#4FC3F7", "#FFB74D", "#F06292"]):
            r = results[idx]; v = [r.get(k) if r.get(k) is not None else 0.0 for k in ("accuracy", "ok_recall", "ng_recall", "ball_recall", "map50_95", "min_conf")]
            v = [0.0 if (isinstance(a, float) and np.isnan(a)) else a for a in v]; v += v[:1]
            ax.plot(ang, v, "o-", lw=2, color=color, label=labels[idx]); ax.fill(ang, v, alpha=0.1, color=color)
        ax.set_xticks(ang[:-1]); ax.set_xticklabels(names, fontsize=10); ax.set_ylim(0, 110); ax.set_yticks([20, 40, 60, 80, 100])
        ax.set_yticklabels(["20", "40", "60", "80", "100"], fontsize=8, color="#aaa")
        ax.set_title(f"Radar chart - {title}", fontsize=14, fontweight="bold", pad=20)
        ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=10); plt.tight_layout(); savefig("08_radar_chart.png")

        # 9 ─ GPU timeline
        print(f"\n{tag} [graph 9/10] GPU timeline")
        if GPU_LOG:
            t = [(g["time"] - GPU_LOG[0]["time"]) / 60 for g in GPU_LOG]
            fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
            a1.plot(t, [g["used_mib"] for g in GPU_LOG], color="#4FC3F7", lw=1.8); a1.fill_between(t, [g["used_mib"] for g in GPU_LOG], alpha=0.2, color="#4FC3F7")
            a1.axhline(GPU_LOG[0]["total_mib"] * 0.90, color="#F44336", ls="--", lw=1.2, label="warning line (90%)")
            a1.set_ylabel("VRAM used (MiB)", fontsize=11); a1.legend(fontsize=9); a1.grid(); a1.set_title("GPU VRAM usage timeline (whole notebook run)", fontsize=13, fontweight="bold")
            a2.plot(t, [g["util_pct"] for g in GPU_LOG], color="#FFB74D", lw=1.8); a2.fill_between(t, [g["util_pct"] for g in GPU_LOG], alpha=0.2, color="#FFB74D")
            a2.set_ylabel("GPU utilisation (%)", fontsize=11); a2.grid(); a2.set_ylim(0, 110)
            a3.plot(t, [g["temp_c"] for g in GPU_LOG], color="#F06292", lw=1.8); a3.fill_between(t, [g["temp_c"] for g in GPU_LOG], alpha=0.2, color="#F06292")
            a3.axhline(85, color="#F44336", ls="--", lw=1.2, label="temperature warning (85°C)"); a3.set_ylabel("GPU temperature (°C)", fontsize=11)
            a3.set_xlabel("Elapsed time (min)", fontsize=12); a3.legend(fontsize=9); a3.grid(); plt.tight_layout(); savefig("09_gpu_timeline.png", show=show_gpu)
        else:
            print("  (GPU_LOG is empty: the nvidia-smi sampler did not run -> graph 9 skipped; no simulated data is drawn)")

        # 10: six readable plots; no dense ranking table.
        fig = draw_reference_dashboard(results, title, diagnose_fit_yolo)
        savefig("10_dashboard.png")
    return saved


# ── draw the 10 graphs for every experiment group that has results ──
FINAL_GROUPS = []
if "results_round1" in globals() and "results_round2" in globals():
    _filt = [dict(baseline_result, label="Original")] + [dict(r, label=r["name"]) for r in list(results_round1) + list(results_round2)]
    FINAL_GROUPS.append(("filters", _filt, "preprocessing filter", "Preprocessing filter"))
for _ax in SWEEP_ORDER:
    if _ax in SWEEP_RESULTS:
        FINAL_GROUPS.append((f"sweep_{_ax}", SWEEP_RESULTS[_ax], SWEEP_TITLE[_ax][0], SWEEP_TITLE[_ax][1]))
for _nz, _rs in GRID_RESULTS.items():
    FINAL_GROUPS.append((f"grid_noise{_nz:g}", _rs, f"filter at noise σ={_nz:g}", f"Preprocessing filter (noise σ={_nz:g})"))

if not USE_CV:
    print("The reference graph set needs the CV results (USE_CV = True) -> skipped.")
else:
    ALL_GRAPH_PATHS = {}
    for _i, (_key, _res, _title, _xl) in enumerate(FINAL_GROUPS):
        if len(_res) < 1 or any(r.get("map50_95") is None for r in _res):
            print(f"[{_key}] skipped: results without CV metrics (run the CV cells first)")
            continue
        print("=" * 70 + f"\n{_key}: {len(_res)} conditions\n" + "=" * 70)
        ALL_GRAPH_PATHS[_key] = make_reference_graphs(_key, _res, _title, _xl, show_gpu=(_i == 0))
        _df = results_to_df(_res[1:], _res[0])
        _df.insert(0, "Level", [r.get("label", r["name"]) for r in _res[:1] + _res[1:]])
        _df.to_csv(os.path.join(FILTER_FIG_DIR, f"table_{_key}.csv"), index=False, encoding="utf-8-sig")
    print("\n" + "=" * 70)
    print(f"Done: {sum(len(v) for v in ALL_GRAPH_PATHS.values())} graphs in {len(ALL_GRAPH_PATHS)} groups -> {os.path.abspath(FILTER_FIG_DIR)}")
    for _k, _v in ALL_GRAPH_PATHS.items():
        print(f"  {_k:<22s} {len(_v)} graphs   (table_{_k}.csv)")
    with open(os.path.join(FILTER_FIG_DIR, "sweep_results.json"), "w", encoding="utf-8") as f:
        json.dump({k: v for k, v in dict(sweeps=SWEEP_RESULTS, grid={str(k): v for k, v in GRID_RESULTS.items()}).items()}, f, ensure_ascii=False, indent=1)

## 25. Verification: is the model *guessing*? (new in v6)

A 100% OK/NG accuracy has two possible explanations: the detector really finds the balls, or it learned something that correlates with the label (a fixed layout it can "remember", lighting, image-level cues). Five checks that need **no new training** (the fold detectors from Sections 14-17 are re-used from the weights cache, always on the images they never trained on):

| # | Check | If the model really sees the balls | If it is guessing |
|---|---|---|---|
| 1 | **Permutation test**: shuffle the ground truth across images, 10 000 times | shuffled accuracy ~50%, real accuracy far above it | real accuracy inside the shuffled cloud |
| 2 | **Erase all balls** (inpaint every labelled ball) and run the detector | ~0 detections | it still outputs boxes where balls "usually are" (position prior) |
| 3 | **Remove one ball** from an OK image | that ball is not detected, the other 12 are, verdict flips to NG with the right region count | verdict stays OK, or a ghost box appears at the erased spot |
| 4 | **Insert one ball** (copy a real ball of the same image to a free spot inside the rectangle) | the new ball is detected, verdict flips to NG | verdict stays OK (informational: pasted balls are synthetic) |
| 5 | **Global shortcut baseline**: logistic regression on a 32x32 thumbnail / global gray statistics (no localisation at all) | ~chance (cannot solve OK/NG without finding balls) | high accuracy = the data has a shortcut |

Checks 2-4 are *interventions*: a detector that only reads global cues cannot react to one erased ball, a detector that reads the balls must.


In [ ]:
# ── 25a. Helpers (verification + hard mode). Nothing here trains a detector: fold detectors come from the weights cache. ──
VERIFY_SEED = SEED + 7
_MODELS = {}


def cached_fold_model(name, fn, fold):
    """The fold detector Sections 14-17 trained for (name, fold), loaded from the weights cache; None if it is not there."""
    if not os.path.isfile(FILTER_WEIGHTS_JSON):
        return None
    with open(FILTER_WEIGHTS_JSON, encoding="utf-8") as f:
        cache = json.load(f)
    root = filtered_root_for(name, fn)
    key = (f"{os.path.basename(os.path.normpath(root))}_cv{CV_FOLDS}f{fold}{FOLD_TAG}_os{OVERSAMPLE_MAX_FACTOR if OVERSAMPLE else 0}"
           f"_e{FILTER_EPOCHS}_tf{STRESS_TRAIN_FRAC:g}_sz{CV_IMGSZ}_v{CV_VAL_FRAC:g}")
    w = cache.get(key)
    return YOLO(w) if (w and os.path.isfile(w)) else None


def get_model(name, fn, fold):
    if (name, fold) not in _MODELS:
        _MODELS[(name, fold)] = cached_fold_model(name, fn, fold)
    return _MODELS[(name, fold)]


def predict_gray(model, gray, imgsz=None):
    return yolo_predict_full(model, cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR), imgsz or CV_IMGSZ)


def erase_boxes(gray, boxes, pad=0.3):
    """Paint the given balls (cx, cy, w, h in px) over with the surrounding background (local cv2.inpaint, so it stays fast)."""
    out = gray.copy()
    H, W = gray.shape
    for cx, cy, bw, bh in boxes:
        r = int(max(bw, bh) / 2 * (1 + pad)) + 1
        x0, x1 = max(0, int(cx) - 3 * r), min(W, int(cx) + 3 * r + 1)
        y0, y1 = max(0, int(cy) - 3 * r), min(H, int(cy) + 3 * r + 1)
        crop = np.ascontiguousarray(out[y0:y1, x0:x1])
        m = np.zeros(crop.shape, np.uint8)
        cv2.circle(m, (int(cx) - x0, int(cy) - y0), r, 255, -1)
        out[y0:y1, x0:x1] = cv2.inpaint(crop, m, 5, cv2.INPAINT_TELEA)
    return out


def paste_ball(gray, box, target_xy, feather=0.3):
    """Copy the ball `box` (with a little surrounding) to target_xy with a soft edge. None if the patch leaves the image."""
    cx, cy, bw, bh = box
    r = int(max(bw, bh) / 2 * 1.4) + 1
    H, W = gray.shape
    sx0, sy0, sx1, sy1 = int(cx) - r, int(cy) - r, int(cx) + r + 1, int(cy) + r + 1
    tx, ty = int(target_xy[0]), int(target_xy[1])
    dx0, dy0, dx1, dy1 = tx - r, ty - r, tx + r + 1, ty + r + 1
    if min(sx0, sy0, dx0, dy0) < 0 or max(sx1, dx1) > W or max(sy1, dy1) > H:
        return None
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    alpha = np.clip((1.0 - np.sqrt(xx ** 2 + yy ** 2) / r) / max(feather, 1e-3), 0, 1)
    out = gray.copy()
    mixed = alpha * gray[sy0:sy1, sx0:sx1].astype(np.float32) + (1 - alpha) * gray[dy0:dy1, dx0:dx1].astype(np.float32)
    out[dy0:dy1, dx0:dx1] = np.clip(mixed, 0, 255).astype(np.uint8)
    return out


def pick_insert_xy(part, boxes, rng, size, tries=400):
    """A random free spot well inside the rectangle (>= 2 ball sizes from its edge, >= 2.5 sizes from every ball)."""
    poly = part["poly"].reshape(-1, 1, 2).astype(np.float32)
    x0, y0, w0, h0 = cv2.boundingRect(poly)
    for _ in range(tries):
        x, y = rng.uniform(x0, x0 + w0), rng.uniform(y0, y0 + h0)
        if cv2.pointPolygonTest(poly, (float(x), float(y)), True) < 2 * size:
            continue
        if all(math.hypot(x - b[0], y - b[1]) > 2.5 * max(b[2], b[3]) for b in boxes):
            return float(x), float(y)
    return None


def run_counterfactuals(kind, items, seed=VERIFY_SEED):
    """kind = 'remove_one' | 'insert_one' | 'erase_all'. Starts from clean OK images, scored by the detector that never saw them."""
    rng = np.random.default_rng(seed)
    rows = []
    for it in tqdm(items, desc=kind, unit="img"):
        g = gt_by_key[it["key"]]
        if g["part"] is None or g["gt_pred"] != "OK":
            continue
        model = get_model("Original", identity_filter, FOLD_OF[it["key"]])
        if model is None:
            continue
        gray = cv2.imread(it["img"], cv2.IMREAD_GRAYSCALE)
        boxes = [tuple(b) for b in g["gt_boxes"]]
        wh = (g["W"], g["H"])
        inside = judge_boxes([(b[0], b[1]) for b in boxes], g["part"], wh)["inside"]
        in_idx = [i for i, ins in enumerate(inside) if ins]
        if not in_idx:
            continue
        target_xy = tsize = None
        if kind == "remove_one":
            i = int(rng.choice(in_idx))
            tb = boxes[i]
            new_gray, new_boxes = erase_boxes(gray, [tb]), [b for j, b in enumerate(boxes) if j != i]
            target_xy, tsize = (tb[0], tb[1]), max(tb[2], tb[3])
        elif kind == "insert_one":
            src = boxes[int(rng.choice(in_idx))]
            xy = pick_insert_xy(g["part"], boxes, rng, max(src[2], src[3]))
            new_gray = paste_ball(gray, src, xy) if xy is not None else None
            if new_gray is None:
                continue
            new_boxes = boxes + [(xy[0], xy[1], src[2], src[3])]
            target_xy, tsize = xy, max(src[2], src[3])
        elif kind == "erase_all":
            new_gray, new_boxes = erase_boxes(gray, boxes), []
        else:
            raise ValueError(kind)
        xywh, conf = predict_gray(model, new_gray)
        pts = [(float(x), float(y)) for x, y, _, _ in xywh]
        j_new = judge_boxes(pts, g["part"], wh)
        j_exp = judge_boxes([(b[0], b[1]) for b in new_boxes], g["part"], wh)
        st = ball_match_stats(new_boxes, xywh, conf) if new_boxes else dict(miss=0, extra=len(pts))
        near = target_xy is not None and any(math.hypot(x - target_xy[0], y - target_xy[1]) <= 0.5 * tsize for x, y in pts)
        rows.append(dict(key=it["key"], kind=kind, n_det=len(pts), near_target=bool(near), pred=j_new["pred"],
                         exp_pred=j_exp["pred"], counts_equal=bool(j_new["counts"] == j_exp["counts"]),
                         miss=st["miss"], extra=st["extra"]))
    return rows


def permutation_test(rows, n_perm=10000, seed=VERIFY_SEED):
    """Accuracy of the real predictions vs the accuracy they would get against shuffled ground truth."""
    gt = np.array([r["gt_pred"] == "NG" for r in rows])
    pr = np.array([r["pred"] == "NG" for r in rows])
    acc = float((gt == pr).mean())
    rng = np.random.default_rng(seed)
    null = np.array([(rng.permutation(gt) == pr).mean() for _ in range(n_perm)])
    return acc, null, float((1 + (null >= acc).sum()) / (n_perm + 1))


def global_shortcut_baseline(items, size=32):
    """Image-level classifiers without any localisation, scored on the SAME folds as the detector.
    Returns {name: (accuracy %, balanced accuracy %)} or None if scikit-learn is missing."""
    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
    except ImportError:
        print("scikit-learn is not installed (pip install scikit-learn) -> check 5 skipped")
        return None
    thumbs, stats, y, fold = [], [], [], []
    for it in items:
        g = cv2.imread(it["img"], cv2.IMREAD_GRAYSCALE)
        thumbs.append(cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32).ravel())
        stats.append(np.percentile(g, [1, 5, 10, 25, 50, 75, 90, 95, 99]).tolist() + [float(g.mean()), float(g.std())])
        y.append(gt_by_key[it["key"]]["gt_pred"] == "NG")
        fold.append(FOLD_OF[it["key"]])
    y, fold = np.array(y), np.array(fold)
    out = {}
    for name, X in (("32x32 thumbnail", np.array(thumbs)), ("global gray statistics", np.array(stats))):
        pred = np.zeros(len(y), bool)
        for f in sorted(set(fold)):
            tr, te = fold != f, fold == f
            if len(set(y[tr])) < 2:                  # degenerate fold (cannot happen with stratified folds)
                pred[te] = bool(y[tr][0])
                continue
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=3000, class_weight="balanced"))
            clf.fit(X[tr], y[tr])
            pred[te] = clf.predict(X[te])
        rec_ng = (pred & y).sum() / max(y.sum(), 1)
        rec_ok = (~pred & ~y).sum() / max((~y).sum(), 1)
        out[name] = (100.0 * (pred == y).mean(), 50.0 * (rec_ng + rec_ok))
    return out


# ── difficulty strata (from the number of labelled balls; an OK image has exactly 13) ──
STRAT_ORDER = ["OK (13 balls)", "NG same count (wrong regions)", "NG +-1 ball", "NG >=2 balls off"]


def difficulty_of(key):
    g = gt_by_key[key]
    n = len(g["gt_boxes"])
    if g["gt_pred"] == "OK":
        return STRAT_ORDER[0]
    if n == 13:
        return STRAT_ORDER[1]
    return STRAT_ORDER[2] if abs(n - 13) == 1 else STRAT_ORDER[3]


def stratified_table(names):
    recs = []
    for name in names:
        per = PER_IMAGE.get(name)
        if not per:
            continue
        agg = {d: [0, 0, 0] for d in STRAT_ORDER}
        for r in per:
            a = agg[difficulty_of(r["key"])]
            a[0] += 1
            a[1] += int(r["match"])
            a[2] += int(r.get("miss") == 0 and r.get("extra") == 0)
        rec = {"Method": name}
        for d in STRAT_ORDER:
            n_, ok_, st_ = agg[d]
            rec[f"{d} acc"] = 100.0 * ok_ / n_ if n_ else float("nan")
            rec[f"{d} strict"] = 100.0 * st_ / n_ if n_ else float("nan")
        recs.append(rec)
    return pd.DataFrame(recs)


def strict_mcnemar(ref_name, test_name):
    """Paired exact McNemar on 'strict correct' (every labelled ball found, no extra detection anywhere)."""
    def ok(per):
        return {r["key"]: int(r.get("miss") == 0 and r.get("extra") == 0) for r in per}
    ref, tst = ok(PER_IMAGE[ref_name]), ok(PER_IMAGE[test_name])
    keys = ref.keys() & tst.keys()
    b = sum(ref[k] == 1 and tst[k] == 0 for k in keys)
    c = sum(ref[k] == 0 and tst[k] == 1 for k in keys)
    return b, c, mcnemar_exact(b, c)


# ── photometric shifts for the hard mode (no geometry change, so labels and rectangle stay valid) ──
def corrupt(gray, spec, key):
    kind, v = spec["kind"], spec["v"]
    if kind == "blur":
        return cv2.GaussianBlur(gray, (0, 0), float(v))
    if kind == "down":
        h, w = gray.shape
        small = cv2.resize(gray, (max(1, w // int(v)), max(1, h // int(v))), interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    f = gray.astype(np.float32)
    if kind == "noise":
        rng = np.random.default_rng(int(hashlib.md5((key + "|shift").encode("utf-8")).hexdigest()[:8], 16))
        f = f + rng.normal(0.0, float(v), f.shape)
    elif kind == "contrast":
        f = 128.0 + float(v) * (f - 128.0)
    elif kind == "bright":
        f = f + float(v)
    else:
        raise ValueError(kind)
    return np.clip(f, 0, 255).astype(np.uint8)


def quick_metrics(rows):
    n = len(rows)
    tp = sum(r["gt_pred"] == "NG" and r["pred"] == "NG" for r in rows)
    fn = sum(r["gt_pred"] == "NG" and r["pred"] == "OK" for r in rows)
    fp = sum(r["gt_pred"] == "OK" and r["pred"] == "NG" for r in rows)
    tn = sum(r["gt_pred"] == "OK" and r["pred"] == "OK" for r in rows)
    n_gt = sum(r["n_gt"] for r in rows)
    return dict(n=n, acc=100.0 * (tp + tn) / n, bal=50.0 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1)),
                strict=100.0 * sum(r["miss"] == 0 and r["extra"] == 0 for r in rows) / n,
                ball_recall=100.0 * (1 - sum(r["miss"] for r in rows) / n_gt) if n_gt else float("nan"),
                pred_ng=100.0 * (tp + fp) / n)


def run_shift(methods, conds, items, cache_path):
    """Clean-trained detectors (no retraining) judging corrupted held-out images: corrupt -> filter -> detector of the image's fold.
    methods = [(name, filter_fn)], conds = [(label, spec)]. Finished conditions are read from / written to cache_path."""
    res = {}
    if os.path.isfile(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            res = json.load(f)
    for label, spec in conds:
        if label in res and all(m in res[label] for m, _ in methods):
            print(f"[shift] {label}: reusing the saved result")
            continue
        rows = {m: [] for m, _ in methods}
        for it in tqdm(items, desc=f"shift {label}", unit="img"):
            g, fold = gt_by_key[it["key"]], FOLD_OF[it["key"]]
            gray = corrupt(cv2.imread(it["img"], cv2.IMREAD_GRAYSCALE), spec, it["key"])
            for m, fn in methods:
                filtered, _ = apply_filter_gray(gray, fn)
                xywh, conf = predict_gray(get_model(m, fn, fold), filtered)
                row = _row(it, [(float(x), float(y)) for x, y, _, _ in xywh])
                row.update(ball_match_stats(g["gt_boxes"], xywh, conf))
                rows[m].append(row)
        res[label] = {m: quick_metrics(r) for m, r in rows.items()}
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(res, f)
        print(f"[shift] {label}: " + " | ".join(f"{m} {res[label][m]['strict']:.1f}%" for m, _ in methods[:4]) + " ...")
    return res


print("Verification / hard-mode helpers ready")


In [ ]:
# ── 25b. Run the five checks ──────────────────────────────────────────
VERIFICATION = {}
if not USE_CV:
    print("The checks use the CV fold detectors (USE_CV = True) -> skipped.")
else:
    # (1) permutation test on the saved per-image judgements of Original
    acc1, null1, p1 = permutation_test(PER_IMAGE["Original"])
    VERIFICATION["permutation"] = dict(acc=100 * acc1, null_mean=100 * null1.mean(), null_max=100 * null1.max(), p=p1)
    print(f"(1) Permutation: real accuracy {100 * acc1:.2f}% | shuffled-GT accuracy {100 * null1.mean():.1f}% +- {100 * null1.std():.1f} "
          f"(max over {len(null1)} shuffles {100 * null1.max():.1f}%) | p <= {p1:.4g}")

    # (2)-(4) interventions on the clean OK images, judged by the fold detector that never saw them
    _ok_items = [it for it in eval_items if gt_by_key[it["key"]]["gt_pred"] == "OK" and gt_by_key[it["key"]]["part"] is not None]
    cf = {k: run_counterfactuals(k, _ok_items) for k in ("erase_all", "remove_one", "insert_one")}
    pc = lambda rows, f: 100.0 * sum(f(r) for r in rows) / max(len(rows), 1)
    ea, rm, ins = cf["erase_all"], cf["remove_one"], cf["insert_one"]
    VERIFICATION["erase_all"] = dict(n=len(ea), mean_det=float(np.mean([r["n_det"] for r in ea])) if ea else float("nan"),
                                     zero_det_pct=pc(ea, lambda r: r["n_det"] == 0))
    VERIFICATION["remove_one"] = dict(n=len(rm), ghost_pct=pc(rm, lambda r: r["near_target"]), flip_pct=pc(rm, lambda r: r["pred"] == "NG"),
                                      exact_pct=pc(rm, lambda r: r["counts_equal"]),
                                      others_ok_pct=pc(rm, lambda r: r["miss"] == 0))
    VERIFICATION["insert_one"] = dict(n=len(ins), found_pct=pc(ins, lambda r: r["near_target"]), flip_pct=pc(ins, lambda r: r["pred"] == "NG"),
                                      exact_pct=pc(ins, lambda r: r["counts_equal"]))
    print(f"(2) Erase all balls ({len(ea)} images): mean detections {VERIFICATION['erase_all']['mean_det']:.2f} (a real image has 13) | "
          f"{VERIFICATION['erase_all']['zero_det_pct']:.1f}% of images give no detection at all")
    v = VERIFICATION["remove_one"]
    print(f"(3) Remove one ball ({v['n']} images): ghost box at the erased spot {v['ghost_pct']:.1f}% | verdict flips to NG {v['flip_pct']:.1f}% | "
          f"region counts exactly as expected {v['exact_pct']:.1f}% | the other balls all found {v['others_ok_pct']:.1f}%")
    v = VERIFICATION["insert_one"]
    print(f"(4) Insert one ball ({v['n']} images): new ball found {v['found_pct']:.1f}% | verdict flips to NG {v['flip_pct']:.1f}% | "
          f"region counts exactly as expected {v['exact_pct']:.1f}%   (informational: pasted balls are synthetic)")

    # (5) shortcut baseline without any localisation
    sc = global_shortcut_baseline(CV_ITEMS)
    if sc:
        VERIFICATION["shortcut"] = sc
        for k, (a, b) in sc.items():
            print(f"(5) Image-level classifier on {k}: accuracy {a:.1f}% | balanced {b:.1f}%  (majority-class baseline {baseline_result.get('majority_acc', float('nan')):.1f}%)")

    # verdict table
    V = VERIFICATION
    verdict = [
        ("1 permutation", f"real {V['permutation']['acc']:.1f}% vs shuffled {V['permutation']['null_mean']:.1f}%, p<={V['permutation']['p']:.1g}",
         "p < 0.001", V["permutation"]["p"] < 0.001),
        ("2 erase all balls", f"{V['erase_all']['mean_det']:.2f} detections / image", "<= 0.5 per image", V["erase_all"]["mean_det"] <= 0.5),
        ("3 remove one ball", f"ghost {V['remove_one']['ghost_pct']:.1f}%, exact counts {V['remove_one']['exact_pct']:.1f}%",
         "ghost <= 5% and exact >= 90%", V["remove_one"]["ghost_pct"] <= 5 and V["remove_one"]["exact_pct"] >= 90),
        ("4 insert one ball", f"found {V['insert_one']['found_pct']:.1f}%, flip {V['insert_one']['flip_pct']:.1f}%",
         "informational (>= 80% found)", V["insert_one"]["found_pct"] >= 80),
    ]
    if "shortcut" in V:
        worst = max(b for _, b in V["shortcut"].values())
        verdict.append(("5 global shortcut", f"best balanced accuracy {worst:.1f}%", "<= 75% (no shortcut)", worst <= 75))
    vdf = pd.DataFrame([dict(Check=a, Result=b, Criterion=c, Verdict="PASS" if d else "CHECK") for a, b, c, d in verdict])
    display(vdf)
    vdf.to_csv(os.path.join(FILTER_FIG_DIR, "verification_summary.csv"), index=False, encoding="utf-8-sig")
    with open(os.path.join(FILTER_FIG_DIR, "verification_summary.json"), "w", encoding="utf-8") as f:
        json.dump(V, f, ensure_ascii=False, indent=1, default=float)
    print("CHECK means: read the number, do not conclude 'guessing' from one test alone (inpainting can leave a ghost the detector fires on).")


## 26. Hard mode (new in v6): why every earlier result saturated and what changes

| Why it saturated | What the hard mode changes |
|---|---|
| The rectangle rule ignores detections outside the rectangle, so false positives are invisible (Original: accuracy 100% but **Strict Acc 94.8%**; Data 25%: 99.2% vs 35.8%) | **Strict Acc is the primary metric**, plus a per-difficulty table (26a): most NG images differ by a gross ball count, only the *same-count / +-1 ball* NG images test the region rule |
| Train and test come from the same distribution, and each filter gets its own detector trained on the filtered images, so any information-preserving filter is simply re-learned (13 of 15 filters tie) | **Distribution shift without retraining (26b)**: detectors trained on clean images judge corrupted held-out images (noise, blur, low contrast, brightness, low resolution). This is the situation a pre-processing filter is meant for |
| One stress factor at a time sits on a plateau until it falls off a cliff (blur sigma 32 -> 64, data 25% -> 10%) | **Combined stress with retraining (26c)**: low data + noise together, sitting between the plateau and the cliff, so the methods can separate without everything collapsing to "always NG" |
| 5.4% of the images are hard | each table reports **n per stratum and the exact McNemar test on strict accuracy**, so a 1-3 pp gap is not over-read (1 pp = 5 images) |


In [ ]:
# ── 26a. Difficulty strata + strict-accuracy McNemar (re-uses the saved per-image results, no training) ──
_names_all = ["Original"] + [n for n, _ in FILTERS_ROUND1] + [n for n, _ in FILTERS_ROUND2]
_names_all = [n for n in _names_all if n in PER_IMAGE]
print("Images per stratum: " + ", ".join(f"{d}: {sum(difficulty_of(it['key']) == d for it in CV_ITEMS)}" for d in STRAT_ORDER))
strat_df = stratified_table(_names_all)
display(strat_df.round(1))
strat_df.to_csv(os.path.join(FILTER_FIG_DIR, "table_stratified.csv"), index=False, encoding="utf-8-sig")

_rows = []
for nm in _names_all:
    if nm == "Original":
        continue
    b, c, p = strict_mcnemar("Original", nm)
    _s = lambda name: 100.0 * np.mean([int(r.get("miss") == 0 and r.get("extra") == 0) for r in PER_IMAGE[name]])
    _rows.append({"Method": nm, "Strict Acc (%)": _s(nm), "vs Original (pp)": _s(nm) - _s("Original"),
                  "newly worse": b, "newly better": c, "McNemar p": p,
                  "Significant (Bonferroni)": "Yes" if p < 0.05 / max(len(_names_all) - 1, 1) else "No"})
_sdf = pd.DataFrame(_rows).sort_values("Strict Acc (%)", ascending=False)
print(f"\nStrict accuracy of Original: {100.0 * np.mean([int(r.get('miss') == 0 and r.get('extra') == 0) for r in PER_IMAGE['Original']]):.2f}%  "
      f"(Bonferroni threshold {0.05 / max(len(_names_all) - 1, 1):.4f})")
display(_sdf.round({"Strict Acc (%)": 2, "vs Original (pp)": 2, "McNemar p": 4}))
_sdf.to_csv(os.path.join(FILTER_FIG_DIR, "table_strict_mcnemar.csv"), index=False, encoding="utf-8-sig")


In [ ]:
# ── 26b. Distribution shift without retraining (clean-trained detectors, corrupted held-out images) ──
RUN_HARD_SHIFT = True
HARD_QUICK = False          # True: every second condition only
HARD_SHIFT_FILTERS = None   # None = Original + the 10 Round-1 filters (each with its own cached fold detectors)
HARD_SHIFT_CONDITIONS = [
    ("noise 40", dict(kind="noise", v=40)), ("noise 80", dict(kind="noise", v=80)),
    ("blur 8", dict(kind="blur", v=8)), ("blur 16", dict(kind="blur", v=16)),
    ("contrast x0.4", dict(kind="contrast", v=0.4)), ("contrast x0.2", dict(kind="contrast", v=0.2)),
    ("brightness +60", dict(kind="bright", v=60)), ("brightness -60", dict(kind="bright", v=-60)),
    ("down x4", dict(kind="down", v=4)), ("down x8", dict(kind="down", v=8)),
]

SHIFT_RESULTS = {}
if RUN_HARD_SHIFT and not USE_CV:
    print("The shift test uses the CV fold detectors (USE_CV = True) -> skipped.")
elif RUN_HARD_SHIFT:
    _methods = [("Original", identity_filter)] + [(n, f) for n, f in FILTERS_ROUND1 if HARD_SHIFT_FILTERS is None or n in HARD_SHIFT_FILTERS]
    _missing = [m for m, fn in _methods if any(get_model(m, fn, f) is None for f in sorted(set(FOLD_OF.values())))]
    if _missing:
        print(f"No cached fold detectors for {_missing} -> left out (run Sections 14-17 first).")
        _methods = [(m, fn) for m, fn in _methods if m not in _missing]
    _conds = HARD_SHIFT_CONDITIONS[::2] if HARD_QUICK else HARD_SHIFT_CONDITIONS
    _sig = hashlib.md5(json.dumps([CONF_THRES, IOU_THRES, CV_IMGSZ, FOLD_TAG, [m for m, _ in _methods], _conds], sort_keys=True).encode("utf-8")).hexdigest()[:8]
    print(f"Shift test: {len(_methods)} methods x {len(_conds)} conditions x {len(eval_items)} images (inference only)")
    SHIFT_RESULTS = run_shift(_methods, _conds, eval_items, os.path.join(FILTER_FIG_DIR, f"hard_shift_{_sig}.json"))

    labels = [lb for lb, _ in _conds]
    names_m = [m for m, _ in _methods]
    S = pd.DataFrame({lb: {m: SHIFT_RESULTS[lb][m]["strict"] for m in names_m} for lb in labels})
    A = pd.DataFrame({lb: {m: SHIFT_RESULTS[lb][m]["acc"] for m in names_m} for lb in labels})
    S["mean"], A["mean"] = S.mean(axis=1), A.mean(axis=1)
    S["vs Original"] = S["mean"] - S.loc["Original", "mean"]
    print("\nStrict accuracy (%) per condition (a gap below ~3 pp = 15 images is within noise):")
    display(S.round(1).sort_values("mean", ascending=False))
    print("Plain OK/NG accuracy (%):")
    display(A.round(1).sort_values("mean", ascending=False))
    S.round(2).to_csv(os.path.join(FILTER_FIG_DIR, "table_shift_strict.csv"), encoding="utf-8-sig")
    A.round(2).to_csv(os.path.join(FILTER_FIG_DIR, "table_shift_accuracy.csv"), encoding="utf-8-sig")

    _fig, _ax = plt.subplots(figsize=(1.0 * len(labels) + 5, 0.55 * len(names_m) + 2.5))
    _im = _ax.imshow(S[labels].values, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto")
    _ax.set_xticks(range(len(labels)))
    _ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=12)
    _ax.set_yticks(range(len(names_m)))
    _ax.set_yticklabels(names_m, fontsize=12)
    for i in range(len(names_m)):
        for j in range(len(labels)):
            _ax.text(j, i, f"{S[labels].values[i, j]:.0f}", ha="center", va="center", fontsize=11, fontweight="bold")
    _fig.colorbar(_im, ax=_ax, label="Strict accuracy (%)")
    _fig.tight_layout()
    _p = os.path.join(FILTER_FIG_DIR, "fig_hard_shift_strict_heatmap.png")
    save_reference_figure(_fig, _p, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {_p}")


In [ ]:
# ── 26c. Combined stress WITH retraining (low data + noise), Original vs the filters that did best in 26b ──
RUN_HARD_RETRAIN = True
HARD_COND = dict(noise=40, frac=0.25, imgsz=None)     # between the plateau and the cliff of the single-factor sweeps
HARD_FILTERS = ["Gaussian Blur (max)", "Histogram Equalization", "CLAHE (max)", "Gamma 0.3", "Invert"]

HARD_RESULTS = []
if RUN_HARD_RETRAIN and not USE_CV:
    print("The hard retraining needs USE_CV = True -> skipped.")
elif RUN_HARD_RETRAIN:
    _tag = f"noise {HARD_COND['noise']:g}, data {int(round(100 * HARD_COND['frac']))}%" + (f", {HARD_COND['imgsz']}px" if HARD_COND["imgsz"] else "")
    _todo = [("Original", SAME)] + [(n, f) for n, f in FILTERS_ROUND1 if n in HARD_FILTERS and n not in SKIP_FILTERS]
    print(f"Hard retraining ({_tag}): {len(_todo)} methods x {len(_run_folds)} folds = {len(_todo) * len(_run_folds)} trainings")
    for fname, fn in _todo:
        nm = f"HARD {fname} | {_tag}"
        print(f"\n[Hard] {fname} ...")
        r = run_condition(nm, fn, **HARD_COND)
        r["label"] = fname
        r["hard_name"] = nm
        HARD_RESULTS.append(r)
        print("  -> " + _result_line(r))
        if r["pred_ng_pct"] >= 95 or r["pred_ng_pct"] <= 5:
            print("     WARNING: the detector collapsed onto one answer -> this level is too hard to compare methods (lower the noise or raise the data share)")
    ref_nm = HARD_RESULTS[0]["hard_name"]
    rows_h = []
    for r in HARD_RESULTS:
        b, c, p = (0, 0, 1.0) if r["hard_name"] == ref_nm else strict_mcnemar(ref_nm, r["hard_name"])
        rows_h.append({"Method": r["label"], "Accuracy (%)": r["accuracy"], "Strict Acc (%)": r["strict_acc"], "Ball Recall (%)": r["ball_recall"],
                       "mAP50-95 (%)": r["map50_95"], "Fold Acc Std (pp)": r["fold_std"], "Strict: newly worse": b, "Strict: newly better": c, "Strict McNemar p": p})
    hdf = pd.DataFrame(rows_h)
    print(f"\nHard mode ({_tag}); the McNemar test compares strict correctness with Original under the same condition:")
    display(hdf.round(3))
    hdf.to_csv(os.path.join(FILTER_FIG_DIR, "table_hard_retrain.csv"), index=False, encoding="utf-8-sig")
    print("\nPer difficulty stratum:")
    sdf = stratified_table([r["hard_name"] for r in HARD_RESULTS])
    sdf["Method"] = sdf["Method"].map({r["hard_name"]: r["label"] for r in HARD_RESULTS})
    display(sdf.round(1))


In [ ]:
# ── 27. What to report (printed from the numbers above) ──
print("=" * 70)
print("Summary for the paper")
print("=" * 70)
if VERIFICATION:
    for a, b, c, d in verdict:
        print(f"  [{'PASS ' if d else 'CHECK'}] {a}: {b}   (criterion: {c})")
else:
    print("  (run Section 25b first)")
if SHIFT_RESULTS:
    _best = S.drop(index="Original")["mean"].idxmax()
    print(f"\n  Shift test: mean strict accuracy Original {S.loc['Original', 'mean']:.1f}% | best filter {_best} {S.loc[_best, 'mean']:.1f}% "
          f"({S.loc[_best, 'mean'] - S.loc['Original', 'mean']:+.1f} pp) | worst {S['mean'].idxmin()} {S['mean'].min():.1f}%")
if HARD_RESULTS:
    print("\n  Hard retraining (" + _tag + "):")
    for r in sorted(HARD_RESULTS, key=lambda x: -x["strict_acc"]):
        print(f"    {r['label']:<24s} accuracy {r['accuracy']:6.2f}% | strict {r['strict_acc']:6.2f}% | mAP50-95 {r['map50_95']:5.1f}%")
print("\nReport: (a) Strict Acc and the majority-class baseline next to every accuracy, (b) the verification table, "
      "(c) shift / hard-mode tables with the McNemar p-values, (d) the stratified table (n per stratum). Do not report Sections 5-8.")


## 28. v7 diagnostics (inference only - no new training)

Three open questions of the v6 run, answered with the **cached base-seed fold detectors** (`Original`, normal condition) on the held-out images they never trained on:

| # | Question | How it is answered |
|---|---|---|
| 28a | In the *remove-one-ball* test (v6: 7.7 %) a box still appeared at the erased spot. Is that a **memorised position** or an **inpainting artefact**? | The same erasing is done with three methods (OpenCV Telea, OpenCV Navier-Stokes, noise fill) at the ball position **and at a free spot where no ball ever was** (control), plus the untouched image. A position prior would fire at the ball position for every method and not at the free spot; a residue of the ball depends on the method |
| 28b | In v6 an image-level classifier (32x32 thumbnail, no localisation) reached 92.3 % balanced accuracy. Does it read **the balls** or a **global shooting condition**? | (1) the same classifier on images with **all labelled balls inpainted away**, per difficulty stratum; (2) AUC of simple global features (file number, file time, brightness, part size/position) against the label - an AUC far from 0.5 means the label is predictable without any ball |
| 28c | How much do the headline numbers depend on **my choices**: confidence threshold 0.25, match tolerance 0.5 x ball size, and the 13 images where the part was not found (centre fallback)? | Strict / plain accuracy of the Original detector over a grid of thresholds and tolerances; the same numbers without the 13 fallback images |

In [ ]:
# ── 28a. Where do the 'ghost' boxes of the remove-one test come from? ─────────────────────────────────
import re
RUN_DIAG = True
DIAG_MAX_IMAGES = None                          # None = every clean OK image; e.g. 120 for a quick look
GHOST_ERASE_METHODS = ["none", "telea", "ns", "ring"]
DIAG_SEED = VERIFY_SEED + 1


def erase_boxes_m(gray, boxes, method="telea", pad=0.3, seed=0):
    """Like erase_boxes, but the way a ball is painted over can be chosen:
    none = untouched image, telea / ns = two OpenCV inpainting algorithms, ring = noise with the mean / std of the ring around the ball."""
    if method == "none":
        return gray.copy()
    if method == "telea":
        return erase_boxes(gray, boxes, pad)
    out = gray.copy()
    H, W = gray.shape
    rng = np.random.default_rng(seed)
    for cx, cy, bw, bh in boxes:
        r = int(max(bw, bh) / 2 * (1 + pad)) + 1
        x0, x1 = max(0, int(cx) - 3 * r), min(W, int(cx) + 3 * r + 1)
        y0, y1 = max(0, int(cy) - 3 * r), min(H, int(cy) + 3 * r + 1)
        crop = np.ascontiguousarray(out[y0:y1, x0:x1])
        c = (int(cx) - x0, int(cy) - y0)
        m = np.zeros(crop.shape, np.uint8)
        cv2.circle(m, c, r, 255, -1)
        if method == "ns":
            out[y0:y1, x0:x1] = cv2.inpaint(crop, m, 5, cv2.INPAINT_NS)
        elif method == "ring":
            ring = np.zeros(crop.shape, np.uint8)
            cv2.circle(ring, c, int(r * 1.6) + 1, 255, -1)
            ring[m > 0] = 0
            vals = crop[ring > 0].astype(np.float32)
            if vals.size < 10:
                continue
            fill = rng.normal(float(vals.mean()), float(vals.std()), crop.shape)
            new = crop.copy()
            new[m > 0] = np.clip(fill[m > 0], 0, 255).astype(np.uint8)
            out[y0:y1, x0:x1] = new
        else:
            raise ValueError(method)
    return out


def run_ghost_diagnosis(items, methods=GHOST_ERASE_METHODS, seed=DIAG_SEED):
    """One ball per clean OK image is erased (site 'ball'); the same patch size is erased at a free spot (site 'free', control).
    found = a detection within 0.5 x ball size of the site."""
    rng = np.random.default_rng(seed)
    rows = []
    for it in tqdm(items, desc="ghost diagnosis", unit="img"):
        g = gt_by_key[it["key"]]
        if g["part"] is None or g["gt_pred"] != "OK":
            continue
        model = get_model("Original", identity_filter, FOLD_OF[it["key"]])
        if model is None:
            continue
        boxes = [tuple(b) for b in g["gt_boxes"]]
        inside = judge_boxes([(b[0], b[1]) for b in boxes], g["part"], (g["W"], g["H"]))["inside"]
        in_idx = [i for i, ins in enumerate(inside) if ins]
        if not in_idx:
            continue
        tb = boxes[int(rng.choice(in_idx))]
        size = max(tb[2], tb[3])
        free = pick_insert_xy(g["part"], boxes, rng, size)
        gray = cv2.imread(it["img"], cv2.IMREAD_GRAYSCALE)
        sites = [("ball", (tb[0], tb[1]))] + ([("free", free)] if free is not None else [])
        for meth in methods:
            for site, (sx, sy) in sites:
                img = erase_boxes_m(gray, [(sx, sy, tb[2], tb[3])], meth, seed=int(rng.integers(1 << 30)))
                xywh, conf = predict_gray(model, img)
                near = [float(c) for (x, y, _, _), c in zip(xywh, conf) if math.hypot(x - sx, y - sy) <= 0.5 * size]
                rows.append(dict(key=it["key"], method=meth, site=site, found=bool(near),
                                 conf=max(near) if near else 0.0, n_det=len(xywh)))
    return rows


def ghost_summary(rows):
    """(table of found-% per method x site, n per cell, verdict text)."""
    gdf = pd.DataFrame(rows)
    piv = 100.0 * gdf.pivot_table(index="method", columns="site", values="found", aggfunc="mean")
    n = gdf.groupby(["method", "site"]).size().unstack()
    piv = piv.reindex([m for m in GHOST_ERASE_METHODS if m in piv.index])
    er = [m for m in ("telea", "ns", "ring") if m in piv.index]
    ball = piv.loc[er, "ball"] if "ball" in piv else None
    free = piv.loc[er, "free"] if "free" in piv else None
    if ball is None or free is None or not er:
        return piv, n, "not enough data for a verdict"
    spread, gap = float(ball.max() - ball.min()), float((ball - free).min())
    if float(ball.max()) <= 2.0:
        v = "After erasing, a box at the erased spot is (almost) never found -> no ghost: the detector does not keep predicting balls where they were removed."
    elif gap <= 2.0:
        v = ("The boxes appear at free spots as often as at the erased ball -> they are generic false positives of the erased/inpainted look, "
             "not a memory of ball positions.")
    elif spread >= 5.0:
        v = ("The rate depends strongly on how the ball was erased (spread %.1f pp) -> the ghost is a residue of the ball / the inpainting, "
             "not a memory of positions." % spread)
    else:
        v = ("The rate is similar for every erase method (spread %.1f pp) and higher than at free spots (>= %.1f pp) -> a position prior is possible; "
             "do not claim the detector is purely appearance-based, report the number instead." % (spread, gap))
    return piv, n, v


GHOST_ROWS = []
if RUN_DIAG and not USE_CV:
    print("The diagnostics use the CV fold detectors (USE_CV = True) -> skipped.")
elif RUN_DIAG:
    _it = [it for it in eval_items if gt_by_key[it["key"]]["gt_pred"] == "OK" and gt_by_key[it["key"]]["part"] is not None]
    if DIAG_MAX_IMAGES and len(_it) > DIAG_MAX_IMAGES:
        _it = random.Random(DIAG_SEED).sample(_it, DIAG_MAX_IMAGES)
    GHOST_ROWS = run_ghost_diagnosis(_it)
    if GHOST_ROWS:
        _piv, _n, _verdict = ghost_summary(GHOST_ROWS)
        print("\n(28a) Detection within 0.5 x ball size of the erased site (%):")
        display(_piv.round(1))
        print("images per cell:")
        display(_n)
        _gdf = pd.DataFrame(GHOST_ROWS)
        print("mean confidence of those detections:", _gdf[_gdf.found].groupby(["method", "site"]).conf.mean().round(2).to_dict())
        print("\nVerdict:", _verdict)
        print("Reading guide: 'none'/'ball' is the untouched image (should be ~100 %: the ball itself); 'none'/'free' is the natural false-positive rate of the detector (should be ~0 %).")
        _gdf.to_csv(os.path.join(FILTER_FIG_DIR, "diag_ghost_rows.csv"), index=False, encoding="utf-8-sig")
        _piv.round(2).to_csv(os.path.join(FILTER_FIG_DIR, "diag_ghost_table.csv"), encoding="utf-8-sig")
    else:
        print("No usable images (no cached Original fold detectors?).")

In [ ]:
# ── 28b. Shortcut diagnosis: balls or global shooting condition? ───────────────────────────────────────
def thumb_cv(items, loader, size=32, C=0.05):
    """Image-level classifier (logistic regression on a size x size thumbnail, no localisation) scored on the SAME folds as the detector.
    Returns (pred NG bool array, y NG bool array) or None when scikit-learn is missing."""
    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
    except ImportError:
        print("scikit-learn is not installed (pip install scikit-learn) -> skipped")
        return None
    X = np.array([cv2.resize(loader(it), (size, size), interpolation=cv2.INTER_AREA).astype(np.float32).ravel() for it in items])
    y = np.array([gt_by_key[it["key"]]["gt_pred"] == "NG" for it in items])
    fold = np.array([FOLD_OF[it["key"]] for it in items])
    pred = np.zeros(len(y), bool)
    for f in sorted(set(fold)):
        tr, te = fold != f, fold == f
        if len(set(y[tr])) < 2:
            pred[te] = bool(y[tr][0])
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000, class_weight="balanced"))
        clf.fit(X[tr], y[tr])
        pred[te] = clf.predict(X[te])
    return pred, y


def _acc_bal(pred, y):
    rec_ng = (pred & y).sum() / max(y.sum(), 1)
    rec_ok = (~pred & ~y).sum() / max((~y).sum(), 1)
    return 100.0 * (pred == y).mean(), 50.0 * (rec_ng + rec_ok)


def stratum_acc(items, pred, y):
    """Accuracy (%) of an image-level classifier per difficulty stratum + n."""
    out = {}
    strata = np.array([difficulty_of(it["key"]) for it in items])
    for d in STRAT_ORDER:
        sel = strata == d
        out[d] = (100.0 * (pred[sel] == y[sel]).mean() if sel.any() else float("nan"), int(sel.sum()))
    return out


def load_raw(it):
    return cv2.imread(it["img"], cv2.IMREAD_GRAYSCALE)


def load_erased(it):
    g = gt_by_key[it["key"]]
    return erase_boxes(load_raw(it), [tuple(b) for b in g["gt_boxes"]])


def _rank_auc_perm(x, y, n_perm=2000, seed=0):
    """AUC of feature x for the label y (True = NG) from ranks, and a two-sided permutation p-value. NaNs are dropped."""
    x, y = np.asarray(x, float), np.asarray(y, bool)
    ok = ~np.isnan(x)
    x, y = x[ok], y[ok]
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0 or len(x) < 10:
        return float("nan"), float("nan"), len(x)
    r = pd.Series(x).rank(method="average").to_numpy()
    base = n1 * (n1 + 1) / 2.0
    auc = (r[y].sum() - base) / (n1 * n0)
    rng = np.random.default_rng(seed)
    cnt = 0
    for _ in range(n_perm):
        a = (r[rng.permutation(y)].sum() - base) / (n1 * n0)
        cnt += abs(a - 0.5) >= abs(auc - 0.5) - 1e-12
    return float(auc), float((cnt + 1) / (n_perm + 1)), len(x)


def global_features(items):
    """Features that say nothing about single balls: file number, file time, global brightness, size / position of the part."""
    feats = {k: [] for k in ("file number", "file time", "mean gray", "gray std", "part area share", "part centre x", "part centre y")}
    for it in items:
        g = gt_by_key[it["key"]]
        num = re.findall(r"\d+", it["stem"])
        feats["file number"].append(float(num[0]) if num else float("nan"))
        try:
            feats["file time"].append(float(os.path.getmtime(it["img"])))
        except OSError:
            feats["file time"].append(float("nan"))
        gray = load_raw(it)
        feats["mean gray"].append(float(gray.mean()))
        feats["gray std"].append(float(gray.std()))
        part = g["part"]
        feats["part area share"].append(float("nan") if part is None else float(part["area_frac"]))
        feats["part centre x"].append(float("nan") if part is None else float(part["cx"]) / g["W"])
        feats["part centre y"].append(float("nan") if part is None else float(part["cy"]) / g["H"])
    return {k: np.array(v, float) for k, v in feats.items()}


SHORTCUT_DIAG = {}
if RUN_DIAG and not USE_CV:
    pass
elif RUN_DIAG:
    print("(28b-1) image-level classifier on raw vs ball-erased images (same folds as the detector)")
    _items = CV_ITEMS
    _raw = thumb_cv(_items, load_raw)
    if _raw is not None:
        _t0 = time.time()
        _ers = thumb_cv(_items, load_erased)
        print(f"   erased-image classifier finished ({time.time() - _t0:.0f} s)")
        _y = _raw[1]
        _rows = []
        for nm, (pr, _) in (("raw 32x32 thumbnail", _raw), ("all balls inpainted away", _ers)):
            a, b = _acc_bal(pr, _y)
            sa = stratum_acc(_items, pr, _y)
            _rows.append(dict(Input=nm, **{"Accuracy (%)": a, "Balanced (%)": b}, **{f"{d} (n={sa[d][1]})": sa[d][0] for d in STRAT_ORDER}))
        _sdf = pd.DataFrame(_rows)
        display(_sdf.round(1))
        _sdf.to_csv(os.path.join(FILTER_FIG_DIR, "diag_shortcut_thumbnail.csv"), index=False, encoding="utf-8-sig")
        SHORTCUT_DIAG["thumbnail"] = _rows
        _b_raw, _b_ers = _rows[0]["Balanced (%)"], _rows[1]["Balanced (%)"]
        if _b_ers >= _b_raw - 5:
            print(f"   -> erasing the balls barely changes the score ({_b_raw:.1f}% -> {_b_ers:.1f}%): the classifier does NOT need the balls, "
                  "the label is predictable from the rest of the image (global shooting condition / part appearance).")
        elif _b_ers <= 60:
            print(f"   -> without the balls the classifier falls to ~chance ({_b_ers:.1f}%): the thumbnail classifier of v6 was reading the balls.")
        else:
            print(f"   -> partly: {_b_raw:.1f}% -> {_b_ers:.1f}%; part of the signal is in the balls, part elsewhere.")
        print("   caveat: inpainting leaves a trace per erased ball, so an erased OK image (13 traces) differs slightly from an erased NG image (11-15 traces).")

    print("\n(28b-2) can the label be predicted from global features alone? (AUC 0.5 = no information; permutation p, 2000 shuffles)")
    _feat = global_features(CV_ITEMS)
    _yy = np.array([gt_by_key[it["key"]]["gt_pred"] == "NG" for it in CV_ITEMS])
    _rows = []
    for k, x in _feat.items():
        auc, p, n_ = _rank_auc_perm(x, _yy)
        _rows.append({"Feature": k, "n": n_, "AUC (NG vs OK)": auc, "|AUC-0.5|x2": abs(auc - 0.5) * 2 if not math.isnan(auc) else float("nan"), "perm. p": p})
    _adf = pd.DataFrame(_rows)
    display(_adf.round(3))
    _adf.to_csv(os.path.join(FILTER_FIG_DIR, "diag_global_feature_auc.csv"), index=False, encoding="utf-8-sig")
    SHORTCUT_DIAG["global_auc"] = _rows
    _strong = [r for r in _rows if r["perm. p"] < 0.01 and abs(r["AUC (NG vs OK)"] - 0.5) >= 0.1]
    if _strong:
        print("   -> label-related global features: " + ", ".join(f"{r['Feature']} (AUC {r['AUC (NG vs OK)']:.2f})" for r in _strong)
              + " -> OK and NG images differ in more than the balls (shooting order / lighting / part placement).")
    else:
        print("   -> no global feature separates OK from NG (all |AUC-0.5| < 0.1 or p >= 0.01).")

In [ ]:
# ── 28c. Operating-point sensitivity of the Original detector (conf threshold x match tolerance) ──────────
OP_CONFS = [0.10, 0.25, 0.40, 0.50, 0.60]
OP_TOLS = [0.3, 0.5, 0.7]


def predict_conf_floor(model, gray, conf_floor):
    pred = model.predict(cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR), imgsz=CV_IMGSZ, conf=conf_floor, iou=IOU_THRES, verbose=False)[0]
    xywh = pred.boxes.xywh.cpu().numpy() if len(pred.boxes) else np.zeros((0, 4))
    conf = pred.boxes.conf.cpu().numpy() if len(pred.boxes) else np.zeros(0)
    return xywh, conf


def operating_point_table(preds, items, confs, tols, skip_keys=()):
    """preds[key] = (xywh, conf) at the lowest threshold. Raising the threshold afterwards equals a run at that threshold
    (NMS keeps the highest-scoring box first, so low-score boxes never suppress high-score ones).
    Returns (strict DataFrame conf x tol, accuracy Series per conf, n images)."""
    skip = set(skip_keys)
    use = [it for it in items if it["key"] not in skip and it["key"] in preds]
    strict = {c: {} for c in confs}
    acc = {}
    for c in confs:
        ok_acc, st = 0, {t: 0 for t in tols}
        for it in use:
            g = gt_by_key[it["key"]]
            xywh, cf = preds[it["key"]]
            sel = cf >= c
            xs, cs = xywh[sel], cf[sel]
            pj = judge_boxes([(float(x), float(y)) for x, y, _, _ in xs], g["part"], (g["W"], g["H"]))
            ok_acc += int(pj["pred"] == g["gt_pred"])
            for t in tols:
                m = ball_match_stats(g["gt_boxes"], xs, cs, tol_frac=t)
                st[t] += int(m["miss"] == 0 and m["extra"] == 0)
        acc[c] = 100.0 * ok_acc / max(len(use), 1)
        for t in tols:
            strict[c][t] = 100.0 * st[t] / max(len(use), 1)
    sdf = pd.DataFrame(strict).T
    sdf.index.name, sdf.columns.name = "conf", "tol"
    return sdf, pd.Series(acc, name="accuracy"), len(use)


OP_RESULT = {}
if RUN_DIAG and USE_CV:
    _preds, _miss = {}, 0
    for it in tqdm(eval_items, desc="operating point", unit="img"):
        model = get_model("Original", identity_filter, FOLD_OF[it["key"]])
        if model is None:
            _miss += 1
            continue
        _preds[it["key"]] = predict_conf_floor(model, load_raw(it), min(OP_CONFS))
    if _preds:
        _sdf, _adf2, _n = operating_point_table(_preds, eval_items, OP_CONFS, OP_TOLS)
        print(f"\n(28c) Strict accuracy (%) of Original, {_n} images (rows: confidence threshold, columns: match tolerance x ball size)")
        display(_sdf.round(1))
        print("Plain OK/NG accuracy (%) per confidence threshold:", _adf2.round(1).to_dict())
        _ref = np.mean([int(r.get("miss") == 0 and r.get("extra") == 0) for r in PER_IMAGE["Original"]]) * 100 if PER_IMAGE.get("Original") else float("nan")
        print(f"Sanity: the cell (0.25, 0.5) should reproduce the CV result ({_ref:.1f}%): {_sdf.loc[0.25, 0.5]:.1f}%" if 0.25 in OP_CONFS and 0.5 in OP_TOLS else "")
        _fail = [it["key"] for it in eval_items if gt_by_key[it["key"]].get("loc_fail")]
        _sdf2, _adf3, _n2 = operating_point_table(_preds, eval_items, [0.25], [0.5], skip_keys=_fail)
        print(f"Without the {len(_fail)} images where the part was not found (centre fallback): {_n2} images | strict {_sdf2.loc[0.25, 0.5]:.1f}% | accuracy {_adf3[0.25]:.1f}%")
        _sdf.round(2).to_csv(os.path.join(FILTER_FIG_DIR, "diag_operating_point_strict.csv"), encoding="utf-8-sig")
        OP_RESULT = dict(strict=_sdf.round(2).to_dict(), acc=_adf2.round(2).to_dict(), n_fail=len(_fail),
                         strict_without_fail=float(_sdf2.loc[0.25, 0.5]), acc_without_fail=float(_adf3[0.25]))
    else:
        print("No cached Original fold detectors -> skipped.")

## 29. Agreement with the real OK/NG list (optional - fill in one of the three settings)

All accuracies so far are against a **label-derived** ground truth (same rectangle, same rule). If you know which images are *really* defective, give it here and this section reports how well

1. the label rule itself agrees with the real list (95 % Wilson interval), and which images disagree (saved as CSV for you to look at),
2. every method's judgement agrees with the real list (the cached per-image results of Sections 14-17 are re-scored, **no training**),
3. the detector agrees with the real list **better or worse than the label rule does** (paired exact McNemar).

| Setting | Meaning |
|---|---|
| `REAL_LABEL_CSV` | CSV with columns `name,label` (`label` = OK / NG; `name` with or without extension) |
| `REAL_LABEL_DIRS` | `{"OK": "/path/to/OK", "NG": "/path/to/NG"}` - folders that contain the original photos sorted by the real result (matched by file name) |
| `REAL_LABEL_FROM_NAME` | `True` if the file names themselves contain `OK` / `NG` |

File names are matched exactly first, then by prefix (a name like `img001_png.rf.xxxx` matches `img001`) - only if the match is unique. The cell prints how many images were matched; if it is far below 500, check the names.

In [ ]:
# ── 29. Real OK/NG list ───────────────────────────────────────────────────────────────────────────────
REAL_LABEL_CSV = None            # e.g. r"/root/추계학술대회/real_okng.csv"   (columns: name,label)
REAL_LABEL_DIRS = {}             # e.g. {"OK": r"/root/.../OK", "NG": r"/root/.../NG"}
REAL_LABEL_FROM_NAME = False
REAL_LABEL_COLUMNS = ("name", "label")


def _norm_label(v):
    s = str(v).strip().upper()
    if s in ("OK", "GOOD", "PASS", "양품", "정상"):
        return "OK"
    if s in ("NG", "BAD", "FAIL", "불량"):
        return "NG"
    return None


def load_real_labels(csv_path=None, dirs=None, from_name=False, items=()):
    """stem -> 'OK' / 'NG' from a CSV, from folders, and / or from the file names."""
    real = {}
    if csv_path and os.path.isfile(csv_path):
        with open(csv_path, newline="", encoding="utf-8-sig") as f:
            for r in csv.DictReader(f):
                nm, lab = (r.get(REAL_LABEL_COLUMNS[0]) or "").strip(), _norm_label(r.get(REAL_LABEL_COLUMNS[1]) or "")
                if nm and lab:
                    real[os.path.splitext(os.path.basename(nm))[0]] = lab
    for lab, d in (dirs or {}).items():
        lab = _norm_label(lab)
        if lab is None or not os.path.isdir(d):
            continue
        for p in glob.glob(os.path.join(d, "**", "*"), recursive=True):
            if p.lower().endswith(IMG_EXTS):
                real[os.path.splitext(os.path.basename(p))[0]] = lab
    if from_name:
        for it in items:
            low = it["stem"].lower()
            m = re.search(r"(?<![a-z])(ok|ng)(?![a-z])", low)
            lab = m.group(1).upper() if m else ("OK" if "양품" in low else ("NG" if "불량" in low else None))
            if lab:
                real[it["stem"]] = lab
    return real


def match_real_labels(real, items):
    """key -> label. Exact stem first, then a unique prefix match (followed by a non-alphanumeric character or the end)."""
    by_stem = {}
    for it in items:
        by_stem.setdefault(it["stem"], []).append(it["key"])
    out, ambiguous, unmatched = {}, [], []
    for rs, lab in real.items():
        if rs in by_stem:
            for k in by_stem[rs]:
                out[k] = lab
            continue
        pat = re.compile(re.escape(rs) + r"(?![0-9A-Za-z])")
        cands = [k for st, ks in by_stem.items() if pat.match(st) for k in ks]
        if len(cands) == 1:
            out[cands[0]] = lab
        elif cands:
            ambiguous.append(rs)
        else:
            unmatched.append(rs)
    return out, ambiguous, unmatched


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100.0 * (c - h), 100.0 * (c + h)


REAL_RESULT = {}
if REAL_LABEL_CSV or REAL_LABEL_DIRS or REAL_LABEL_FROM_NAME:
    _real = load_real_labels(REAL_LABEL_CSV, REAL_LABEL_DIRS, REAL_LABEL_FROM_NAME, CV_ITEMS)
    REAL_BY_KEY, _amb, _unm = match_real_labels(_real, CV_ITEMS)
    print(f"Real labels read: {len(_real)} | matched to dataset images: {len(REAL_BY_KEY)} of {len(CV_ITEMS)} | ambiguous: {len(_amb)} | not found in the dataset: {len(_unm)}")
    if _unm:
        print("   e.g. not found:", _unm[:5], "| dataset names look like:", [it["stem"] for it in CV_ITEMS[:3]])
    if len(REAL_BY_KEY) < 0.5 * len(CV_ITEMS):
        print("WARNING: fewer than half of the images are matched -> check REAL_LABEL_* and the file names before using the numbers below.")
    _keys = [k for k in REAL_BY_KEY if k in gt_by_key]
    if _keys:
        _agree = sum(gt_by_key[k]["gt_pred"] == REAL_BY_KEY[k] for k in _keys)
        lo, hi = wilson_ci(_agree, len(_keys))
        print(f"\n(29-1) Label rule vs real list: {_agree}/{len(_keys)} = {100 * _agree / len(_keys):.2f}%  (95% Wilson CI {lo:.1f} - {hi:.1f})")
        cm = pd.crosstab(pd.Series([REAL_BY_KEY[k] for k in _keys], name="real"), pd.Series([gt_by_key[k]["gt_pred"] for k in _keys], name="label rule"))
        display(cm)
        _dis = [dict(key=k, real=REAL_BY_KEY[k], label_rule=gt_by_key[k]["gt_pred"], n_balls=len(gt_by_key[k]["gt_boxes"]),
                     counts=json.dumps(gt_by_key[k]["gt_counts"]), loc_fail=bool(gt_by_key[k].get("loc_fail")))
                for k in _keys if gt_by_key[k]["gt_pred"] != REAL_BY_KEY[k]]
        pd.DataFrame(_dis).to_csv(os.path.join(FILTER_FIG_DIR, "real_label_disagreements.csv"), index=False, encoding="utf-8-sig")
        print(f"   {len(_dis)} disagreeing images saved to real_label_disagreements.csv "
              f"(rule OK / real NG: {sum(d['real'] == 'NG' for d in _dis)} -> defect not visible in the ball pattern; "
              f"rule NG / real OK: {sum(d['real'] == 'OK' for d in _dis)} -> labelling or rectangle problem)")
        _rows = []
        for nm, per in PER_IMAGE.items():
            sub = [r for r in per if r["key"] in REAL_BY_KEY]
            if len(sub) < 20 or nm.startswith(("HARD ", "Noise std", "Blur sigma", "Data ", "Res ")) or "| noise" in nm:
                continue
            a = sum(r["pred"] == REAL_BY_KEY[r["key"]] for r in sub)
            _rows.append({"Method": nm, "n": len(sub), "Agreement with real (%)": 100.0 * a / len(sub),
                          "Agreement with label rule (%)": 100.0 * sum(r["pred"] == r["gt_pred"] for r in sub) / len(sub)})
        if _rows:
            _rdf = pd.DataFrame(_rows).sort_values("Agreement with real (%)", ascending=False)
            print("\n(29-2) Every method against the real list (normal condition):")
            display(_rdf.round(2))
            _rdf.round(3).to_csv(os.path.join(FILTER_FIG_DIR, "real_label_methods.csv"), index=False, encoding="utf-8-sig")
        if "Original" in PER_IMAGE:
            _o = {r["key"]: r for r in PER_IMAGE["Original"] if r["key"] in REAL_BY_KEY}
            _b = sum(gt_by_key[k]["gt_pred"] == REAL_BY_KEY[k] and _o[k]["pred"] != REAL_BY_KEY[k] for k in _o)
            _c = sum(gt_by_key[k]["gt_pred"] != REAL_BY_KEY[k] and _o[k]["pred"] == REAL_BY_KEY[k] for k in _o)
            print(f"\n(29-3) Original detector vs label rule, both judged against the real list: rule right / detector wrong {_b}, "
                  f"rule wrong / detector right {_c}, exact McNemar p = {mcnemar_exact(_b, _c):.4f}")
            _ok_real = [k for k in _o if REAL_BY_KEY[k] == "NG"]
            _ng_rec = sum(_o[k]["pred"] == "NG" for k in _ok_real)
            print(f"   real-NG images the detector flags: {_ng_rec}/{len(_ok_real)} ({100 * _ng_rec / max(len(_ok_real), 1):.1f}%) | "
                  f"real-OK images it passes: {sum(_o[k]['pred'] == 'OK' for k in _o if REAL_BY_KEY[k] == 'OK')}/{sum(REAL_BY_KEY[k] == 'OK' for k in _o)}")
        REAL_RESULT = dict(n_matched=len(_keys), rule_agree=_agree, rule_agree_pct=100.0 * _agree / len(_keys), ci=[lo, hi], n_disagree=len(_dis))
else:
    print("Real OK/NG list not given (REAL_LABEL_CSV / REAL_LABEL_DIRS / REAL_LABEL_FROM_NAME) -> section skipped; the paper keeps this as a limitation.")

## 30. Multi-seed replication (the biggest open limitation of v6: one run per condition)

Every v6 number is **one training run per fold with seed 42**. The hard-mode ranking (CLAHE 61.6 > HistEq 47.6 > Gamma 44.4 > Original 41.8 > ...) can therefore be a lucky run: the fold-to-fold std of Original was 3.5 pp, and two single-factor stress levels were not even ordered consistently.

This section repeats the key conditions with **new seeds** (`MS_SEEDS`). One seed changes **three things at once**: the fold assignment (which images are held out), the inner validation split, and the YOLO training seed (initialisation / augmentation / batch order). For noise conditions the noise realisation changes as well. The base seed (42) is **not retrained** - its results come from the caches of Sections 14-26, because the cache keys of the base seed are unchanged.

| Condition | Methods (defaults) | Why |
|---|---|---|
| normal | Original, Invert, CLAHE, Invert+CLAHE | the only significant normal-condition effects of v6 (Invert+CLAHE +3.2 pp, Invert -4.6 pp) |
| hard (noise 40 + 25 % data) | Original, CLAHE, HistEq, Gamma 0.3, Gaussian Blur, Invert | Table 2 of the paper |
| noise 80 (full data) | Original, Gaussian Blur, CLAHE | where Gaussian Blur beat CLAHE |

The report (30b) gives, per method and condition: strict accuracy per seed, mean +- std over seeds, the paired difference to Original (mean over seeds with a **cluster bootstrap 95 % CI** that resamples near-duplicate groups, not single images), the number of seeds in which the method beats Original, and the per-seed exact McNemar p. A method is called **robust** only if all seeds agree in sign **and** the CI excludes 0.

Set `RUN_MULTISEED = False` to skip, `MS_SEEDS = [42, 43]` for one extra seed (half the time), or remove methods / conditions from `MS_CONDITIONS`. Runs are cached; an interrupted run continues where it stopped.

In [ ]:
# ── 30a. Multi-seed runs ──────────────────────────────────────────────────────────────────────────────
RUN_MULTISEED = True
MS_SEEDS = [42, 43, 44]
MS_BASE_SEED = SEED                      # the seed of Sections 1-27 (its results are read from the caches, not retrained)
MS_CONDITIONS = {
    "normal": dict(kind="normal", noise=0.0, frac=1.0, imgsz=None,
                   methods=["Original", "Invert", "CLAHE (max)", "Invert+CLAHE"]),
    "hard (noise 40 + 25% data)": dict(kind="hard", noise=40, frac=0.25, imgsz=None,
                                       methods=["Original", "CLAHE (max)", "Histogram Equalization", "Gamma 0.3", "Gaussian Blur (max)", "Invert"]),
    "noise 80": dict(kind="noise", noise=80, frac=1.0, imgsz=None,
                     methods=["Original", "Gaussian Blur (max)", "CLAHE (max)"]),
}
_SEED_TAG = ""                           # "" for the base seed, "_s43" ... otherwise (part of every cache key)


def _noisy(gray, key):                   # [v7] same as v6 for the base seed; a new noise realisation for every other seed
    if not STRESS_NOISE_STD:
        return gray
    k = key + _SEED_TAG
    rng = np.random.default_rng(int(hashlib.md5(k.encode("utf-8")).hexdigest()[:8], 16))
    return np.clip(gray.astype(np.float32) + rng.normal(0.0, STRESS_NOISE_STD, gray.shape), 0, 255).astype(np.uint8)


def set_condition(noise=0.0, train_frac=1.0, imgsz=None):      # [v7] noisy copies of other seeds get their own folder
    global STRESS_NOISE_STD, STRESS_TRAIN_FRAC, STRESS_IMGSZ, CV_IMGSZ, _STRESS_TAG
    STRESS_NOISE_STD, STRESS_TRAIN_FRAC, STRESS_IMGSZ = noise, train_frac, imgsz
    CV_IMGSZ = imgsz or IMGSZ
    _STRESS_TAG = (f"_n{noise:g}" + _SEED_TAG) if noise else ""


class SeedContext:
    """with SeedContext(43): ...  -> new folds, new inner val split, new training seed, new noise realisation, separate cache keys."""

    def __init__(self, seed):
        self.seed = seed

    def __enter__(self):
        g = globals()
        self.saved = {k: g[k] for k in ("SEED", "FOLD_OF", "FOLD_TAG", "_SEED_TAG")}
        g["_MODELS"].clear()
        if self.seed != MS_BASE_SEED:
            g["SEED"] = self.seed
            g["_SEED_TAG"] = f"_s{self.seed}"
            g["FOLD_OF"] = assign_folds(CV_ITEMS, CV_FOLDS, self.seed)
            g["FOLD_TAG"] = self.saved["FOLD_TAG"] + g["_SEED_TAG"]
            set_condition(STRESS_NOISE_STD, STRESS_TRAIN_FRAC, STRESS_IMGSZ)
        return self

    def __exit__(self, *exc):
        g = globals()
        g.update(self.saved)
        g["_MODELS"].clear()
        set_condition(STRESS_NOISE_STD, STRESS_TRAIN_FRAC, STRESS_IMGSZ)
        return False


def _ms_hard_tag(noise, frac, imgsz):
    return f"noise {noise:g}, data {int(round(100 * frac))}%" + (f", {imgsz}px" if imgsz else "")


def ms_method_name(kind, method, noise, frac, imgsz):
    """The names under which Sections 14-26 saved these runs (so the base seed reuses their caches)."""
    if kind == "normal":
        return method
    if kind == "hard":
        return f"HARD {method} | {_ms_hard_tag(noise, frac, imgsz)}"
    if kind == "noise":
        return f"Noise std={noise:g}" if method == "Original" else f"{method} | noise {noise:g}"
    raise ValueError(kind)


def ms_filter(method):
    if method == "Original":
        return SAME
    allf = dict(FILTERS_ROUND1 + FILTERS_ROUND2)
    if method not in allf:
        raise KeyError(f"unknown filter {method!r}; known: {sorted(allf)}")
    return allf[method]


_RES_KEYS = ("accuracy", "bal_acc", "strict_acc", "ball_recall", "map50_95", "fold_std", "pred_ng_pct", "ng_recall", "ok_recall")
MS_ROWS, MS_RES = {}, {}                 # (condition, seed, method) -> per-image rows / scalar results


def ms_cleanup(name, fn, noise, frac, imgsz):
    """Delete the filtered copy of a finished run (several GB). The shared 'original' folder is never touched."""
    if fn is SAME and not noise:
        return
    prev = (STRESS_NOISE_STD, STRESS_TRAIN_FRAC, STRESS_IMGSZ)
    set_condition(noise, frac, imgsz)
    try:
        cleanup_filtered(name, fn)
    finally:
        set_condition(*prev)


def run_multiseed(conditions, seeds):
    global SWEEP_CLEAN_FILTERED
    snapshot = dict(PER_IMAGE)
    clean_prev = SWEEP_CLEAN_FILTERED
    SWEEP_CLEAN_FILTERED = False         # cleaned here instead: a filtered copy is reused by all seeds when it has no noise
    try:
        for cond, spec in conditions.items():
            for method in spec["methods"]:
                fn = ms_filter(method)
                nm = ms_method_name(spec["kind"], method, spec["noise"], spec["frac"], spec["imgsz"])
                for si, seed in enumerate(seeds):
                    print(f"\n[multi-seed] {cond} | {method} | seed {seed}" + ("  (base seed: from the caches)" if seed == MS_BASE_SEED else ""))
                    with SeedContext(seed):
                        r = run_condition(nm, fn, spec["noise"], spec["frac"], spec["imgsz"])
                        MS_ROWS[(cond, seed, method)] = [dict(x) for x in PER_IMAGE[nm]]
                        MS_RES[(cond, seed, method)] = {k: r.get(k) for k in _RES_KEYS}
                        print("  -> " + _result_line(r))
                        if spec["noise"] or si == len(seeds) - 1:
                            ms_cleanup(nm, fn, spec["noise"], spec["frac"], spec["imgsz"])
    finally:
        SWEEP_CLEAN_FILTERED = clean_prev
        PER_IMAGE.clear()
        PER_IMAGE.update(snapshot)       # the multi-seed rows must not overwrite the base-seed rows of the later sections


def ms_count_trainings(conditions, seeds):
    n = 0
    for spec in conditions.values():
        n += len(spec["methods"]) * len([s for s in seeds if s != MS_BASE_SEED]) * CV_FOLDS
    return n


if RUN_MULTISEED and not USE_CV:
    print("Multi-seed needs USE_CV = True -> skipped.")
elif RUN_MULTISEED:
    _n = ms_count_trainings(MS_CONDITIONS, MS_SEEDS)
    print(f"Multi-seed: seeds {MS_SEEDS} (base seed {MS_BASE_SEED} from the caches), {_n} new trainings of {FILTER_EPOCHS} epochs "
          f"-> roughly {_n * 1.5 / 60:.1f} - {_n * 2.5 / 60:.1f} h on a fast GPU (data-25% runs are faster). Cached runs are skipped.")
    run_multiseed(MS_CONDITIONS, MS_SEEDS)
    _rows_out = [dict(condition=c, seed=s, method=m, key=r["key"], gt_pred=r["gt_pred"], pred=r["pred"], match=r["match"],
                      miss=r.get("miss"), extra=r.get("extra")) for (c, s, m), rows in MS_ROWS.items() for r in rows]
    pd.DataFrame(_rows_out).to_csv(os.path.join(FILTER_FIG_DIR, "ms_per_image.csv"), index=False, encoding="utf-8-sig")
    pd.DataFrame([dict(condition=c, seed=s, method=m, **v) for (c, s, m), v in MS_RES.items()]).to_csv(
        os.path.join(FILTER_FIG_DIR, "ms_run_results.csv"), index=False, encoding="utf-8-sig")
    print(f"\nSaved ms_per_image.csv ({len(_rows_out)} rows) and ms_run_results.csv")

In [ ]:
# ── 30b. Multi-seed report ────────────────────────────────────────────────────────────────────────────
def _strict_vec(rows):
    return {r["key"]: int(r.get("miss") == 0 and r.get("extra") == 0) for r in rows}


def _acc_vec(rows):
    return {r["key"]: int(r["match"]) for r in rows}


def ms_cluster_boot(diff_by_key, group_of, n_boot=5000, seed=0):
    """Mean of the per-image differences and its 95% percentile CI when whole near-duplicate groups are resampled."""
    keys = sorted(diff_by_key)
    uniq = {}
    idx = np.array([uniq.setdefault(group_of.get(k, k), len(uniq)) for k in keys])
    d = np.array([diff_by_key[k] for k in keys], float)
    c = len(uniq)
    S = np.bincount(idx, weights=d, minlength=c)
    N = np.bincount(idx, minlength=c).astype(float)
    pick = np.random.default_rng(seed).integers(0, c, (n_boot, c))
    boots = S[pick].sum(axis=1) / N[pick].sum(axis=1)
    return float(d.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def ms_report(rows_by, seeds, conditions, ref="Original", group_of=None, n_boot=5000):
    """One DataFrame per condition (see Section 30 text). rows_by[(cond, seed, method)] = per-image rows."""
    group_of = group_of or {}
    tables = {}
    for cond, spec in conditions.items():
        methods = [m for m in spec["methods"] if all((cond, s, m) in rows_by for s in seeds)]
        if ref not in methods:
            continue
        seeds_ = list(seeds)
        sv = {m: {s: _strict_vec(rows_by[(cond, s, m)]) for s in seeds_} for m in methods}
        av = {m: {s: _acc_vec(rows_by[(cond, s, m)]) for s in seeds_} for m in methods}
        rec = []
        for m in methods:
            ps = [100.0 * np.mean(list(sv[m][s].values())) for s in seeds_]
            pa = [100.0 * np.mean(list(av[m][s].values())) for s in seeds_]
            r = {"Method": m}
            for s, v in zip(seeds_, ps):
                r[f"strict s{s}"] = v
            r["Strict mean"], r["Strict std"] = float(np.mean(ps)), (float(np.std(ps, ddof=1)) if len(ps) > 1 else float("nan"))
            r["Acc mean"], r["Acc std"] = float(np.mean(pa)), (float(np.std(pa, ddof=1)) if len(pa) > 1 else float("nan"))
            if m != ref:
                ref_ps = [100.0 * np.mean(list(sv[ref][s].values())) for s in seeds_]
                deltas = [a - b for a, b in zip(ps, ref_ps)]
                keys = set.intersection(*[set(sv[m][s]) & set(sv[ref][s]) for s in seeds_])
                dk = {k: float(np.mean([sv[m][s][k] - sv[ref][s][k] for s in seeds_])) for k in keys}
                mean_d, lo, hi = ms_cluster_boot(dk, group_of, n_boot)
                pvals = []
                for s in seeds_:
                    b = sum(sv[ref][s][k] == 1 and sv[m][s][k] == 0 for k in keys)
                    c = sum(sv[ref][s][k] == 0 and sv[m][s][k] == 1 for k in keys)
                    pvals.append(mcnemar_exact(b, c))
                r.update({"Delta vs Original (pp)": 100.0 * mean_d, "CI low": 100.0 * lo, "CI high": 100.0 * hi,
                          "seeds better": f"{sum(d > 0 for d in deltas)}/{len(deltas)}", "seeds worse": f"{sum(d < 0 for d in deltas)}/{len(deltas)}",
                          "per-seed McNemar p": " / ".join(f"{p:.3g}" for p in pvals)})
                same_sign = all(d > 0 for d in deltas) or all(d < 0 for d in deltas)
                r["Verdict"] = ("robust better" if mean_d > 0 else "robust worse") if (same_sign and (lo > 0 or hi < 0)) else (
                    "inconsistent" if not same_sign else "not significant")
            rec.append(r)
        tables[cond] = pd.DataFrame(rec)
    return tables


MS_TABLES = {}
if RUN_MULTISEED and USE_CV and MS_ROWS:
    MS_TABLES = ms_report(MS_ROWS, MS_SEEDS, MS_CONDITIONS, group_of=globals().get("DUP_GROUP_OF", {}))
    for cond, df in MS_TABLES.items():
        print(f"\n=== {cond}: strict accuracy (%) over seeds {MS_SEEDS}  (std = across seeds, ddof=1) ===")
        display(df.round(2))
        df.round(3).to_csv(os.path.join(FILTER_FIG_DIR, f"ms_table_{safe_filename(cond)}.csv"), index=False, encoding="utf-8-sig")
    _n = len(MS_TABLES)
    if _n:
        _fig, _axes = plt.subplots(1, _n, figsize=(6.5 * _n, 4.6), squeeze=False)
        for ax, (cond, df) in zip(_axes[0], MS_TABLES.items()):
            x = np.arange(len(df))
            ax.bar(x, df["Strict mean"], yerr=df["Strict std"].fillna(0), capsize=4, color="#9bb7d4", edgecolor="black")
            for s in MS_SEEDS:
                ax.scatter(x, df[f"strict s{s}"], s=26, zorder=3, label=f"seed {s}")
            ax.set_xticks(x)
            ax.set_xticklabels(df["Method"], rotation=30, ha="right", fontsize=11)
            ax.set_ylabel("Strict accuracy (%)")
            ax.set_title(cond, fontsize=13)
            ax.set_ylim(0, 105)
            ax.grid(axis="y", alpha=0.3)
        _axes[0][0].legend(fontsize=9)
        _fig.tight_layout()
        _p = os.path.join(FILTER_FIG_DIR, "fig_multiseed_strict.png")
        save_reference_figure(_fig, _p, dpi=300, bbox_inches="tight", facecolor="white")
        plt.show()
        print(f"Saved: {_p}")
elif RUN_MULTISEED:
    print("No multi-seed rows (USE_CV off or the runs were skipped).")

## 31. What to send back / paper numbers

Prints the numbers that replace the "single run" and "unresolved" statements of the paper and bundles the new tables into one zip (`v7_results_bundle.zip` in the figure folder). After the run, send **this notebook with its outputs** (the `.ipynb`) - the zip is only a convenience.

In [ ]:
# ── 31. Paper numbers + bundle ────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("v7 summary for the paper")
print("=" * 70)
PAPER_V7 = {}
if GHOST_ROWS:
    _p, _n_, _v = ghost_summary(GHOST_ROWS)
    PAPER_V7["ghost"] = dict(table=_p.round(2).to_dict(), verdict=_v)
    print("  [28a] ghost-at-erased-site % per method x site:", _p.round(1).to_dict())
    print("        ->", _v)
if SHORTCUT_DIAG.get("thumbnail"):
    PAPER_V7["thumbnail"] = SHORTCUT_DIAG["thumbnail"]
    for r in SHORTCUT_DIAG["thumbnail"]:
        print(f"  [28b] {r['Input']}: accuracy {r['Accuracy (%)']:.1f}% | balanced {r['Balanced (%)']:.1f}%")
if SHORTCUT_DIAG.get("global_auc"):
    PAPER_V7["global_auc"] = SHORTCUT_DIAG["global_auc"]
    print("  [28b] global-feature AUC:", {r["Feature"]: round(r["AUC (NG vs OK)"], 2) for r in SHORTCUT_DIAG["global_auc"]})
if OP_RESULT:
    PAPER_V7["operating_point"] = OP_RESULT
    print(f"  [28c] strict without the {OP_RESULT['n_fail']} fallback images: {OP_RESULT['strict_without_fail']:.1f}% (accuracy {OP_RESULT['acc_without_fail']:.1f}%)")
if REAL_RESULT:
    PAPER_V7["real_label"] = REAL_RESULT
    print(f"  [29]  label rule vs real list: {REAL_RESULT['rule_agree']}/{REAL_RESULT['n_matched']} = {REAL_RESULT['rule_agree_pct']:.2f}% "
          f"(CI {REAL_RESULT['ci'][0]:.1f} - {REAL_RESULT['ci'][1]:.1f}), {REAL_RESULT['n_disagree']} disagree")
for cond, df in MS_TABLES.items():
    PAPER_V7[f"ms::{cond}"] = df.round(3).to_dict(orient="records")
    print(f"\n  [30] {cond}")
    for _, r in df.sort_values("Strict mean", ascending=False).iterrows():
        extra = "" if r["Method"] == "Original" else f" | delta {r['Delta vs Original (pp)']:+.1f} pp [{r['CI low']:+.1f}, {r['CI high']:+.1f}] seeds better {r['seeds better']} -> {r['Verdict']}"
        print(f"      {r['Method']:<24s} strict {r['Strict mean']:5.1f} +- {r['Strict std']:4.1f} | accuracy {r['Acc mean']:5.1f} +- {r['Acc std']:4.1f}{extra}")

with open(os.path.join(FILTER_FIG_DIR, "paper_numbers_v7.json"), "w", encoding="utf-8") as f:
    json.dump(PAPER_V7, f, ensure_ascii=False, indent=1, default=float)

import zipfile
_patterns = ("ms_*.csv", "diag_*.csv", "real_label_*.csv", "table_*.csv", "verification_summary.*", "paper_numbers_v7.json",
             "fig_multiseed_strict.png", "fig_hard_shift_strict_heatmap.png", "hard_shift_*.json")
_bundle = os.path.join(FILTER_FIG_DIR, "v7_results_bundle.zip")
with zipfile.ZipFile(_bundle, "w", zipfile.ZIP_DEFLATED) as z:
    _seen = set()
    for pat in _patterns:
        for p in glob.glob(os.path.join(FILTER_FIG_DIR, pat)):
            if p not in _seen:
                z.write(p, os.path.basename(p))
                _seen.add(p)
print(f"\nBundle: {os.path.abspath(_bundle)} ({len(_seen)} files)")
print("Send back: the .ipynb with outputs (and the zip if the notebook is too big).")